In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import glob
from tqdm import tqdm
dest = os.getcwd()  # use r'' no Windows


os.makedirs(dest, exist_ok=True)             # cria recursivamente se não existir
os.chdir(dest)

# Exibir todas as colunas
pd.set_option('display.max_columns', None)

# Exibir todas as linhas
pd.set_option('display.max_rows', 30)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import seaborn
seaborn.set(style='whitegrid')
seaborn.set_context('talk')

In [ ]:
import importlib
import utils
importlib.reload(utils)

import phyil
importlib.reload(phyil)

dict_quantiles = {2: 0, 4: 1, 8: 2, 16: 3, 32: 4, 64: 5}

# Carregamento de dados

## Dataset

In [ ]:
df_dataset = pd.read_csv('df_dataset.csv', sep=';', decimal=',')
pontos_df = df_dataset.copy()
df_dataset.head()

In [ ]:
df_dataset.columns

In [ ]:
df_all = pd.read_csv('df_all.csv', sep=';', decimal=',')
df_all.head()

In [ ]:
df_all.columns

## First e Last quantis

In [ ]:
# CARREGAMENTO DOS FIRSTS E LASTS QUANTIS
# --- 1. CONFIGURAÇÃO ---

# Lista de quantis para procurar as pastas e arquivos. A ordem aqui definirá a ordem nas listas de resultado.
quantis = [2, 4, 8, 16, 32, 64]

# Diretório raiz onde as pastas '{qq}_quantis' estão localizadas.
diretorio_raiz = os.getcwd()

# ALTERADO: Inicializa duas listas vazias.
first_quantis = []
last_quantis = []

print(f"Iniciando a busca e organização de arquivos em listas: {diretorio_raiz}")

# --- 2. LOOP PRINCIPAL PARA LER OS ARQUIVOS E ADICIONAR ÀS LISTAS ---

for qq in tqdm(quantis, desc="Processando quantis"):
    # Constrói o caminho para a pasta do quantil atual
    folder_path = os.path.join(diretorio_raiz, f'{qq}_quantis')

    # --- Processa o arquivo 'first_quantil' ---
    first_filepath = os.path.join(folder_path, f'first_quantil_{qq}.csv')

    if os.path.exists(first_filepath):
        try:
            # Lê o DataFrame
            df_first = pd.read_csv(first_filepath, sep=';', decimal=',')
            # ALTERADO: Adiciona o DataFrame à lista 'first_quantis'
            first_quantis.append(df_first)
        except Exception as e:
            print(f"\nOcorreu um erro ao ler o arquivo {first_filepath}: {e}")
            first_quantis.append(None) # Adiciona None em caso de erro de leitura
    else:
        print(f"\nAVISO: Arquivo 'first' não encontrado para qq={qq}, adicionando None à lista.")
        # ALTERADO: Adiciona None para manter o alinhamento dos índices
        first_quantis.append(None)

    # --- Processa o arquivo 'last_quantil' ---
    last_filepath = os.path.join(folder_path, f'last_quantil_{qq}.csv')

    if os.path.exists(last_filepath):
        try:
            df_last = pd.read_csv(last_filepath, sep=';', decimal=',')
            # ALTERADO: Adiciona o DataFrame à lista 'last_quantis'
            last_quantis.append(df_last)
        except Exception as e:
            print(f"\nOcorreu um erro ao ler o arquivo {last_filepath}: {e}")
            last_quantis.append(None)
    else:
        print(f"\nAVISO: Arquivo 'last' não encontrado para qq={qq}, adicionando None à lista.")
        # ALTERADO: Adiciona None para manter o alinhamento dos índices
        last_quantis.append(None)

# --- 3. VERIFICAÇÃO FINAL ---
print("\n--- Processo de Carregamento Concluído ---")

# ALTERADO: Verificação para a lista 'first_quantis'
print("\nResumo dos dados carregados na lista 'first_quantis':")
for i, df in enumerate(first_quantis):
    qq_correspondente = quantis[i]
    if df is not None:
        print(f"  - Posição {i} (qq={qq_correspondente}) carregada. Dimensões: {df.shape}")
    else:
        print(f"  - Posição {i} (qq={qq_correspondente}) não foi carregada (arquivo não encontrado ou erro).")

# ALTERADO: Verificação para a lista 'last_quantis'
print("\nResumo dos dados carregados na lista 'last_quantis':")
for i, df in enumerate(last_quantis):
    qq_correspondente = quantis[i]
    if df is not None:
        print(f"  - Posição {i} (qq={qq_correspondente}) carregada. Dimensões: {df.shape}")
    else:
        print(f"  - Posição {i} (qq={qq_correspondente}) não foi carregada (arquivo não encontrado ou erro).")

## Rótulos

In [ ]:
# Lembrando que os rotulos originais, sem pertubacao, se referente àqueles onde o valor da pertubação é zero, em qualquer sigma e valor de quantil
# Carregar o arquivo salvo
output_folder = os.path.join(os.getcwd(), 'rotulos')
output_filename = os.path.join(output_folder, 'rotulos_unificados_todos_quantis_todos_sigmas.csv')
df_rotulos = pd.read_csv(output_filename, sep=';', decimal=',')
df_rotulos.head()

# Gráficos

## Gráfico do first e last para cada valor de quantil

### Gráficos dos objetivos

In [ ]:
pwd

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- Supõe-se que df_dataset e df_all já estejam carregados na memória ---

# Parâmetros
q_values = [2,4,8,16]
# q_values = [4]

print("--- Preparando dados baseados no ranqueamento NSGA-II ---")

# 1. Preparação dos dados
pontos_df_sorted = df_all.sort_values(by='nsga2_rank').reset_index(drop=True)

# 2. Verificações e Ranges Globais
use_global_ranges = True

if use_global_ranges:
    # Extraindo os dados para cálculo dos limites
    x_all = pontos_df_sorted["numero_feiras_livres"].to_numpy(dtype=float)
    y_all = pontos_df_sorted["ipvs"].to_numpy(dtype=float)
    z_all = pontos_df_sorted["obj_in_natura"].to_numpy(dtype=float)

    def pad_range(arr, pad_frac=0.05):
        lo, hi = float(np.min(arr)), float(np.max(arr))
        span = hi - lo
        if span == 0: span = 1.0
        pad = span * pad_frac
        return (lo - pad, hi + pad)

    xlim = pad_range(x_all)
    ylim = pad_range(y_all)
    zlim = pad_range(z_all)
else:
    xlim = ylim = zlim = None

# 3. Loop principal para gerar e salvar gráficos com Plotly
print("\n--- Iniciando a geração das imagens com Plotly ---")

output_dir = "graficos_plotly_quantis_nsga2"
os.makedirs(output_dir, exist_ok=True)

for q_value in q_values:
    print(f"\nProcessando para q = {q_value}...")

    # Divide o dataframe ordenado em 'q_value' partes
    quantis = np.array_split(pontos_df_sorted, q_value)

    first = quantis[0]  # Melhores ranqueados
    last = quantis[-1]  # Piores ranqueados

    if q_value > 2:
        rest = pd.concat(quantis[1:-1])
    else:
        rest = pd.DataFrame(columns=pontos_df_sorted.columns)

    fig = go.Figure()

    # --- Adiciona os traços ---

    # Primeiro Quantil (Melhores - Verde)
    if not first.empty:
        fig.add_trace(go.Scatter3d(
            x=first['numero_feiras_livres'], y=first['ipvs'], z=first['obj_in_natura'],
            mode='markers',
            marker=dict(size=5, color='green', opacity=0.8),
            name='Primeiro Quantil'
        ))

    # Último Quantil (Piores - Vermelho)
    if not last.empty:
        fig.add_trace(go.Scatter3d(
            x=last['numero_feiras_livres'], y=last['ipvs'], z=last['obj_in_natura'],
            mode='markers',
            marker=dict(size=5, color='red', opacity=0.8),
            name='Último Quantil'
        ))

    # Quantis Intermediários (Azul)
    if not rest.empty:
        fig.add_trace(go.Scatter3d(
            x=rest['numero_feiras_livres'], y=rest['ipvs'], z=rest['obj_in_natura'],
            mode='markers',
            marker=dict(size=4, color='blue', opacity=0.4),
            name='Quantis Intermediários'
        ))

    # Configuração da Câmera

    # {x: -2.022704374699194, y: -1.7091728570570288, z: 0.6836630436642425}

    camera_eye = dict(x=-2.022704374699194, y=-1.7091728570570288, z=0.6836630436642425)

    fig.update_layout(
        template="plotly_white",
        showlegend=False,
        width=800,
        height=800,

        scene=dict(
            xaxis_title="Open-air markets",
            yaxis_title='IPVS',
            zaxis_title='Obj. In Natura',
            xaxis=dict(range=xlim),
            yaxis=dict(range=ylim),
            zaxis=dict(range=zlim),
            aspectmode='cube',

            camera=dict(
                # ZOOM (AFASTADO): Aumentei os valores de 1.25/1.05 para 1.55/1.35
                # Quanto maiores esses números, mais longe a câmera fica.
                eye=dict(x=-1.55, y=-1.35, z=0.55),

                # CENTER: Mantive o ajuste que centraliza o cubo na tela
                center=dict(x=-0.05, y=0, z=-0.05)
            )
        ),

        margin=dict(l=0, r=0, b=0, t=0),
        paper_bgcolor='white'
    )

    filename_base = os.path.join(output_dir, f"grafico_nsga2_q{q_value}_rotacionado")

    fig.write_html(f"{filename_base}.html")
    print(f"Salvo: {filename_base}.html")

    try:
        # fig.write_image(f"{filename_base}.png", scale=5)
        # print(f"Salvo: {filename_base}.png")

        fig.write_image(f"{filename_base}.pdf", scale=5)
        print(f"Salvo: {filename_base}.pdf")

        # fig.write_image(f"{filename_base}.svg", scale=5)
        # print(f"Salvo: {filename_base}.svg")
    except ValueError as e:
        print(f"\nAVISO: Não foi possível salvar o PNG/PDF. A biblioteca 'kaleido' é necessária.")

## Gráfico de superfície

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
from tqdm import tqdm

# --- PRÉ-REQUISITOS (ASSUMINDO QUE AS VARIÁVEIS JÁ EXISTEM) ---
# df_dataset já deve existir no ambiente.

# --- NOVO: Criação da pasta de saída para os gráficos ---
# Isso é feito uma vez, antes do loop, para manter a organização.
output_dir_graficos = "graficos_superficie"
os.makedirs(output_dir_graficos, exist_ok=True)
print(f"Gráficos serão salvos no diretório: '{output_dir_graficos}'")


for qq in tqdm([2,4,8,16,32,64], desc="Gerando Gráficos por Quantil"):
    # --- Bloco de cálculo dos dados (SEU CÓDIGO ORIGINAL, INALTERADO) ---
    FOLDER_PATH = os.path.join(os.getcwd(), os.path.join('matrizes_margens_perturbadas',f'{qq}_quantis') )
    filename = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'

    try:
        df = pd.read_csv(os.path.join(FOLDER_PATH, filename), sep=';', decimal=',')
    except FileNotFoundError:
        print(f"\nAVISO: Arquivo de margens para qq={qq} não encontrado. Pulando.")
        continue

    sigmas = np.round(np.arange(0.01, 0.31, 0.01), 2)
    hist_data = []
    bin_edges = None

    for sigma in sigmas:
        df_sigma = df[df['sigma'] == sigma]
        # Garante que a coluna exista antes de tentar acessá-la
        if df_dataset.shape[0] > 1:
            margens = df_sigma[[f'margem_{i}' for i in range(1,df_dataset.shape[0])]].values.flatten()
            hist, bins = np.histogram(margens, bins=50, range=(0, 1))
            if bin_edges is None: bin_edges = bins
            hist_data.append(hist)

    if not hist_data: # Verifica se hist_data foi populado
        print(f"\nAVISO: Nenhum dado de margem processado para qq={qq}. Pulando gráfico.")
        continue

    hist_matrix = np.array(hist_data).T
    custom_colorscale = [[0.0,'rgb(50,50,50)'], [0.01,'rgb(230,230,255)'], [0.2,'rgb(68,1,84)'], [0.4,'rgb(58,82,139)'], [0.6,'rgb(32,144,140)'], [0.8,'rgb(94,201,98)'], [1.0,'rgb(253,231,37)']]
    zmax = hist_matrix.max()
    fig = go.Figure()
    fig.add_trace(go.Surface(x=sigmas, y=(bin_edges[:-1] + bin_edges[1:])/2, z=hist_matrix, colorscale=custom_colorscale, cmin=0, cmax=zmax, name='Distribuição', hovertemplate='<b>Sigma</b>: %{x:.2f}<br><b>Margem</b>: %{y:.3f}<br><b>Frequência</b>: %{z}<extra></extra>'))
    fig.add_trace(go.Scatter3d(x=sigmas, y=[0.2]*len(sigmas), z=[hist_matrix[np.abs((bin_edges[:-1] + bin_edges[1:])/2 - 0.2).argmin(), i] for i in range(len(sigmas))], mode='lines', line=dict(color='red', width=4), name='Limite Fraca (0.2)', hovertemplate='<b>Sigma</b>: %{x:.2f}<br><b>Margem</b>: 0.2<br><b>Frequência</b>: %{z}<extra></extra>'))
    fig.add_trace(go.Scatter3d(x=sigmas, y=[0.6]*len(sigmas), z=[hist_matrix[np.abs((bin_edges[:-1] + bin_edges[1:])/2 - 0.6).argmin(), i] for i in range(len(sigmas))], mode='lines', line=dict(color='green', width=4), name='Limite Forte (0.6)', hovertemplate='<b>Sigma</b>: %{x:.2f}<br><b>Margem</b>: 0.6<br><b>Frequência</b>: %{z}<extra></extra>'))
    # fig.update_layout(title=f'Distribuição 3D das Margens por Sigma - {qq} Quantis', scene=dict(xaxis_title='Sigma', yaxis_title='Margem', zaxis_title='Frequência', camera=dict(eye=dict(x=-1.5, y=-1.5, z=0.8))), margin=dict(l=0, r=0, b=0, t=40))
    fig.update_layout(
        title=f'Distribuição 3D das Margens por Sigma - {qq} Quantis',
        scene=dict(
            xaxis_title='Sigma',
            yaxis_title='Margem',
            zaxis_title='Frequência',

# {x: -0.5280903639751849, y: -1.3548913863432783, z: 1.882074623570847}
            camera=dict(eye=dict(x=-0.5280903639751849, y=-1.3548913863432783, z=1.882074623570847))
        ),
        margin=dict(l=0, r=0, b=0, t=40),

        # NOVO: Adiciona e configura a posição da legenda
        legend=dict(
            x=0.05, # Posição 5% a partir da borda esquerda
            y=0.95, # Posição 95% a partir da borda inferior
            yanchor="top", # Âncora o TOPO da legenda nesta posição y
            xanchor="left",# Âncora a ESQUERDA da legenda nesta posição x
            bgcolor='rgba(255, 255, 255, 0.7)', # Fundo branco semi-transparente
            bordercolor='black', # Cor da borda
            borderwidth=1 # Largura da borda
        )
    )
    # --- NOVO: SEÇÃO DE SALVAMENTO DOS GRÁFICOS ---

    # 1. Define um nome base para os arquivos de saída
    filename_base = f"superficie_margens_{qq}_quantis"

    # 2. Salva o gráfico em HTML (interativo)
    html_path = os.path.join(output_dir_graficos, f"{filename_base}.html")
    fig.write_html(html_path)
    print(f"\n  -> Gráfico para qq={qq} salvo em: {os.path.basename(html_path)}")

    # 3. Salva em PNG (alta resolução) e PDF (vetorial)
    #    Usa um bloco try/except para o caso de a biblioteca 'kaleido' não estar instalada.
    try:
        # Salva em PNG com escala 3x (alta resolução)
        png_path = os.path.join(output_dir_graficos, f"{filename_base}.png")
        fig.write_image(png_path, scale=3)
        print(f"  -> Gráfico para qq={qq} salvo em: {os.path.basename(png_path)}")

        # Salva em PDF
        pdf_path = os.path.join(output_dir_graficos, f"{filename_base}.pdf")
        fig.write_image(pdf_path)
        print(f"  -> Gráfico para qq={qq} salvo em: {os.path.basename(pdf_path)}")

    except ValueError as e:
        print(f"\nAVISO: Não foi possível salvar em PNG/PDF. A biblioteca 'kaleido' é necessária.")
        print("Execute no terminal: pip install kaleido")
        print(f"Erro original: {e}")

    # Exibe o gráfico interativo no notebook (comportamento original mantido)
    # fig.show()

print("\n\nProcesso concluído!")

## Gráfico de superficie com escala logaritmica.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
from tqdm import tqdm

# --- PRÉ-REQUISITO (ASSUMINDO QUE A VARIÁVEL JÁ EXISTE) ---
# df_dataset já deve existir no ambiente.

# --- Criação da pasta de saída para os gráficos ---
output_dir_graficos = "graficos_superficie_log"
os.makedirs(output_dir_graficos, exist_ok=True)
print(f"Gráficos serão salvos no diretório: '{output_dir_graficos}'")


for qq in tqdm([2,4,8,16,32,64], desc="Gerando Gráficos por Quantil"):
    FOLDER_PATH = os.path.join(os.getcwd(), os.path.join('matrizes_margens_perturbadas',f'{qq}_quantis') )
    filename = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'
    file_full_path = os.path.join(FOLDER_PATH, filename)

    try:
        # --- ALTERAÇÃO PRINCIPAL AQUI ---
        # Tenta ler o arquivo diretamente. Se não encontrar, o bloco 'except' será acionado.
        # A lógica if/else que gerava dados de exemplo foi removida.
        df = pd.read_csv(file_full_path, sep=';', decimal=',')

    except FileNotFoundError:
        # Se o arquivo não for encontrado, avisa o usuário e pula para o próximo 'qq'.
        print(f"\nAVISO: Arquivo '{file_full_path}' não encontrado. Pulando para o próximo quantil.")
        continue # Pula para a próxima iteração do loop

    # --- O restante do código permanece o mesmo ---

    sigmas = np.round(np.arange(0.01, 0.31, 0.01), 2)
    hist_data = []
    bin_edges = None

    for sigma in sigmas:
        df_sigma = df[df['sigma'] == sigma]
        if df_dataset.shape[0] > 1:
            margens = df_sigma[[f'margem_{i}' for i in range(1,df_dataset.shape[0])]].values.flatten()
            margens = margens[~np.isnan(margens)]
            hist, bins = np.histogram(margens, bins=50, range=(0, 1))
            if bin_edges is None: bin_edges = bins
            hist_data.append(hist)

    if not hist_data:
        print(f"\nAVISO: Nenhum dado de margem processado para qq={qq}. Pulando gráfico.")
        continue

    hist_matrix = np.array(hist_data).T
    log_hist_matrix = np.log10(hist_matrix + 1)

    custom_colorscale = [
        [0.0, 'rgb(50,50,50)'], [0.01, 'rgb(230,230,255)'], [0.2, 'rgb(68,1,84)'],
        [0.4, 'rgb(58,82,139)'], [0.6, 'rgb(32,144,140)'], [0.8, 'rgb(94,201,98)'],
        [1.0, 'rgb(253,231,37)']
    ]

    zmax = log_hist_matrix.max()
    fig = go.Figure()

    fig.add_trace(go.Surface(
        x=sigmas, y=(bin_edges[:-1] + bin_edges[1:])/2, z=log_hist_matrix,
        colorscale=custom_colorscale, cmin=0, cmax=zmax, name='Distribuição',
        colorbar=dict(title='Frequency (Log)'),
        hovertemplate='<b>Sigma</b>: %{x:.2f}<br><b>Margem</b>: %{y:.3f}<br><b>Frequência (log10)</b>: %{z:.2f}<extra></extra>'
    ))

    # fig.add_trace(go.Scatter3d(
    #     x=sigmas, y=[0.2]*len(sigmas), z=np.log10(hist_matrix[np.abs((bin_edges[:-1] + bin_edges[1:])/2 - 0.2).argmin(), :] + 1),
    #     mode='lines', line=dict(color='red', width=4), name='Limite Fraca (0.2)', hoverinfo='skip'
    # ))

    # fig.add_trace(go.Scatter3d(
    #     x=sigmas, y=[0.6]*len(sigmas), z=np.log10(hist_matrix[np.abs((bin_edges[:-1] + bin_edges[1:])/2 - 0.6).argmin(), :] + 1),
    #     mode='lines', line=dict(color='green', width=4), name='Limite Forte (0.6)', hoverinfo='skip'
    # ))

    # --- CONFIGURAÇÃO VISUAL DAS GRADES ---
    grid_style = dict(
        showgrid=True,
        gridcolor='rgb(100, 100, 100)',  # Cinza mais forte (quase escuro)
        gridwidth=2,                     # Linha um pouco mais grossa
        zeroline=False,
        backgroundcolor='white'          # Fundo branco para contraste
    )

    fig.update_layout(
        # title=f'Distribuição 3D das Margens por Sigma - {qq} Quantis (Escala Log)',
        template='plotly_white',
        scene=dict(# Aplica o estilo de grade em todos os eixos
            xaxis=dict(title='Sigma', **grid_style),
            yaxis=dict(title='Confidence Margin', **grid_style),
            zaxis=dict(title='Frequency (Log)', **grid_style),
            xaxis_title='Sigma', yaxis_title='Confidence Margin', zaxis_title='Frequency (Log)',
            camera=dict(eye=dict(x=-0.5280903639751849, y=-1.3548913863432783, z=1.882074623570847))
        ),
        showlegend=False,
        font=dict(family="Arial, sans-serif", size=14, color="black"),
        margin=dict(l=0, r=0, b=0, t=40),
        legend=dict(x=0.05, y=0.95, yanchor="top", xanchor="left", bgcolor='rgba(255, 255, 255, 0.7)', bordercolor='black', borderwidth=1)
    )

    # --- Seção de Salvamento de Arquivos ---
    filename_base = f"superficie_log_margens_{qq}_quantis"
    html_path = os.path.join(output_dir_graficos, f"{filename_base}.html")
    fig.write_html(html_path)
    print(f"\n  -> Gráfico para qq={qq} salvo em: {os.path.basename(html_path)}")

    try:
        # png_path = os.path.join(output_dir_graficos, f"{filename_base}.png")
        # fig.write_image(png_path, scale=5)
        # print(f"  -> Gráfico para qq={qq} salvo em: {os.path.basename(png_path)}")
        pdf_path = os.path.join(output_dir_graficos, f"{filename_base}.pdf")
        fig.write_image(pdf_path, scale=5)
        print(f"  -> Gráfico para qq={qq} salvo em: {os.path.basename(pdf_path)}")

        # pdf_path = os.path.join(output_dir_graficos, f"{filename_base}.svg")
        # fig.write_image(pdf_path, scale=5)
        # print(f"  -> Gráfico para qq={qq} salvo em: {os.path.basename(pdf_path)}")
    except ValueError:
        print(f"\nAVISO: Não foi possível salvar em PNG/PDF. A biblioteca 'kaleido' é necessária.")
        print("Para instalar, use: pip install kaleido")

    # fig.show()

print("\n\nProcesso concluído!")

# Gráficos de superfície coloridas pela taxa de flip-rate.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
from tqdm import tqdm

# --- CONFIGURAÇÃO ---
CAMINHO_FLIP_RATES = os.path.join('flip_rates', 'flip_rates_consolidados.csv')

# --- Carregamento ---
try:
    df_flip_all = pd.read_csv(CAMINHO_FLIP_RATES, sep=';', decimal=',')
    print(f"Flip Rates carregados.")
except FileNotFoundError:
    print(f"ERRO: Arquivo {CAMINHO_FLIP_RATES} não encontrado.")
    df_flip_all = pd.DataFrame()

# --- Saída ---
output_dir_graficos = "graficos_superficie_log_color_flip"
os.makedirs(output_dir_graficos, exist_ok=True)

# --- Loop ---
for qq in tqdm([2,4,8,16], desc="Gerando Gráficos"):
    FOLDER_PATH = os.path.join(os.getcwd(), os.path.join('matrizes_margens_perturbadas',f'{qq}_quantis') )
    filename = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'
    file_full_path = os.path.join(FOLDER_PATH, filename)

    try:
        df_margens = pd.read_csv(file_full_path, sep=';', decimal=',')
    except FileNotFoundError:
        print(f"\nAVISO: Arquivo não encontrado para qq={qq}. Pulando.")
        continue

    # Mapa de Flip Rate
    mapa_flip = {}
    if not df_flip_all.empty:
        df_flip_qq = df_flip_all[df_flip_all['qq'] == qq].copy()
        colunas_pert = [c for c in df_flip_qq.columns if 'pert' in c]
        df_flip_qq['flip_medio'] = df_flip_qq[colunas_pert].mean(axis=1)
        mapa_flip = dict(zip(df_flip_qq['sigma'].round(2), df_flip_qq['flip_medio']))

    # Matrizes
    sigmas = np.sort(df_margens['sigma'].unique())
    hist_data = []
    color_data = []
    bin_edges = None
    bins_config = np.linspace(0, 1, 51)

    for sigma in sigmas:
        df_sigma = df_margens[df_margens['sigma'] == sigma]
        cols_margem = [c for c in df_sigma.columns if c.startswith('margem_')]
        vals = df_sigma[cols_margem].values.flatten()
        vals = vals[~np.isnan(vals)]

        hist, bins = np.histogram(vals, bins=bins_config)
        if bin_edges is None: bin_edges = bins
        hist_data.append(hist)

        flip_val = mapa_flip.get(round(sigma, 2), 0.0)
        color_data.append(np.full_like(hist, flip_val, dtype=float))

    if not hist_data: continue

    hist_matrix = np.array(hist_data).T
    flip_matrix = np.array(color_data).T
    log_hist_matrix = np.log10(hist_matrix + 1)
    y_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # --- PLOTAGEM ---
    fig = go.Figure()

    fig.add_trace(go.Surface(
        x=sigmas,
        y=y_centers,
        z=log_hist_matrix,
        surfacecolor=flip_matrix,
        colorscale='Jet',
        cmin=0.0, cmax=0.5,
        name='Distribuição',

        # --- AJUSTE DA COLORBAR (MUDANÇA AQUI) ---
        colorbar=dict(
            title='Flip Rate',
            thickness=25,
            len=0.75,      # Reduzi levemente a altura para centralizar melhor
            x=0.85,        # <--- TRAZENDO PARA DENTRO (Era 1.02)
            xanchor='left',
            y=0.5,
            yanchor='middle'
        ),

        hovertemplate='<b>Sigma</b>: %{x:.2f}<br><b>Margem</b>: %{y:.3f}<br><b>Freq(Log)</b>: %{z:.2f}<br><b>Flip Rate</b>: %{surfacecolor:.4f}<extra></extra>'
    ))

    # Linhas
    idx_02 = np.abs(y_centers - 0.2).argmin()
    fig.add_trace(go.Scatter3d(
        x=sigmas, y=[0.2]*len(sigmas), z=log_hist_matrix[idx_02, :],
        mode='lines', line=dict(color='white', width=5), name='Limite 0.2'
    ))

    idx_06 = np.abs(y_centers - 0.6).argmin()
    fig.add_trace(go.Scatter3d(
        x=sigmas, y=[0.6]*len(sigmas), z=log_hist_matrix[idx_06, :],
        mode='lines', line=dict(color='black', width=5), name='Limite 0.6'
    ))

    fig.update_layout(
        title=f'Superfície de Margens (Cor=Flip Rate) - {qq} Quantis',
        scene=dict(
            xaxis_title='Sigma',
            yaxis_title='Margem',
            zaxis_title='Frequência (Log)',
            camera=dict(eye=dict(x=-1.5, y=-1.5, z=1.5))
        ),
        # Margem direita reduzida (já que trouxemos a barra para dentro)
        margin=dict(l=0, r=20, b=0, t=40),

        legend=dict(
            x=0.02,
            y=0.90, # Baixei levemente a legenda
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.6)'
        )
    )

    # Salvar
    filename_base = f"superficie_flip_color_{qq}_quantis"
    html_path = os.path.join(output_dir_graficos, f"{filename_base}.html")
    fig.write_html(html_path)
    print(f"  -> Gráfico salvo: {filename_base}.html")

    try:
        png_path = os.path.join(output_dir_graficos, f"{filename_base}.png")
        fig.write_image(png_path, scale=5)
        png_path = os.path.join(output_dir_graficos, f"{filename_base}.pdf")
        fig.write_image(png_path, scale=5)

        png_path = os.path.join(output_dir_graficos, f"{filename_base}.svg")
        fig.write_image(png_path, scale=5)
    except:
        pass

print("\nProcesso finalizado!")

# Grafico superficie colorido por flip-rate usando matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from mpl_toolkits.mplot3d import Axes3D
import os
from tqdm import tqdm

# --- 1. CARREGAMENTO (Igual ao anterior) ---
CAMINHO_FLIP_RATES = os.path.join('flip_rates', 'flip_rates_consolidados.csv')
CAMINHO_DATASET = 'df_dataset.csv'


output_dir = "graficos_matplotlib"
os.makedirs(output_dir, exist_ok=True)

# --- LOOP POR QUANTIL ---
for qq in tqdm([2, 4, 8, 16], desc="Gerando Matplotlib"):

    # Carregar Margens
    FOLDER_PATH = os.path.join(os.getcwd(), os.path.join('matrizes_margens_perturbadas', f'{qq}_quantis'))
    filename = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'
    file_path = os.path.join(FOLDER_PATH, filename)

    try:
        df = pd.read_csv(file_path, sep=';', decimal=',')
    except FileNotFoundError:
        continue

    # Preparar Mapa de Flip Rate
    mapa_flip = {}
    if not df_flip_all.empty:
        df_f = df_flip_all[df_flip_all['qq'] == qq].copy()
        cols_p = [c for c in df_f.columns if 'pert' in c]
        df_f['media'] = df_f[cols_p].mean(axis=1)
        mapa_flip = dict(zip(df_f['sigma'].round(2), df_f['media']))

    # --- PROCESSAMENTO DOS DADOS ---
    sigmas = np.sort(df['sigma'].unique())

    # Configuração dos Bins
    bins_edges = np.linspace(0, 1, 51)
    y_centers = (bins_edges[:-1] + bins_edges[1:]) / 2

    hist_list = []
    flip_list = []

    for s in sigmas:
        # 1. Altura (Z)
        d_s = df[df['sigma'] == s]
        cols = [c for c in d_s.columns if c.startswith('margem_')]
        vals = d_s[cols].values.flatten()
        vals = vals[~np.isnan(vals)]

        h, _ = np.histogram(vals, bins=bins_edges)
        hist_list.append(h)

        # 2. Cor (C) - Flip Rate
        val_flip = mapa_flip.get(round(s, 2), 0.0)
        # Repete o valor do flip para preencher a coluna inteira deste sigma
        flip_list.append(np.full_like(h, val_flip, dtype=float))

    if not hist_list: continue

    # Matrizes Finais (Shape: Sigmas x Margens)
    # Matplotlib prefere X e Y como grid 2D
    X_grid, Y_grid = np.meshgrid(sigmas, y_centers)

    # Z (Altura) - Transposta para alinhar com meshgrid (Margens x Sigmas)
    Z_data = np.array(hist_list).T
    Z_log = np.log10(Z_data + 1)

    # Cor (Flip) - Transposta para alinhar
    C_data = np.array(flip_list).T

    # --- PLOTAGEM COM MATPLOTLIB ---
    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111, projection='3d')

    # 1. Normalização da Cor
    # Isso mapeia os valores de Flip Rate (ex: 0.0 a 0.5) para cores RGBA
    norm = colors.Normalize(vmin=C_data.min(), vmax=C_data.max())
    m = cm.ScalarMappable(norm=norm, cmap='jet') # Pode mudar 'jet' para 'viridis', 'plasma'
    fcolors = m.to_rgba(C_data)

    # 2. Criação da Superfície
    # rstride e cstride controlam a "resolução" da grade desenhada (quanto menor, mais detalhado)
    surf = ax.plot_surface(
        X_grid, Y_grid, Z_log,
        rstride=1, cstride=1,
        facecolors=fcolors, # AQUI: Define a cor independentemente do Z
        linewidth=0,
        antialiased=False,
        shade=False # Importante: shade=False para não escurecer a cor com sombras de luz
    )

    # 3. Linhas de Limite (Fraca/Forte)
    # Desenhamos linhas simples sobre a superfície para referência
    idx_02 = np.abs(y_centers - 0.2).argmin()
    idx_06 = np.abs(y_centers - 0.6).argmin()

    ax.plot(sigmas, [0.2]*len(sigmas), Z_log[idx_02, :], color='white', linewidth=2, label='Limiar 0.2', zorder=10)
    ax.plot(sigmas, [0.6]*len(sigmas), Z_log[idx_06, :], color='black', linewidth=2, label='Limiar 0.6', zorder=10)

    # 4. Ajustes Visuais
    ax.set_xlabel('Sigma')
    ax.set_ylabel('Margem')
    ax.set_zlabel('Frequência (Log10)')
    ax.set_title(f'Superfície de Margens ({qq} Quantis)\nCor = Flip Rate')

    # Barra de Cores dedicada ao Flip Rate
    m.set_array(C_data)
    cbar = plt.colorbar(m, ax=ax, shrink=0.6, pad=0.1)
    cbar.set_label('Flip Rate')

    # Ângulo de visão inicial
    ax.view_init(elev=30, azim=-135)

    # Salvamento
    path_img = os.path.join(output_dir, f'superficie_mpl_{qq}_quantis.png')
    plt.savefig(path_img, dpi=150)
    print(f"Salvo: {path_img}")

    # plt.show() # Descomente se quiser ver no notebook
    plt.close(fig) # Fecha para liberar memória

print("\nProcesso Matplotlib Concluído.")

# Gráfico de superficie colorido por taxa de concordancia para um % especifico

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
from tqdm import tqdm

# --- CONFIGURAÇÃO ---
PERCENTAGEM_DADOS = 0.10  # 10% das amostras (Topo e Fundo do Rank)
LISTA_QUANTIS = [2, 4, 8, 16]

# Mock para df_dataset se não existir
if 'df_dataset' not in locals():
    try:
        df_dataset = pd.read_csv('df_dataset.csv', sep=';', decimal=',')
        n_samples = df_dataset.shape[0]
    except:
        print("Aviso: df_dataset não encontrado. Assumindo 3000 amostras.")
        n_samples = 3000
else:
    n_samples = df_dataset.shape[0]

# Carregar df_rotulos (Base para o cálculo da cor - Perturbação 0)
CAMINHO_ROTULOS = os.path.join('rotulos', 'rotulos_unificados_todos_quantis_todos_sigmas.csv')
try:
    df_rotulos = pd.read_csv(CAMINHO_ROTULOS, sep=';', decimal=',')
    # Filtra APENAS a perturbação 0 para o cálculo da cor
    df_rotulado_base = df_rotulos[df_rotulos['perturbacao'] == 0].copy()
except FileNotFoundError:
    print(f"ERRO CRÍTICO: Arquivo de rótulos não encontrado em {CAMINHO_ROTULOS}")
    df_rotulado_base = pd.DataFrame()

# Pasta de saída
output_dir_graficos = "graficos_superficie_soma_concordancia_percentual"
os.makedirs(output_dir_graficos, exist_ok=True)
print(f"Gráficos serão salvos em: {output_dir_graficos}")

# ==============================================================================
# FASE 1: PRÉ-CÁLCULO DOS VALORES (Para definir Min/Max Global)
# ==============================================================================
print("\n--- FASE 1: Calculando taxas para definir escala global ---")

# Dicionário para armazenar os cálculos e evitar refazer contas na Fase 2
# Estrutura: cache_concordancia[qq] = {sigma: valor_concordancia}
cache_concordancia = {}
todos_valores_calculados = []

for qq in tqdm(LISTA_QUANTIS, desc="Calculando Métricas"):
    cache_concordancia[qq] = {}

    # 1. Carregar Dados Brutos e ORDENAR POR RANK NSGA2
    file_path_dados = os.path.join(f'{qq}_quantis', f'dados_divididos_em_{qq}_quantis.csv')
    try:
        df_dados = pd.read_csv(file_path_dados, sep=';', decimal=',')
        if 'nsga2_rank' in df_dados.columns:
            df_dados = df_dados.sort_values(by='nsga2_rank', ascending=True)
        else:
            print(f"Aviso: 'nsga2_rank' não encontrado para qq={qq}.")
    except FileNotFoundError:
        print(f"Arquivo de dados não encontrado para qq={qq}")
        continue

    # 2. Selecionar Top/Bottom 10%
    total_linhas = len(df_dados)
    n_selecao = max(1, int(total_linhas * PERCENTAGEM_DADOS))

    df_first = df_dados.iloc[:n_selecao]
    df_last = df_dados.iloc[-n_selecao:]

    ids_first = df_first['ID'].tolist()
    ids_last = df_last['ID'].tolist()

    cols_rot_first = [f'rotulo_{int(i)}' for i in ids_first]
    cols_rot_last = [f'rotulo_{int(i)}' for i in ids_last]

    # 3. Calcular Métrica para cada Sigma disponível neste quantil
    # Filtramos o df_rotulado_base para pegar os sigmas deste qq
    df_base_qq = df_rotulado_base[df_rotulado_base['qq'] == qq]
    sigmas_disponiveis = np.sort(df_base_qq['sigma'].unique())

    for sigma in sigmas_disponiveis:
        sigma = round(sigma, 2)
        linha = df_base_qq[df_base_qq['sigma'] == sigma]

        if not linha.empty and ids_first and ids_last:
            # Validar colunas
            valid_first = [c for c in cols_rot_first if c in linha.columns]
            valid_last = [c for c in cols_rot_last if c in linha.columns]

            if valid_first and valid_last:
                # Taxa Inicial (0)
                rot_first = linha[valid_first].values.flatten()
                taxa_first = (len(rot_first) - np.sum(rot_first)) / len(rot_first)

                # Taxa Final (1)
                rot_last = linha[valid_last].values.flatten()
                taxa_last = np.sum(rot_last) / len(rot_last)

                soma_conc = taxa_first + taxa_last

                # Armazena
                cache_concordancia[qq][sigma] = soma_conc
                todos_valores_calculados.append(soma_conc)

# --- DEFINIÇÃO DOS LIMITES GLOBAIS ---
if todos_valores_calculados:
    GLOBAL_CMIN = min(todos_valores_calculados)
    GLOBAL_CMAX = max(todos_valores_calculados)

    # Ajuste fino para não quebrar se min == max
    if GLOBAL_CMIN == GLOBAL_CMAX:
        GLOBAL_CMIN -= 0.1
        GLOBAL_CMAX += 0.1

    # Trava limites lógicos [0, 2]
    GLOBAL_CMIN = max(0.0, GLOBAL_CMIN)
    GLOBAL_CMAX = min(2.0, GLOBAL_CMAX)

    print(f"\nLimites Globais Definidos: Min={GLOBAL_CMIN:.4f}, Max={GLOBAL_CMAX:.4f}")
else:
    print("\nNenhum valor calculado. Verifique seus arquivos.")
    GLOBAL_CMIN, GLOBAL_CMAX = 0, 2

# ==============================================================================
# FASE 2: PLOTAGEM DOS GRÁFICOS (Usando limites fixos)
# ==============================================================================
print("\n--- FASE 2: Gerando Gráficos ---")

for qq in tqdm(LISTA_QUANTIS, desc="Plotando"):

    # 1. Carregar Margens (Geometria)
    FOLDER_PATH_PERT = os.path.join(os.getcwd(), os.path.join('matrizes_margens_perturbadas', f'{qq}_quantis'))
    filename_margens = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'
    path_margens = os.path.join(FOLDER_PATH_PERT, filename_margens)

    try:
        df_margens = pd.read_csv(path_margens, sep=';', decimal=',')
    except FileNotFoundError:
        continue

    # 2. Preparar Matrizes para Plotagem
    sigmas = np.sort(df_margens['sigma'].unique())
    hist_data = []
    color_data = []
    bin_edges = None
    bins_config = np.linspace(0, 1, 51)

    for sigma in sigmas:
        sigma_r = round(sigma, 2)

        # A. Geometria (Margens)
        df_sigma = df_margens[df_margens['sigma'] == sigma]
        cols_m = [c for c in df_sigma.columns if c.startswith('margem_')]
        vals = df_sigma[cols_m].values.flatten()
        vals = vals[~np.isnan(vals)]

        hist, bins = np.histogram(vals, bins=bins_config)
        if bin_edges is None: bin_edges = bins
        hist_data.append(hist)

        # B. Cor (Usando o valor pré-calculado)
        # Se não houver valor calculado para este sigma, usa 0 (ou GLOBAL_CMIN)
        conc_val = cache_concordancia.get(qq, {}).get(sigma_r, 0.0)
        color_data.append(np.full_like(hist, conc_val, dtype=float))

    if not hist_data: continue

    hist_matrix = np.array(hist_data).T
    concordancia_matrix = np.array(color_data).T
    log_hist_matrix = np.log10(hist_matrix + 1)
    y_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # --- PLOTAGEM ---
    fig = go.Figure()

    fig.add_trace(go.Surface(
        x=sigmas,
        y=y_centers,
        z=log_hist_matrix,

        surfacecolor=concordancia_matrix,
        customdata=concordancia_matrix,

        colorscale='Jet',

        # --- AQUI ESTÁ A CHAVE: Usar os limites globais ---
        cmin=GLOBAL_CMIN,
        cmax=GLOBAL_CMAX,
        # --------------------------------------------------

        name='Distribuição',

        colorbar=dict(
            title=f'Soma Concordância<br>(Top/Bot {PERCENTAGEM_DADOS*100:.0f}%)',
            thickness=25,
            len=0.75,
            x=0.85,
            xanchor='left',
            y=0.5,
            yanchor='middle'
        ),

        hovertemplate=(
            '<b>Sigma</b>: %{x:.2f}<br>' +
            '<b>Margem</b>: %{y:.3f}<br>' +
            '<b>Freq(Log)</b>: %{z:.2f}<br>'
            # '<b>Soma Conc.</b>: %{customdata:.4f}<extra></extra>'
        )
    ))

    # Linhas de Limite
    idx_02 = np.abs(y_centers - 0.2).argmin()
    fig.add_trace(go.Scatter3d(
        x=sigmas, y=[0.2]*len(sigmas), z=log_hist_matrix[idx_02, :],
        mode='lines', line=dict(color='white', width=5), name='Limite 0.2'
    ))

    idx_06 = np.abs(y_centers - 0.6).argmin()
    fig.add_trace(go.Scatter3d(
        x=sigmas, y=[0.6]*len(sigmas), z=log_hist_matrix[idx_06, :],
        mode='lines', line=dict(color='black', width=5), name='Limite 0.6'
    ))

    fig.update_layout(
        title=f'Margens ({qq} Quantis) - Escala Global ({GLOBAL_CMIN:.2f}-{GLOBAL_CMAX:.2f})',
        scene=dict(
            xaxis_title='Sigma',
            yaxis_title='Margem',
            zaxis_title='Frequência (Log)',
            camera=dict(eye=dict(x=-1.5, y=-1.5, z=1.5))
        ),
        margin=dict(l=0, r=20, b=0, t=40),
        legend=dict(x=0.02, y=0.90, xanchor='left', yanchor='top', bgcolor='rgba(255,255,255,0.6)')
    )

    # Salvar
    filename_base = f"superficie_conc_percentual_{PERCENTAGEM_DADOS}_{qq}_quantis"
    html_path = os.path.join(output_dir_graficos, f"{filename_base}.html")
    fig.write_html(html_path)

    try:
        png_path = os.path.join(output_dir_graficos, f"{filename_base}.png")
        fig.write_image(png_path, scale=3)
        png_path = os.path.join(output_dir_graficos, f"{filename_base}.pdf")
        fig.write_image(png_path)
    except:
        pass

print(f"\nProcesso finalizado! Limites usados: [{GLOBAL_CMIN}, {GLOBAL_CMAX}]")

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
from tqdm import tqdm

# --- CONFIGURAÇÃO ---
PERCENTAGEM_DADOS = 0.10  # 10% das amostras (Topo e Fundo do Rank)

# Carregar df_rotulos (Base para o cálculo da cor - Perturbação 0)
CAMINHO_ROTULOS = os.path.join('rotulos', 'rotulos_unificados_todos_quantis_todos_sigmas.csv')
try:
    df_rotulos = pd.read_csv(CAMINHO_ROTULOS, sep=';', decimal=',')
    # Filtra APENAS a perturbação 0 para o cálculo da cor
    df_rotulado_base = df_rotulos[df_rotulos['perturbacao'] == 0].copy()
except FileNotFoundError:
    print(f"ERRO CRÍTICO: Arquivo de rótulos não encontrado em {CAMINHO_ROTULOS}")
    df_rotulado_base = pd.DataFrame()

# Pasta de saída
output_dir_graficos = "graficos_superficie_soma_concordancia_percentual_dinamico"
os.makedirs(output_dir_graficos, exist_ok=True)
print(f"Gráficos serão salvos em: {output_dir_graficos}")

# --- LOOP PRINCIPAL ---
for qq in tqdm([2, 4, 8, 16], desc="Processando Quantis"):
# for qq in tqdm([4], desc="Processando Quantis"):

    # 1. Carregar Margens (Define a Geometria Z do gráfico - Todas as perturbações)
    FOLDER_PATH_PERT = os.path.join(os.getcwd(), os.path.join('matrizes_margens_perturbadas', f'{qq}_quantis'))
    filename_margens = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'
    path_margens = os.path.join(FOLDER_PATH_PERT, filename_margens)

    try:
        df_margens = pd.read_csv(path_margens, sep=';', decimal=',')
    except FileNotFoundError:
        print(f"\n[Pular qq={qq}] Arquivo de margens não encontrado.")
        continue

    # 2. Carregar Dados Brutos e ORDENAR POR RANK NSGA2
    file_path_dados = os.path.join(f'{qq}_quantis', f'dados_divididos_em_{qq}_quantis.csv')
    try:
        df_dados = pd.read_csv(file_path_dados, sep=';', decimal=',')

        # --- CORREÇÃO SOLICITADA: Ordenar explicitamente pelo Rank ---
        if 'nsga2_rank' in df_dados.columns:
            df_dados = df_dados.sort_values(by='nsga2_rank', ascending=True)
        else:
            print(f"Aviso: Coluna 'nsga2_rank' não encontrada para qq={qq}. Usando ordem do arquivo.")

    except FileNotFoundError:
        print(f"\n[Pular qq={qq}] Arquivo de dados divididos não encontrado.")
        continue

    # 3. Selecionar Top/Bottom 10% baseado no Rank
    total_linhas = len(df_dados)
    n_selecao = int(total_linhas * PERCENTAGEM_DADOS)
    n_selecao = max(1, n_selecao) # Garante pelo menos 1 amostra

    # Topo do Rank (Esperado Classe 0)
    df_first = df_dados.iloc[:n_selecao]
    ids_first = df_first['ID'].tolist()

    # Fundo do Rank (Esperado Classe 1)
    df_last = df_dados.iloc[-n_selecao:]
    ids_last = df_last['ID'].tolist()

    # Nomes das colunas de rótulo para esses IDs
    cols_rot_first = [f'rotulo_{int(i)}' for i in ids_first]
    cols_rot_last = [f'rotulo_{int(i)}' for i in ids_last]

    # 4. Calcular Métrica de Cor por Sigma (Apenas Perturbação 0)
    mapa_concordancia = {}
    sigmas = np.sort(df_margens['sigma'].unique())
    sigmas = np.round(sigmas, 2)

    for sigma in sigmas:
        # Busca os rótulos da perturbação 0 para este sigma
        linha_rotulo = df_rotulado_base[
            (df_rotulado_base['qq'] == qq) &
            (df_rotulado_base['sigma'] == sigma)
        ]

        if not linha_rotulo.empty and ids_first and ids_last:
            valid_cols_first = [c for c in cols_rot_first if c in linha_rotulo.columns]
            valid_cols_last = [c for c in cols_rot_last if c in linha_rotulo.columns]

            if valid_cols_first and valid_cols_last:
                # Taxa de Concordância INICIAL (Esperado 0)
                # Conta quantos zeros existem
                rotulos_first = linha_rotulo[valid_cols_first].values.flatten()
                taxa_first = (len(rotulos_first) - np.sum(rotulos_first)) / len(rotulos_first)

                # Taxa de Concordância FINAL (Esperado 1)
                # Conta quantos uns existem
                rotulos_last = linha_rotulo[valid_cols_last].values.flatten()
                taxa_last = np.sum(rotulos_last) / len(rotulos_last)

                # Soma das taxas (Ideal = 2.0)
                mapa_concordancia[sigma] = taxa_first + taxa_last
            else:
                mapa_concordancia[sigma] = 0.0
        else:
            mapa_concordancia[sigma] = 0.0

    # 5. Preparar Matrizes para Plotagem
    hist_data = []
    color_data = []
    bin_edges = None
    bins_config = np.linspace(0, 1, 51)

    for sigma in sigmas:
        # A. Geometria (Usa todas as perturbações para desenhar a "montanha")
        df_sigma = df_margens[df_margens['sigma'] == sigma]
        cols_m = [c for c in df_sigma.columns if c.startswith('margem_')]
        vals = df_sigma[cols_m].values.flatten()
        vals = vals[~np.isnan(vals)]

        hist, bins = np.histogram(vals, bins=bins_config)
        if bin_edges is None: bin_edges = bins
        hist_data.append(hist)

        # B. Cor (Usa apenas a métrica da Pert 0)
        conc_val = mapa_concordancia.get(round(sigma, 2), 0.0)
        color_data.append(np.full_like(hist, conc_val, dtype=float))

    if not hist_data: continue

    hist_matrix = np.array(hist_data).T
    concordancia_matrix = np.array(color_data).T
    log_hist_matrix = np.log10(hist_matrix + 1)
    y_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # --- Cálculo Dinâmico da Barra de Cores ---
    cmin_val = concordancia_matrix.min()
    cmax_val = concordancia_matrix.max()

    if cmin_val == cmax_val:
        cmax_val += 0.001

    # Limites de segurança visual [0, 2]
    cmin_val = max(0.0, cmin_val)
    cmax_val = min(2.0, cmax_val)

    # --- PLOTAGEM ---
    fig = go.Figure()

    fig.add_trace(go.Surface(
        x=sigmas,
        y=y_centers,
        z=log_hist_matrix,

        surfacecolor=concordancia_matrix,
        customdata=concordancia_matrix,

        colorscale='Jet',
        cmin=cmin_val,
        cmax=cmax_val,
        name='Distribuição',

        colorbar=dict(
            title=f'Soma Concordância (Pert 0)<br>(Top/Bot {PERCENTAGEM_DADOS*100:.0f}% Rank NSGA2)',
            thickness=25,
            len=0.75,
            x=0.85,
            xanchor='left',
            y=0.5,
            yanchor='middle'
        ),

        hovertemplate=(
            '<b>Sigma</b>: %{x:.2f}<br>' +
            '<b>Margem</b>: %{y:.3f}<br>' +
            '<b>Freq(Log)</b>: %{z:.2f}<br>' +
            '<b>Soma Conc.</b>: %{customdata:.4f}<extra></extra>'
        )
    ))

    # Linhas de Limite
    idx_02 = np.abs(y_centers - 0.2).argmin()
    fig.add_trace(go.Scatter3d(
        x=sigmas, y=[0.2]*len(sigmas), z=log_hist_matrix[idx_02, :],
        mode='lines', line=dict(color='white', width=5), name='Limite 0.2'
    ))

    idx_06 = np.abs(y_centers - 0.6).argmin()
    fig.add_trace(go.Scatter3d(
        x=sigmas, y=[0.6]*len(sigmas), z=log_hist_matrix[idx_06, :],
        mode='lines', line=dict(color='black', width=5), name='Limite 0.6'
    ))

    fig.update_layout(
        title=f'Margens ({qq} Quantis) - Cor: Concordância Pert 0 (Top/Bot {PERCENTAGEM_DADOS*100:.0f}% Rank)',
        scene=dict(
            xaxis_title='Sigma',
            yaxis_title='Margem',
            zaxis_title='Frequência (Log)',
            camera=dict(eye=dict(x=-1.5, y=-1.5, z=1.5))
        ),
        margin=dict(l=0, r=20, b=0, t=40),
        legend=dict(x=0.02, y=0.90, xanchor='left', yanchor='top', bgcolor='rgba(255,255,255,0.6)')
    )

    # Salvar
    filename_base = f"superficie_conc_percentual_{PERCENTAGEM_DADOS}_{qq}_quantis"
    html_path = os.path.join(output_dir_graficos, f"{filename_base}.html")
    fig.write_html(html_path)
    print(f"  -> Salvo: {os.path.basename(html_path)}")

    try:
        png_path = os.path.join(output_dir_graficos, f"{filename_base}.png")
        fig.write_image(png_path, scale=2)
        pdf_path = os.path.join(output_dir_graficos, f"{filename_base}.pdf")
        fig.write_image(pdf_path)
    except:
        pass

print("\nProcesso finalizado!")

# Gráfico de dispersão

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
from tqdm import tqdm


# --- Criação da pasta de saída para os gráficos ---
output_dir_graficos = "graficos_dispersao_log"
os.makedirs(output_dir_graficos, exist_ok=True)
print(f"Gráficos serão salvos no diretório: '{output_dir_graficos}'")


for qq in tqdm([2,4,8,16,32,64], desc="Gerando Gráficos por Quantil"):
    # --- Bloco de cálculo dos dados (Inalterado) ---
    FOLDER_PATH = os.path.join(os.getcwd(), os.path.join('matrizes_margens_perturbadas',f'{qq}_quantis') )
    filename = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'

    try:
        df = pd.read_csv(os.path.join(FOLDER_PATH, filename), sep=';', decimal=',')
    except FileNotFoundError:
        print(f"\nAVISO: Arquivo de margens para qq={qq} não encontrado. Pulando.")
        continue

    sigmas = np.round(np.arange(0.01, 0.31, 0.01), 2)
    hist_data = []
    bin_edges = None

    for sigma in sigmas:
        df_sigma = df[df['sigma'] == sigma]
        if df_dataset.shape[0] > 1:
            margens = df_sigma[[f'margem_{i}' for i in range(1,df_dataset.shape[0])]].values.flatten()
            margens = margens[~np.isnan(margens)]
            hist, bins = np.histogram(margens, bins=50, range=(0, 1))
            if bin_edges is None: bin_edges = bins
            hist_data.append(hist)

    if not hist_data:
        print(f"\nAVISO: Nenhum dado de margem processado para qq={qq}. Pulando gráfico.")
        continue

    hist_matrix = np.array(hist_data).T
    log_hist_matrix = np.log10(hist_matrix + 1)

    y_centers = (bin_edges[:-1] + bin_edges[1:])/2
    X_grid, Y_grid = np.meshgrid(sigmas, y_centers)
    x_points = X_grid.flatten()
    y_points = Y_grid.flatten()
    z_points = log_hist_matrix.flatten()
    mask = z_points > 0
    x_points, y_points, z_points = x_points[mask], y_points[mask], z_points[mask]
    colors = z_points

    # --- Construção do Gráfico (Inalterado) ---
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=x_points, y=y_points, z=z_points, mode='markers',
        marker=dict(size=3.5, color=colors, colorscale='Viridis', colorbar=dict(title='Frequência (Log10)'), showscale=True),
        name='Distribuição',
        hovertemplate='<b>Sigma</b>: %{x:.2f}<br><b>Margem</b>: %{y:.3f}<br><b>Frequência (log10)</b>: %{z:.2f}<extra></extra>'
    ))
    fig.add_trace(go.Scatter3d(x=sigmas, y=[0.2]*len(sigmas), z=np.log10(hist_matrix[np.abs(y_centers - 0.2).argmin(), :] + 1), mode='lines', line=dict(color='red', width=4), name='Limite Fraca (0.2)', hoverinfo='skip'))
    fig.add_trace(go.Scatter3d(x=sigmas, y=[0.6]*len(sigmas), z=np.log10(hist_matrix[np.abs(y_centers - 0.6).argmin(), :] + 1), mode='lines', line=dict(color='green', width=4), name='Limite Forte (0.6)', hoverinfo='skip'))

    # --- Layout e Salvamento (com a alteração) ---

    fig.update_layout(
        title=f'Dispersão 3D das Margens por Sigma - {qq} Quantis (Escala Log)',
        # ALTERAÇÃO PRINCIPAL AQUI: Adiciona o template de fundo branco
        template='plotly_white',
        scene=dict(
            xaxis_title='Sigma',
            yaxis_title='Margem',
            zaxis_title='Frequência (Log)',
            camera=dict(eye=dict(x=-0.82, y=-2.04, z=0.89))
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        legend=dict(x=0.05, y=0.95, yanchor="top", xanchor="left", bgcolor='rgba(255, 255, 255, 0.7)', bordercolor='black', borderwidth=1)
    )

    # (Seção de salvamento permanece a mesma)
    filename_base = f"dispersao_log_margens_{qq}_quantis"
    html_path = os.path.join(output_dir_graficos, f"{filename_base}.html")
    fig.write_html(html_path)
    print(f"\n  -> Gráfico para qq={qq} salvo em: {os.path.basename(html_path)}")
    try:
        png_path = os.path.join(output_dir_graficos, f"{filename_base}.png")
        fig.write_image(png_path, scale=3)
        print(f"  -> Gráfico para qq={qq} salvo em: {os.path.basename(png_path)}")
        pdf_path = os.path.join(output_dir_graficos, f"{filename_base}.pdf")
        fig.write_image(pdf_path)
        print(f"  -> Gráfico para qq={qq} salvo em: {os.path.basename(pdf_path)}")
    except ValueError:
        print(f"\nAVISO: Não foi possível salvar em PNG/PDF. A biblioteca 'kaleido' é necessária.")

    # fig.show()

print("\n\nProcesso concluído!")

## Histograma para cada quantil para um valor especifico de sigma

In [ ]:
sigma = 0.02
# for qq in [2,4,8,16]:
for qq in [2]:

    # qq = 16
    # FOLDER_PATH = f'G:\\Meu Drive\\Doutorado\\analiseResultadosRegrasAssociacao\\matrizes_margens_perturbadas\\{qq}_quantis'
    FOLDER_PATH = os.path.join(os.getcwd(), os.path.join('matrizes_margens_perturbadas',f'{qq}_quantis') )
    filename = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'
    df = pd.read_csv(os.path.join(FOLDER_PATH, filename), sep=';', decimal=',')

    df = df[df['sigma'] == 0.02]

    # Derretendo o DataFrame para formato "long"
    df_long = df.melt(
        id_vars=['sigma','perturbacao'],
        value_vars=[f'margem_{i}' for i in range(1,df_dataset.shape[0])],
        var_name='amostra',
        value_name='margem'
    )

    def categoria(m):
        if m < 0.2:   return 'Fraca'
        if m > 0.6:   return 'Forte'
        return 'Média'

    df_long['categoria'] = df_long['margem'].map(categoria)

    # Gráfico de barras de proporção usando Plotly Express
    fig = px.histogram(
        df_long,
        x='categoria',
        color='categoria',
        category_orders={'categoria': ['Fraca', 'Média', 'Forte']},
        color_discrete_sequence=px.colors.qualitative.Set2,
        text_auto=True
    )

    fig.update_layout(
        title=f'Contagem de estado de confiança (todas perturbações) - {qq} quantis',
        xaxis_title='Categoria',
        yaxis_title='Contagem',
        showlegend=False
    )

    fig.show()

# Grafico das comunidades e amostras selecionadas de cada uma

In [ ]:
df_dataset.columns

In [ ]:
df_all.columns

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import os
import pickle

# --- PRÉ-REQUISITOS (ASSUMINDO QUE AS VARIÁVEIS JÁ EXISTEM) ---
# df_dataset: O DataFrame com as coordenadas e IDs.

# --- LOOP PRINCIPAL ---
for qq in [2,4,8,16]:
    pickle_path = f'dados_rotulados_{qq}_quantis.pkl'
    try:
        with open(pickle_path, 'rb') as f:
            dados_carregados = pickle.load(f)
            indices_grupos = dados_carregados[6]
            labeled_first_ids = set(dados_carregados[1])
            labeled_last_ids = set(dados_carregados[2])
        print(f"Dados de '{pickle_path}' carregados com sucesso.")
    except (FileNotFoundError, IndexError, TypeError) as e:
        print(f"ERRO: Não foi possível carregar dados de grupos de '{pickle_path}'. Erro: {e}")
        indices_grupos = {'FIRST': pd.DataFrame(columns=['original_ID', 'comunidade_id']),
                          'LAST': pd.DataFrame(columns=['original_ID', 'comunidade_id'])}
        labeled_first_ids, labeled_last_ids = set(), set()

    # --- 1. PREPARAÇÃO DOS DADOS ---
    df_map_first = indices_grupos.get('FIRST', pd.DataFrame())
    df_map_last = indices_grupos.get('LAST', pd.DataFrame())

    # Garante que df_dataset tem a coluna ID para o merge
    if 'ID' not in df_dataset.columns:
        df_dataset['ID'] = df_dataset.index

    df_plot_first = df_dataset.merge(df_map_first, left_on='ID', right_on='original_ID')
    df_plot_last = df_dataset.merge(df_map_last, left_on='ID', right_on='original_ID')

    df_highlight_first = df_dataset[df_dataset['ID'].isin(labeled_first_ids)]
    df_highlight_last = df_dataset[df_dataset['ID'].isin(labeled_last_ids)]

    paleta_first = px.colors.qualitative.Plotly
    paleta_last = px.colors.qualitative.T10

    comunidades_first = sorted(df_plot_first['comunidade_id'].unique())
    comunidades_last = sorted(df_plot_last['comunidade_id'].unique())
    color_map_first = {comp_id: paleta_first[i % len(paleta_first)] for i, comp_id in enumerate(comunidades_first)}
    color_map_last = {comp_id: paleta_last[i % len(paleta_last)] for i, comp_id in enumerate(comunidades_last)}

    # --- SEÇÃO 2 - GRÁFICO APENAS PARA O GRUPO FIRST ---
    print("\nGerando o gráfico 3D para o grupo FIRST...")
    fig_first = go.Figure()
    if not df_plot_first.empty:
        for comp_id, color in color_map_first.items():
            df_subset = df_plot_first[df_plot_first['comunidade_id'] == comp_id]
            fig_first.add_trace(go.Scatter3d(
                # ALTERAÇÃO AQUI: Variáveis Específicas
                x=df_subset['numero_feiras_livres'],
                y=df_subset['ipvs'],
                z=df_subset['obj_in_natura'],
                mode='markers', marker=dict(size=5, color=color, symbol='circle'),
                text=[f"ID: {row.ID}<br>Comunidade: {row.comunidade_id}" for _, row in df_subset.iterrows()],
                hoverinfo='text', name=f'FIRST - Com. {comp_id}'
            ))
        if not df_highlight_first.empty:
            fig_first.add_trace(go.Scatter3d(
                # ALTERAÇÃO AQUI: Variáveis Específicas
                x=df_highlight_first['numero_feiras_livres'],
                y=df_highlight_first['ipvs'],
                z=df_highlight_first['obj_in_natura'],
                mode='markers',
                marker=dict(size=8, color='gold', symbol='cross', line=dict(width=2, color='black')),
                name='Amostras Rotuladas (FIRST)'
            ))
    fig_first.update_layout(
        title=f'<b>Visualização das Comunidades - Grupo FIRST (qq={qq})</b>',
        template='plotly_white',
        scene=dict(
            xaxis_title='Nº Feiras Livres',
            yaxis_title='IPVS',
            zaxis_title='Obj. In Natura',
            aspectmode='cube',
            camera=dict(eye=dict(x= 2.136, y= -1.450, z= 0.213))
        ),
        legend_title_text='<b>Comunidades FIRST</b>', margin=dict(l=0, r=0, b=0, t=40)
    )

    # --- SEÇÃO 3 - GRÁFICO APENAS PARA O GRUPO LAST ---
    print("\nGerando o gráfico 3D para o grupo LAST...")
    fig_last = go.Figure()
    if not df_plot_last.empty:
        for comp_id, color in color_map_last.items():
            df_subset = df_plot_last[df_plot_last['comunidade_id'] == comp_id]
            fig_last.add_trace(go.Scatter3d(
                # ALTERAÇÃO AQUI: Variáveis Específicas
                x=df_subset['numero_feiras_livres'],
                y=df_subset['ipvs'],
                z=df_subset['obj_in_natura'],
                mode='markers',
                marker=dict(size=5, color=color, symbol='circle'),
                text=[f"ID: {row.ID}<br>Comunidade: {row.comunidade_id}" for _, row in df_subset.iterrows()],
                hoverinfo='text', name=f'LAST - Com. {comp_id}'
            ))
        if not df_highlight_last.empty:
            fig_last.add_trace(go.Scatter3d(
                # ALTERAÇÃO AQUI: Variáveis Específicas
                x=df_highlight_last['numero_feiras_livres'],
                y=df_highlight_last['ipvs'],
                z=df_highlight_last['obj_in_natura'],
                mode='markers',
                marker=dict(size=8, color='cyan', symbol='cross', line=dict(width=2, color='black')),
                name='Amostras Rotuladas (LAST)'
            ))
    fig_last.update_layout(
        title=f'<b>Visualização das Comunidades - Grupo LAST (qq={qq})</b>',
        template='plotly_white',
        scene=dict(
            xaxis_title='Nº Feiras Livres',
            yaxis_title='IPVS',
            zaxis_title='Obj. In Natura',
            aspectmode='cube',
            camera=dict(eye=dict(x= 2.136, y= -1.450, z= 0.213))
        ),
        legend_title_text='<b>Comunidades LAST</b>', margin=dict(l=0, r=0, b=0, t=40)
    )

    # --- SEÇÃO 4 - SALVAMENTO DOS GRÁFICOS ---
    output_dir_graficos = "graficos_comunidades_separados"
    os.makedirs(output_dir_graficos, exist_ok=True)
    print(f"\n--- Salvando os gráficos em '{output_dir_graficos}' ---")

    try:
        print("\nSalvando gráficos para o grupo FIRST...")
        fig_first.write_html(os.path.join(output_dir_graficos, f"comunidades_first_qq{qq}.html"))
        fig_first.write_image(os.path.join(output_dir_graficos, f"comunidades_first_qq{qq}.png"), scale=3, width=1000, height=800)
        fig_first.write_image(os.path.join(output_dir_graficos, f"comunidades_first_qq{qq}.pdf"), width=1000, height=800)
        print(" -> Gráficos do grupo FIRST salvos com sucesso.")
    except ValueError:
        print(" -> AVISO: Não foi possível salvar em PNG/PDF. Instale 'kaleido'.")

    try:
        print("\nSalvando gráficos para o grupo LAST...")
        fig_last.write_html(os.path.join(output_dir_graficos, f"comunidades_last_qq{qq}.html"))
        fig_last.write_image(os.path.join(output_dir_graficos, f"comunidades_last_qq{qq}.png"), scale=3, width=1000, height=800)
        fig_last.write_image(os.path.join(output_dir_graficos, f"comunidades_last_qq{qq}.pdf"), width=1000, height=800)
        print(" -> Gráficos do grupo LAST salvos com sucesso.")
    except ValueError:
        pass

# Gráficos de linhas e dispersão para as margens para um conjunto de sigma

In [ ]:
df_all.head(20)

In [ ]:
df.head()

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from tqdm import tqdm
import pickle

# --- PRÉ-REQUISITO ---
if 'df_all' not in locals():
    print("ERRO: df_all não encontrado na memória.")

# --- PARÂMETROS ---
sigmas_a_rodar = [0.01, 0.03, 0.05, 0.07, 0.10, 0.13, 0.16, 0.20, 0.23, 0.26, 0.30]
quantis_map = [2, 4, 8, 16]

output_dir_base = "graficos_lineplot_com_confianca"
os.makedirs(output_dir_base, exist_ok=True)

for sigma in tqdm(sigmas_a_rodar, desc="Processando Sigmas"):
    subfolder_name = f"sigma_{sigma:.2f}"
    output_dir_sigma = os.path.join(output_dir_base, subfolder_name)
    os.makedirs(output_dir_sigma, exist_ok=True)
    output_dir_html = os.path.join(output_dir_sigma, "html")
    os.makedirs(output_dir_html, exist_ok=True)

    for qq in tqdm(quantis_map, desc=f"Quantis (Sigma={sigma:.2f})", leave=False):
        # 1. CARREGAMENTO (Inalterado)
        FOLDER_PATH = os.path.join(os.getcwd(), 'matrizes_margens_perturbadas', f'{qq}_quantis')
        filename = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'

        try:
            df = pd.read_csv(os.path.join(FOLDER_PATH, filename), sep=';', decimal=',')
        except FileNotFoundError:
            continue

        df = df[df['sigma'] == sigma]
        if df.empty: continue

        labeled_first = set(); labeled_last = set()
        pickle_path = f'dados_rotulados_{qq}_quantis.pkl'
        try:
            with open(pickle_path, 'rb') as f:
                dados_carregados = pickle.load(f)
                labeled_first = set(dados_carregados[1])
                labeled_last = set(dados_carregados[2])
        except Exception: pass

        # 2. PREPARAÇÃO E AGREGAÇÃO (Cálculo do IC é feito aqui, por ID)
        df_long = df.melt(
            id_vars=['sigma','perturbacao'],
            value_vars=[col for col in df.columns if col.startswith('margem_')],
            var_name='amostra', value_name='margem'
        )
        df_long['amostra_id'] = df_long['amostra'].str.replace('margem_', '').astype(int)

        # Classificação dos grupos
        conditions = [df_long['amostra_id'].isin(labeled_first), df_long['amostra_id'].isin(labeled_last)]
        choices = ['First Rotulado', 'Last Rotulado']
        df_long['grupo_destaque'] = np.select(conditions, choices, default='Outros')

        # Agregação (Ainda independente da ordem de plotagem)
        df_agregado = df_long.groupby(['amostra_id', 'grupo_destaque'], as_index=False).agg(
            media_margem=('margem', 'mean'),
            std_margem=('margem', 'std'),
            n_observacoes=('margem', 'count')
        )

        # Cálculo Estatístico do IC (Mantendo integridade matemática)
        z_score = 1.96
        df_agregado['erro_padrao'] = df_agregado['std_margem'] / np.sqrt(df_agregado['n_observacoes'])
        df_agregado['ci_superior'] = df_agregado['media_margem'] + (z_score * df_agregado['erro_padrao'])
        df_agregado['ci_inferior'] = df_agregado['media_margem'] - (z_score * df_agregado['erro_padrao'])

        # 3. REORDENAÇÃO PELO RANK (Apenas para visualização)
        df_agregado = df_agregado.merge(df_all[['ID', 'nsga2_rank']], left_on='amostra_id', right_on='ID', how='inner')

        # Ordena pelo Rank
        df_agregado.sort_values(by=['nsga2_rank', 'amostra_id'], inplace=True)

        # Cria eixo X sequencial contínuo
        df_agregado['x_plot_sequence'] = range(len(df_agregado))

        # 4. PLOTAGEM (CORRIGIDA PARA MOSTRAR O IC)
        fig = go.Figure()

        # CAMADA 1: Intervalo de Confiança (Plotado como um único bloco contínuo cinza)
        # Isso garante que ele apareça independente da mistura de grupos
        fig.add_trace(go.Scatter(
            x=df_agregado['x_plot_sequence'],
            y=df_agregado['ci_superior'],
            mode='lines', line=dict(width=0),
            showlegend=False, hoverinfo='skip'
        ))

        fig.add_trace(go.Scatter(
            x=df_agregado['x_plot_sequence'],
            y=df_agregado['ci_inferior'],
            mode='lines', line=dict(width=0),
            fill='tonexty',
            fillcolor='rgba(150, 150, 150, 0.2)', # Cinza translúcido para o IC
            name='IC 95% (Global)',
            showlegend=True
        ))

        # CAMADA 2: Linha da Média (Contínua e fina para guiar o olhar)
        fig.add_trace(go.Scatter(
            x=df_agregado['x_plot_sequence'],
            y=df_agregado['media_margem'],
            mode='lines',
            line=dict(color='rgba(100,100,100,0.5)', width=1),
            showlegend=False, hoverinfo='skip'
        ))

        # CAMADA 3: Marcadores Coloridos por Grupo (Plotados por cima)
        color_map = {'First Rotulado': 'green', 'Last Rotulado': 'red', 'Outros': 'darkgray'}
        grupos_ordenados = ['Outros', 'First Rotulado', 'Last Rotulado']

        for grupo in grupos_ordenados:
            df_grupo = df_agregado[df_agregado['grupo_destaque'] == grupo]
            if df_grupo.empty: continue

            fig.add_trace(go.Scatter(
                x=df_grupo['x_plot_sequence'],
                y=df_grupo['media_margem'],
                mode='markers', # Apenas bolinhas
                name=grupo,
                marker=dict(color=color_map[grupo], size=4),
                # Hover personalizado completo
                customdata=np.stack((df_grupo['nsga2_rank'], df_grupo['amostra_id'], df_grupo['ci_inferior'], df_grupo['ci_superior']), axis=-1),
                hovertemplate=(
                    "<b>Grupo:</b> " + grupo + "<br>" +
                    "<b>Rank:</b> %{customdata[0]}<br>" +
                    "<b>ID:</b> %{customdata[1]}<br>" +
                    "<b>Média:</b> %{y:.3f}<br>" +
                    "<b>IC:</b> [%{customdata[2]:.3f}, %{customdata[3]:.3f}]<extra></extra>"
                )
            ))

        # Linhas de limite
        fig.add_hline(y=0.2, line_dash='dash', line_color='red', annotation_text='0.2')
        fig.add_hline(y=0.6, line_dash='dash', line_color='green', annotation_text='0.6')

        fig.update_layout(
            title=f'<b>Margens por Rank NSGA-II (com IC) <br> {qq} Quantis (Sigma: {sigma:.2f})</b>',
            xaxis_title='Amostras Ordenadas por Rank (Melhor -> Pior)',
            yaxis_title='Margem Média',
            template='plotly_white',
            height=600, width=1200,
            margin=dict(l=50, r=50, b=100, t=80),
            xaxis=dict(showticklabels=False)
        )

        # Salvamento
        filename_base = f"lineplot_{qq}_quantis"
        html_path = os.path.join(output_dir_html, f"{filename_base}.html")
        fig.write_html(html_path)

        try:
            png_path = os.path.join(output_dir_sigma, f"{filename_base}.png")
            fig.write_image(png_path, scale=3)
            pdf_path = os.path.join(output_dir_sigma, f"{filename_base}.pdf")
            fig.write_image(pdf_path)
        except ValueError:
            pass

print("\n\nProcesso concluído!")

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from tqdm import tqdm
import pickle

# --- PRÉ-REQUISITO ---
if 'df_all' not in locals():
    print("ERRO: df_all não encontrado na memória. Carregue-o antes de prosseguir.")

# --- PARÂMETROS ---
# sigmas_a_rodar = [0.01, 0.03, 0.05, 0.07, 0.10, 0.13, 0.16, 0.20, 0.23, 0.26, 0.30]
sigmas_a_rodar = [0.01, 0.03, 0.05, 0.07, 0.10, 0.13, 0.18, 0.20]
quantis_map = [2, 4, 8, 16]

# ALTERADO: Nome da pasta de saída
output_dir_base = "graficos_scatterplot_com_confianca"
os.makedirs(output_dir_base, exist_ok=True)
print(f"Gráficos de dispersão serão salvos no diretório base: '{output_dir_base}'")

for sigma in tqdm(sigmas_a_rodar, desc="Processando Sigmas"):
    subfolder_name = f"sigma_{sigma:.2f}"
    output_dir_sigma = os.path.join(output_dir_base, subfolder_name)
    os.makedirs(output_dir_sigma, exist_ok=True)
    output_dir_html = os.path.join(output_dir_sigma, "html")
    os.makedirs(output_dir_html, exist_ok=True)

    for qq in tqdm(quantis_map, desc=f"Quantis (Sigma={sigma:.2f})", leave=False):
        # 1. CARREGAMENTO
        FOLDER_PATH = os.path.join(os.getcwd(), 'matrizes_margens_perturbadas', f'{qq}_quantis')
        filename = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'

        try:
            df = pd.read_csv(os.path.join(FOLDER_PATH, filename), sep=';', decimal=',')
        except FileNotFoundError:
            continue

        df = df[df['sigma'] == sigma]
        if df.empty: continue

        labeled_first = set(); labeled_last = set()
        pickle_path = f'dados_rotulados_{qq}_quantis.pkl'
        try:
            with open(pickle_path, 'rb') as f:
                dados_carregados = pickle.load(f)
                labeled_first = set(dados_carregados[1])
                labeled_last = set(dados_carregados[2])
        except Exception: pass

        # 2. PREPARAÇÃO E AGREGAÇÃO
        df_long = df.melt(
            id_vars=['sigma','perturbacao'],
            value_vars=[col for col in df.columns if col.startswith('margem_')],
            var_name='amostra', value_name='margem'
        )
        df_long['amostra_id'] = df_long['amostra'].str.replace('margem_', '').astype(int)

        conditions = [df_long['amostra_id'].isin(labeled_first), df_long['amostra_id'].isin(labeled_last)]
        choices = ['First Rotulado', 'Last Rotulado']
        df_long['grupo_destaque'] = np.select(conditions, choices, default='Outros')

        df_agregado = df_long.groupby(['amostra_id', 'grupo_destaque'], as_index=False).agg(
            media_margem=('margem', 'mean'),
            std_margem=('margem', 'std'),
            n_observacoes=('margem', 'count')
        )

        z_score = 1.96
        df_agregado['erro_padrao'] = df_agregado['std_margem'] / np.sqrt(df_agregado['n_observacoes'])
        df_agregado['ci_superior'] = df_agregado['media_margem'] + (z_score * df_agregado['erro_padrao'])
        df_agregado['ci_inferior'] = df_agregado['media_margem'] - (z_score * df_agregado['erro_padrao'])

        # 3. REORDENAÇÃO PELO RANK
        df_agregado = df_agregado.merge(df_all[['ID', 'nsga2_rank']], left_on='amostra_id', right_on='ID', how='inner')
        df_agregado.sort_values(by=['nsga2_rank', 'amostra_id'], inplace=True)
        df_agregado['x_plot_sequence'] = range(len(df_agregado))

        # 4. PLOTAGEM (TIPO SCATTER)
        fig = go.Figure()

        # CAMADA 1: Intervalo de Confiança (Background Ribbon)
        # Mantemos isso como linhas invisíveis com preenchimento para criar a "faixa" de confiança
        fig.add_trace(go.Scatter(
            x=df_agregado['x_plot_sequence'],
            y=df_agregado['ci_superior'],
            mode='lines', line=dict(width=0),
            showlegend=False, hoverinfo='skip'
        ))

        fig.add_trace(go.Scatter(
            x=df_agregado['x_plot_sequence'],
            y=df_agregado['ci_inferior'],
            mode='lines', line=dict(width=0),
            fill='tonexty',
            fillcolor='rgba(150, 150, 150, 0.2)',
            name='IC 95%',
            showlegend=True
        ))

        # CAMADA 2 (REMOVIDA): A linha contínua de média foi removida para caracterizar o scatterplot.

        # CAMADA 3: Marcadores (Pontos)
        color_map = {'First Rotulado': 'green', 'Last Rotulado': 'red', 'Outros': 'darkgray'}
        grupos_ordenados = ['Outros', 'First Rotulado', 'Last Rotulado']

        for grupo in grupos_ordenados:
            df_grupo = df_agregado[df_agregado['grupo_destaque'] == grupo]
            if df_grupo.empty: continue

            fig.add_trace(go.Scatter(
                x=df_grupo['x_plot_sequence'],
                y=df_grupo['media_margem'],
                mode='markers', # Modo explícito de marcadores (scatterplot)
                name=grupo,
                marker=dict(color=color_map[grupo], size=4),
                customdata=np.stack((df_grupo['nsga2_rank'], df_grupo['amostra_id'], df_grupo['ci_inferior'], df_grupo['ci_superior']), axis=-1),
                hovertemplate=(
                    "<b>Grupo:</b> " + grupo + "<br>" +
                    "<b>Rank:</b> %{customdata[0]}<br>" +
                    "<b>ID:</b> %{customdata[1]}<br>" +
                    "<b>Média:</b> %{y:.3f}<br>" +
                    "<b>IC:</b> [%{customdata[2]:.3f}, %{customdata[3]:.3f}]<extra></extra>"
                )
            ))

        # Linhas de limite
        fig.add_hline(y=0.2, line_dash='dash', line_color='red', annotation_text='0.2')
        fig.add_hline(y=0.6, line_dash='dash', line_color='green', annotation_text='0.6')

        # Layout atualizado para Scatterplot
        fig.update_layout(
            # title=f'<b>Dispersão das Margens por Rank NSGA-II <br> {qq} Quantis (Sigma: {sigma:.2f})</b>',
            xaxis_title='Amostras Ranqueadas pelo NDS',
            showlegend=False,
            yaxis_title='Margem Média',
            template='plotly_white',
            height=600, width=1200,
            margin=dict(l=50, r=50, b=100, t=80),
            xaxis=dict(showticklabels=False)
        )

        # Salvamento com novo nome
        filename_base = f"scatterplot_{qq}_quantis"
        html_path = os.path.join(output_dir_html, f"{filename_base}.html")
        fig.write_html(html_path)

        try:
            # png_path = os.path.join(output_dir_sigma, f"{filename_base}.png")
            # fig.write_image(png_path, scale=3)
            pdf_path = os.path.join(output_dir_sigma, f"{filename_base}.pdf")
            fig.write_image(pdf_path, scale=5)
        except ValueError:
            pass

print("\n\nProcesso concluído!")

# Scatterplot sem confianca

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from tqdm import tqdm
import pickle

# --- PRÉ-REQUISITO ---
if 'df_all' not in locals():
    print("ERRO: df_all não encontrado na memória. Carregue-o antes de prosseguir.")

# --- PARÂMETROS ---
sigmas_a_rodar = [0.01, 0.03, 0.05, 0.07, 0.10, 0.13, 0.18, 0.20]
quantis_map = [2, 4, 8, 16]

# ALTERADO: Nome da pasta de saída para refletir a ausência de IC
output_dir_base = "graficos_scatterplot_sem_confianca"
os.makedirs(output_dir_base, exist_ok=True)
print(f"Gráficos de dispersão serão salvos no diretório base: '{output_dir_base}'")

for sigma in tqdm(sigmas_a_rodar, desc="Processando Sigmas"):
    subfolder_name = f"sigma_{sigma:.2f}"
    output_dir_sigma = os.path.join(output_dir_base, subfolder_name)
    os.makedirs(output_dir_sigma, exist_ok=True)
    output_dir_html = os.path.join(output_dir_sigma, "html")
    os.makedirs(output_dir_html, exist_ok=True)

    for qq in tqdm(quantis_map, desc=f"Quantis (Sigma={sigma:.2f})", leave=False):
        # 1. CARREGAMENTO
        FOLDER_PATH = os.path.join(os.getcwd(), 'matrizes_margens_perturbadas', f'{qq}_quantis')
        filename = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'

        try:
            df = pd.read_csv(os.path.join(FOLDER_PATH, filename), sep=';', decimal=',')
        except FileNotFoundError:
            continue

        df = df[df['sigma'] == sigma]
        if df.empty: continue

        labeled_first = set(); labeled_last = set()
        pickle_path = f'dados_rotulados_{qq}_quantis.pkl'
        try:
            with open(pickle_path, 'rb') as f:
                dados_carregados = pickle.load(f)
                labeled_first = set(dados_carregados[1])
                labeled_last = set(dados_carregados[2])
        except Exception: pass

        # 2. PREPARAÇÃO E AGREGAÇÃO (Removido cálculo de STD e Erro Padrão)
        df_long = df.melt(
            id_vars=['sigma','perturbacao'],
            value_vars=[col for col in df.columns if col.startswith('margem_')],
            var_name='amostra', value_name='margem'
        )
        df_long['amostra_id'] = df_long['amostra'].str.replace('margem_', '').astype(int)

        conditions = [df_long['amostra_id'].isin(labeled_first), df_long['amostra_id'].isin(labeled_last)]
        choices = ['First Rotulado', 'Last Rotulado']
        df_long['grupo_destaque'] = np.select(conditions, choices, default='Outros')

        # [cite_start]Agregação apenas da média da margem de confiança (M)
        df_agregado = df_long.groupby(['amostra_id', 'grupo_destaque'], as_index=False).agg(
            media_margem=('margem', 'mean')
        )

        # [cite_start]3. REORDENAÇÃO PELO RANK (Baseado na ordenação NDS) [cite: 9, 33]
        df_agregado = df_agregado.merge(df_all[['ID', 'nsga2_rank']], left_on='amostra_id', right_on='ID', how='inner')
        df_agregado.sort_values(by=['nsga2_rank', 'amostra_id'], inplace=True)
        df_agregado['x_plot_sequence'] = range(len(df_agregado))

        # 4. PLOTAGEM (SCATTER SIMPLES)
        fig = go.Figure()

        # CAMADA DE MARCADORES (Pontos)
        color_map = {'First Rotulado': 'green', 'Last Rotulado': 'red', 'Outros': 'darkgray'}
        grupos_ordenados = ['Outros', 'First Rotulado', 'Last Rotulado']

        for grupo in grupos_ordenados:
            df_grupo = df_agregado[df_agregado['grupo_destaque'] == grupo]
            if df_grupo.empty: continue

            fig.add_trace(go.Scatter(
                x=df_grupo['x_plot_sequence'],
                y=df_grupo['media_margem'],
                mode='markers',
                name=grupo,
                marker=dict(color=color_map[grupo], size=4),
                # Customdata simplificado para remover o IC do hover
                customdata=np.stack((df_grupo['nsga2_rank'], df_grupo['amostra_id']), axis=-1),
                hovertemplate=(
                    "<b>Grupo:</b> " + grupo + "<br>" +
                    "<b>Rank:</b> %{customdata[0]}<br>" +
                    "<b>ID:</b> %{customdata[1]}<br>" +
                    "<b>Média:</b> %{y:.3f}<extra></extra>"
                )
            ))

        # [cite_start]Linhas de limite de estabilidade [cite: 14]
        # fig.add_hline(y=0.2, line_dash='dash', line_color='red', annotation_text='0.2')
        # fig.add_hline(y=0.6, line_dash='dash', line_color='green', annotation_text='0.6')

        fig.update_layout(
            xaxis_title='Samples Ranked by NDS',
            showlegend=False,
            yaxis_title='Mean Margin',
            template='plotly_white',
            height=600, width=1200,
            margin=dict(l=50, r=50, b=100, t=80),
            xaxis=dict(showticklabels=False)
        )

        # Salvamento
        filename_base = f"scatterplot_{qq}_quantis"
        html_path = os.path.join(output_dir_html, f"{filename_base}.html")
        fig.write_html(html_path)

        try:
            pdf_path = os.path.join(output_dir_sigma, f"{filename_base}.pdf")
            fig.write_image(pdf_path, scale=5)
        except ValueError:
            pass

print("\n\nProcesso concluído!")

## Boxplot para as pertubacoes de cada amostra para um valor de sigma

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from tqdm import tqdm
import pickle

# --- PRÉ-REQUISITO ---
if 'df_all' not in locals():
    print("ERRO: df_all não encontrado na memória. Carregue-o antes de prosseguir.")
    # Exemplo: df_all = pd.read_csv('df_all.csv', sep=';', decimal=',')

# --- PARÂMETROS ---
# sigmas_a_rodar = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
sigmas_a_rodar = [0.01, 0.03, 0.05, 0.07, 0.10, 0.13, 0.16, 0.20, 0.23, 0.26, 0.30]
quantis_map = [2, 4, 8, 16]

# Criação da pasta de saída (nome atualizado para refletir o ranqueamento)
output_dir_base = "graficos_boxplot_destacados_por_sigma"
os.makedirs(output_dir_base, exist_ok=True)
print(f"Gráficos de boxplot serão salvos no diretório base: '{output_dir_base}'")

# --- LOOP PRINCIPAL SOBRE OS VALORES DE SIGMA ---
for sigma in tqdm(sigmas_a_rodar, desc="Processando Sigmas"):

    subfolder_name = f"sigma_{sigma:.2f}"
    output_dir_sigma = os.path.join(output_dir_base, subfolder_name)
    os.makedirs(output_dir_sigma, exist_ok=True)

    # --- Loop interno sobre os valores de quantil (qq) ---
    for qq in tqdm(quantis_map, desc=f"Quantis (Sigma={sigma:.2f})", leave=False):
        # Carregamento dos dados de margens
        FOLDER_PATH = os.path.join(os.getcwd(), 'matrizes_margens_perturbadas', f'{qq}_quantis')
        filename = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'

        try:
            df = pd.read_csv(os.path.join(FOLDER_PATH, filename), sep=';', decimal=',')
        except FileNotFoundError:
            continue

        df = df[df['sigma'] == sigma]
        if df.empty: continue

        # Carregamento dos IDs das amostras rotuladas
        labeled_first = set(); labeled_last = set()
        pickle_path = f'dados_rotulados_{qq}_quantis.pkl'
        try:
            with open(pickle_path, 'rb') as f:
                dados_carregados = pickle.load(f)
                labeled_first = set(dados_carregados[1])
                labeled_last = set(dados_carregados[2])
        except Exception: pass

        # Preparação do DataFrame Longo
        df_long = df.melt(
            id_vars=['sigma','perturbacao'],
            value_vars=[col for col in df.columns if col.startswith('margem_')],
            var_name='amostra', value_name='margem'
        )
        df_long['amostra_id'] = df_long['amostra'].str.replace('margem_', '').astype(int)

        # Classificação dos grupos
        conditions = [df_long['amostra_id'].isin(labeled_first), df_long['amostra_id'].isin(labeled_last)]
        choices = ['First Rotulado', 'Last Rotulado']
        df_long['grupo_destaque'] = np.select(conditions, choices, default='Outros')

        # --- ALTERAÇÃO CRÍTICA: INSERÇÃO E ORDENAÇÃO PELO RANK ---

        # 1. Merge com df_all para trazer o 'nsga2_rank'
        # O df_long ficará com uma nova coluna 'nsga2_rank'
        df_long = df_long.merge(df_all[['ID', 'nsga2_rank']], left_on='amostra_id', right_on='ID', how='inner')

        # 2. Ordenar o DataFrame pelo Rank e depois pelo ID (para desempate consistente)
        df_long.sort_values(by=['nsga2_rank', 'amostra_id'], inplace=True)

        # 3. Capturar a ordem exata da coluna 'amostra' após a ordenação
        # Isso cria a lista que o Plotly vai respeitar no eixo X
        ordem_eixo_x_rankeada = df_long['amostra'].unique().tolist()

        # ---------------------------------------------------------

        # Plotagem
        color_map = {'First Rotulado': 'green', 'Last Rotulado': 'red', 'Outros': 'lightgray'}

        fig = px.box(
            df_long,
            x='amostra',
            y='margem',
            color='grupo_destaque',
            color_discrete_map=color_map,
            # AQUI ESTÁ A MÁGICA: Forçamos a ordem do eixo X a ser a lista rankeada
            category_orders={
                'amostra': ordem_eixo_x_rankeada,
                'grupo_destaque': ['Outros', 'First Rotulado', 'Last Rotulado']
            },
            points='outliers',
            # Adicionando dados ao hover para ver o rank
            hover_data=['nsga2_rank', 'amostra_id']
        )

        fig.add_hline(y=0.2, line_dash='dash', line_color='red', annotation_text='0.2')
        fig.add_hline(y=0.6, line_dash='dash', line_color='green', annotation_text='0.6')

        fig.update_layout(
            title=f'<b>Boxplot das Margens por Rank NSGA-II - {qq} Quantis (Sigma: {sigma:.2f})</b>',
            xaxis_title='Amostras (Ordenadas por Rank: Melhor -> Pior)',
            yaxis_title='Valor da Margem',
            template='plotly_white',
            showlegend=True,
            legend_title_text='<b>Grupo da Amostra</b>',
            height=600, width=1200,
            margin=dict(l=50, r=50, b=150, t=60),
            # Ajuste do eixo X para não ficar ilegível com muitos labels
            xaxis=dict(tickangle=90, tickfont=dict(size=8), tickmode='auto', showticklabels=False)
        )

        # Salvamento
        filename_base = f"boxplot_margens_{qq}_quantis"

        # html_path = os.path.join(output_dir_sigma, f"{filename_base}.html")
        # fig.write_html(html_path)

        try:
            png_path = os.path.join(output_dir_sigma, f"{filename_base}.png")
            fig.write_image(png_path, scale=3)
            # pdf_path = os.path.join(output_dir_sigma, f"{filename_base}.pdf")
            # fig.write_image(pdf_path)
        except ValueError:
            pass

print("\n\nProcesso concluído!")

## Boxplot de todas as margens para todos os sigmas

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from tqdm import tqdm


# --- NOVA SEÇÃO: CRIAÇÃO DO DIRETÓRIO DE SAÍDA ---
output_dir = "graficos_boxplot_margens"
os.makedirs(output_dir, exist_ok=True)
print(f"Os gráficos serão salvos no diretório: '{output_dir}'")


for qq in tqdm([2,4,8,16], desc="Gerando e Salvando Gráficos"):

    # Carregamento e preparação dos dados (seu código original)
    FOLDER_PATH = os.path.join(os.getcwd(), os.path.join('matrizes_margens_perturbadas',f'{qq}_quantis') )
    filename = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'

    try:
        df = pd.read_csv(os.path.join(FOLDER_PATH, filename), sep=';', decimal=',')
    except FileNotFoundError:
        print(f"\nAVISO: Arquivo para qq={qq} não encontrado em '{FOLDER_PATH}'. Pulando.")
        continue

    margem_cols = [f'margem_{i}' for i in range(1,df_dataset.shape[0])]
    df_long = df.melt(
        id_vars=['sigma','perturbacao'],
        value_vars=margem_cols,
        var_name='amostra',
        value_name='margem'
    )
    df_long['sigma_str'] = df_long['sigma'].map(lambda x: f"{x:.2f}")
    sigmas_unicos = sorted(df_long['sigma'].unique())
    sigmas_str = [f"{x:.2f}" for x in sigmas_unicos]
    df_long['sigma_str'] = pd.Categorical(df_long['sigma_str'], categories=sigmas_str, ordered=True)

    # Plotagem (seu código original)
    fig = px.box(
        df_long,
        x='sigma_str',
        y='margem',
        points='outliers',
        category_orders={'sigma_str': sigmas_str},
        labels={'sigma_str':'Sigma','margem':'Margem'},
        title=f'Distribuição das Margens por Sigma (todas as perturbações) - {qq} quantis'
    )
    fig.add_hline(y=0.2, line_dash='dash', line_color='red',
                  annotation_text='Limite Fraca (0.2)',
                  annotation_position='bottom left')
    fig.add_hline(y=0.6, line_dash='dash', line_color='green',
                  annotation_text='Limite Forte (0.6)',
                  annotation_position='top left')
    fig.update_layout(
        showlegend=False,
        xaxis_title='Sigma',
        yaxis_title='Margem',
        template='plotly_white',
        width=1200, # Define uma largura para consistência
        height=700  # Define uma altura para consistência
    )

    # --- NOVA SEÇÃO: SALVAMENTO DOS GRÁFICOS ---

    # Define um nome base para os arquivos de saída desta iteração
    filename_base = f"boxplot_margens_{qq}_quantis"

    # 1. Salvar em HTML (interativo)
    html_path = os.path.join(output_dir, f"{filename_base}.html")
    fig.write_html(html_path)
    print(f"  -> Gráfico para qq={qq} salvo em: {os.path.basename(html_path)}")

    # 2. Salvar em PNG (alta resolução) e PDF
    #    Este bloco precisa da biblioteca 'kaleido' instalada.
    try:
        # Salvar em PNG em alta resolução (aumentando a escala)
        png_path = os.path.join(output_dir, f"{filename_base}.png")
        fig.write_image(png_path, scale=3) # scale=3 gera uma imagem com 3x a resolução padrão
        print(f"  -> Gráfico para qq={qq} salvo em: {os.path.basename(png_path)}")

        # Salvar em PDF (vetorial, alta qualidade)
        pdf_path = os.path.join(output_dir, f"{filename_base}.pdf")
        fig.write_image(pdf_path)
        print(f"  -> Gráfico para qq={qq} salvo em: {os.path.basename(pdf_path)}")

    except ValueError as e:
        print(f"\nAVISO: Não foi possível salvar em PNG/PDF. A biblioteca 'kaleido' é necessária.")
        print("Instale com: pip install kaleido")
        # Interrompe o loop principal se kaleido não estiver instalado
        # para não repetir este aviso a cada iteração.
        break

    # A linha abaixo foi substituída pela lógica de salvamento.
    # Descomente se, além de salvar, você também quiser exibir na tela.
    # fig.show()

print("\n\nProcesso concluído!")

# Gráfico dos flip rates para cada valor de quantil

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import os

# Função auxiliar para converter cores HEX para um trio RGB numérico
def hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
    """Converte uma cor em formato hexadecimal (ex: '#RRGGBB') para uma tupla de inteiros (R, G, B)."""
    hex_color = hex_color.lstrip('#') # Remove o '#' do início
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

# --- 1. CONFIGURAÇÃO E CARREGAMENTO DOS DADOS ---

FOLDER_PATH = os.path.join(os.getcwd(), 'flip_rates')
csv_filename = os.path.join(FOLDER_PATH, 'flip_rates_consolidados.csv')

try:
    df_unificado = pd.read_csv(csv_filename, sep=';', decimal=',')
    print("Arquivo CSV unificado carregado com sucesso!")
    print(f"O DataFrame tem {len(df_unificado)} linhas e {len(df_unificado.columns)} colunas.")
except FileNotFoundError:
    print(f"ERRO: O arquivo '{csv_filename}' não foi encontrado.")
    exit()

# --- 2. PREPARAÇÃO PARA O PLOT ---

flip_rate_cols = [col for col in df_unificado.columns if col.startswith('flip_rate_pert_')]
n_perturbacoes = len(flip_rate_cols)

if n_perturbacoes == 0:
    print("ERRO: Nenhuma coluna 'flip_rate_pert_...' foi encontrada no arquivo.")
    exit()

print(f"Encontradas {n_perturbacoes} colunas de perturbação para análise.")

qq_values = sorted(df_unificado['qq'].unique())
OPACIDADE_IC = 0.3 # Opacidade ajustada para melhor visualização

color_sequence = px.colors.qualitative.Plotly
color_map = {qq: color_sequence[i % len(color_sequence)] for i, qq in enumerate(qq_values)}
print("\nMapa de cores definido para os quantis:")
for qq, color in color_map.items():
    print(f"  - qq={qq}: {color}")

fig = go.Figure()

# --- 3. CÁLCULO E PLOTAGEM PARA CADA QUANTIL ---

for qq in qq_values:
    print(f"Processando dados para qq = {qq}...")

    df_qq = df_unificado[df_unificado['qq'] == qq].copy()
    df_qq = df_qq.sort_values('sigma')

    if df_qq.empty:
        print(f"  -> Aviso: Nenhum dado encontrado para qq = {qq}. Pulando.")
        continue

    sigma_values = df_qq['sigma']

    confidence_means = df_qq[flip_rate_cols].mean(axis=1)
    std_devs = df_qq[flip_rate_cols].std(axis=1)
    std_error = std_devs / np.sqrt(n_perturbacoes)
    confidence_lower = confidence_means - 1.96 * std_error
    confidence_upper = confidence_means + 1.96 * std_error

    hex_color = color_map[qq]
    r, g, b = hex_to_rgb(hex_color)

    # Adiciona o intervalo de confiança PRIMEIRO para que a linha principal fique por cima
    fig.add_trace(go.Scatter(
        x=list(sigma_values) + list(sigma_values[::-1]),
        y=list(confidence_upper) + list(confidence_lower[::-1]),
        fill='toself',
        fillcolor=f'rgba({r}, {g}, {b}, {OPACIDADE_IC})',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False,
        name=f'IC qq={qq}'
    ))

    # Adiciona a linha principal DEPOIS
    fig.add_trace(go.Scatter(
        x=sigma_values,
        y=confidence_means,
        mode='lines+markers',
        name=f'{int(qq)}',
        line=dict(color=color_map[qq]),
        marker=dict(color=color_map[qq])
    ))

# --- 4. LAYOUT FINAL E SALVAMENTO ---

fig.update_layout(
    # title_text="<b>Flip Rate Médio por Quantil (com Intervalo de Confiança de 95%)</b>",
    xaxis_title="<b>Sigma (σ)</b>",
    yaxis_title="<b>Mean Flip Rate</b>",
    template="plotly_white",
    legend_title_text='<b>Quantiles</b>',
    font=dict(family="Arial, sans-serif", size=12)
)

# NOVO: Criação da pasta de saída para os gráficos
output_dir_graficos = "graficos_flip_rate"
os.makedirs(output_dir_graficos, exist_ok=True)

# NOVO: SEÇÃO DE SALVAMENTO DOS GRÁFICOS
print("\n--- Salvando os gráficos em diferentes formatos ---")
filename_base = "flip_rate_medio_por_quantil"

# Salva o gráfico em HTML (interativo)
html_path = os.path.join(output_dir_graficos, f"{filename_base}.html")
fig.write_html(html_path)
print(f"Gráfico salvo em: {html_path}")

try:
    # Salva em PNG com escala 3x (alta resolução)
    png_path = os.path.join(output_dir_graficos, f"{filename_base}.png")
    fig.write_image(png_path, scale=5, width=1000, height=600)
    print(f"Gráfico salvo em: {png_path}")

    # Salva em PDF
    pdf_path = os.path.join(output_dir_graficos, f"{filename_base}.pdf")
    fig.write_image(pdf_path, width=1000, height=600, scale=5)
    print(f"Gráfico salvo em: {pdf_path}")


    # Salva em SVG
    pdf_path = os.path.join(output_dir_graficos, f"{filename_base}.svg")
    fig.write_image(pdf_path, width=1000, height=600, scale=5)
    print(f"Gráfico salvo em: {pdf_path}")
except ValueError:
    print("\nAVISO: Não foi possível salvar em PNG/PDF. A biblioteca 'kaleido' é necessária.")
    print("Execute no terminal: pip install kaleido")

# Exibe o gráfico interativo no final
print("\nExibindo o gráfico...")
# fig.show()

## Graficos de linha ilustrando a taxa de concordância nos quantis das extremidades

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from tqdm import tqdm
import os

# ==============================================================================
# ==============================================================================
quantis_map = [2, 4, 8, 16]
# sigmas_exemplo = np.round(np.arange(0.01, 0.31, 0.01), 2)



# ==============================================================================
# SEÇÃO 1: PARÂMETROS E CRIAÇÃO DO DIRETÓRIO DE SAÍDA
# ==============================================================================
quantis_plot = [2, 4, 8, 16]
sigmas = np.round(np.arange(0.01, 0.31, 0.01), 2)

acertos_first = {q: [] for q in quantis_plot}
acertos_last = {q: [] for q in quantis_plot}

df_rotulado_base = df_rotulos[df_rotulos['perturbacao'] == 0].copy()

# --- NOVO: Criação do diretório de saída ---
output_dir = "graficos_taxa_concordancia"
os.makedirs(output_dir, exist_ok=True)
print(f"Os gráficos serão salvos no diretório: '{output_dir}'")

# ==============================================================================
# SEÇÃO 2: CÁLCULO DAS TAXAS DE ACERTO (Inalterado)
# ==============================================================================
for q in tqdm(quantis_plot, desc="Calculando Concordância por Quantil"):
    idx = quantis_map.index(q)
    df_first = first_quantis[idx]
    df_last = last_quantis[idx]

    ids_first = [int(id_val) for id_val in df_first['ID'].tolist()]
    ids_last = [int(id_val) for id_val in df_last['ID'].tolist()]
    colunas_rotulos_first = [f'rotulo_{i}' for i in ids_first]
    colunas_rotulos_last = [f'rotulo_{i}' for i in ids_last]

    for sigma in sigmas:
        linha_rotulos = df_rotulado_base[(df_rotulado_base['qq'] == q) & (df_rotulado_base['sigma'] == sigma)]
        if not linha_rotulos.empty:
            rotulos_first = linha_rotulos[colunas_rotulos_first].values.flatten()
            taxa_first = (len(rotulos_first) - np.sum(rotulos_first)) / len(rotulos_first)
            acertos_first[q].append(taxa_first)
            rotulos_last = linha_rotulos[colunas_rotulos_last].values.flatten()
            taxa_last = np.sum(rotulos_last) / len(rotulos_last)
            acertos_last[q].append(taxa_last)
        else:
            acertos_first[q].append(np.nan)
            acertos_last[q].append(np.nan)

# ==============================================================================
# SEÇÃO 3: PLOTAGEM DOS GRÁFICOS (Inalterado)
# ==============================================================================
print("\n--- Seção 3: Gerando os objetos dos gráficos ---")
# Gráfico para FIRST
fig_first = go.Figure()
for q in quantis_plot:
    fig_first.add_trace(go.Scatter(x=sigmas, y=acertos_first[q], mode='lines+markers', name=f'Quantil {q}'))
fig_first.update_layout(
    title='Taxa de Concordância - FIRST (Rótulo esperado = 0)',
    xaxis_title='Sigma', yaxis_title='Taxa de Concordância', template='plotly_white',
    width=1000, height=600 # Define um tamanho padrão
)

# Gráfico para LAST
fig_last = go.Figure()
for q in quantis_plot:
    fig_last.add_trace(go.Scatter(x=sigmas, y=acertos_last[q], mode='lines+markers', name=f'Quantil {q}'))
fig_last.update_layout(
    title='Taxa de Concordância - LAST (Rótulo esperado = 1)',
    xaxis_title='Sigma', yaxis_title='Taxa de Concordância', template='plotly_white',
    width=1000, height=600 # Define um tamanho padrão
)
print("Gráficos gerados com sucesso.")

# ==============================================================================
# SEÇÃO 4: SALVAMENTO DOS GRÁFICOS
# ==============================================================================
print("\n--- Seção 4: Salvando os gráficos em arquivos ---")

# Lista dos gráficos e seus nomes base para os arquivos
graficos_para_salvar = [
    (fig_first, "taxa_concordancia_first"),
    (fig_last, "taxa_concordancia_last")
]

try:
    for fig, filename_base in graficos_para_salvar:
        # 1. Salvar em HTML
        html_path = os.path.join(output_dir, f"{filename_base}.html")
        fig.write_html(html_path)
        print(f"  -> Gráfico salvo em: {os.path.basename(html_path)}")

        # 2. Salvar em PNG (alta resolução)
        png_path = os.path.join(output_dir, f"{filename_base}.png")
        fig.write_image(png_path, scale=3)
        print(f"  -> Gráfico salvo em: {os.path.basename(png_path)}")

        # 3. Salvar em PDF
        pdf_path = os.path.join(output_dir, f"{filename_base}.pdf")
        fig.write_image(pdf_path)
        print(f"  -> Gráfico salvo em: {os.path.basename(pdf_path)}")

except ValueError:
    print("\nAVISO: Para salvar em PNG/PDF, a biblioteca 'kaleido' é necessária.")
    print("Instale com: pip install kaleido")

# As linhas abaixo foram comentadas. Descomente se, além de salvar,
# você também quiser exibir os gráficos na tela.
# print("\nExibindo gráficos...")
# fig_first.show()
# fig_last.show()

print("\nProcesso concluído!")

# Analise Multi-Objetivo das Taxas de Concordancia para o First e Last Quantis

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import plotly.graph_objects as go
from IPython.display import display
import os
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

# quantis_map_exemplo = [2, 4, 8, 16, 32, 64]
# sigmas_exemplo = np.round(np.arange(0.01, 0.31, 0.01), 2)


# ==============================================================================
# SEÇÃO 1: CÁLCULO DOS DADOS DE ENTRADA (Inalterado)
# ==============================================================================
print("--- Seção 1: Calculando todas as taxas de concordância ---")
quantis_map = [2, 4, 8, 16]
sigma_limite = 0.30
passo = 0.01
sigmas = np.arange(0.01, sigma_limite + passo, passo)
acertos_first = {q: [] for q in quantis_map}
acertos_last = {q: [] for q in quantis_map}
df_rotulado_base = df_rotulos[df_rotulos['perturbacao'] == 0].copy()
for q in tqdm(quantis_map, desc="Calculando Concordância"):
    idx = quantis_map.index(q)
    df_first, df_last = first_quantis[idx], last_quantis[idx]
    ids_first = [int(id_val) for id_val in df_first['ID'].tolist()]
    ids_last = [int(id_val) for id_val in df_last['ID'].tolist()]
    colunas_rotulos_first = [f'rotulo_{i}' for i in ids_first]
    colunas_rotulos_last = [f'rotulo_{i}' for i in ids_last]
    for sigma in sigmas:
        sigma = round(sigma, 2)
        linha_rotulos = df_rotulado_base[(df_rotulado_base['qq'] == q) & (df_rotulado_base['sigma'] == sigma)]
        if not linha_rotulos.empty:
            rotulos_first = linha_rotulos[colunas_rotulos_first].values.flatten()
            acertos_first[q].append((len(rotulos_first) - np.sum(rotulos_first)) / len(rotulos_first))
            rotulos_last = linha_rotulos[colunas_rotulos_last].values.flatten()
            acertos_last[q].append(np.sum(rotulos_last) / len(rotulos_last))
        else:
            acertos_first[q].append(np.nan)
            acertos_last[q].append(np.nan)

# ==============================================================================
# SEÇÃO 2: PREPARAÇÃO DOS DADOS (Inalterado)
# ==============================================================================
print("\n--- Seção 2: Preparando dados ---")
dados_otimizacao = []
for i, q in enumerate(quantis_map):
    for j, s in enumerate(sigmas):
        dados_otimizacao.append({'qq': q, 'sigma': round(s, 2), 'taxa_first': acertos_first[q][j], 'taxa_last': acertos_last[q][j]})
df_lookup = pd.DataFrame(dados_otimizacao).dropna()

# ==============================================================================
# SEÇÃO 3: CÁLCULO DA FRONTEIRA DE PARETO (Inalterado)
# ==============================================================================
print("\n--- Seção 3: Calculando a fronteira de Pareto ---")
objective_values = -df_lookup[['taxa_first', 'taxa_last']].values
nds = NonDominatedSorting()
fronts = nds.do(objective_values)
pareto_indices = fronts[0]

# ==============================================================================
# SEÇÃO 4: ANÁLISE DOS RESULTADOS (Inalterado)
# ==============================================================================
print("\n--- Seção 4: Análise dos Resultados ---")
fronteira_de_pareto = df_lookup.iloc[pareto_indices].copy()
fronteira_de_pareto = fronteira_de_pareto.sort_values(by=['qq', 'sigma'],ascending=[True, True]).reset_index(drop=True)
fronteira_de_pareto['soma_taxas'] = fronteira_de_pareto['taxa_first'] + fronteira_de_pareto['taxa_last']
print("\n\n*** MELHORES CONFIGURAÇÕES ENCONTRADAS (FRONTEIRA DE PARETO) ***")
display(fronteira_de_pareto.style.apply(lambda s: ['color: red; font-weight: bold' if v else '' for v in s == s.max()], subset=['soma_taxas']))


# ==============================================================================
# SEÇÃO 5: PLOTAGEM DO GRÁFICO (Inalterado)
# ==============================================================================
print("\n--- Seção 5: Gerando o gráfico comparativo ---")
dominados_indices = np.setdiff1d(df_lookup.index.values, pareto_indices)
pontos_dominados = df_lookup.iloc[dominados_indices]
max_soma = fronteira_de_pareto['soma_taxas'].max()
pontos_max_soma = fronteira_de_pareto[fronteira_de_pareto['soma_taxas'] == max_soma]
fronteira_restante = fronteira_de_pareto[fronteira_de_pareto['soma_taxas'] != max_soma]

padding = 0.02
range_x = [df_lookup['taxa_first'].min() - padding, df_lookup['taxa_first'].max() + padding]
range_y = [df_lookup['taxa_last'].min() - padding, df_lookup['taxa_last'].max() + padding]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pontos_dominados['taxa_first'], y=pontos_dominados['taxa_last'], mode='markers',
    marker=dict(size=6, color='lightgray', opacity=0.7),
    text=[f"<b>Dominado</b><br>qq={row.qq}, sigma={row.sigma:.2f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in pontos_dominados.iterrows()],
    hoverinfo='text', name='Soluções Dominadas'
))
fig.add_trace(go.Scatter(
    x=fronteira_restante['taxa_first'], y=fronteira_restante['taxa_last'], mode='markers',
    marker=dict(size=10, color='blue', symbol='circle', line=dict(width=1, color='DarkSlateGrey')),
    text=[f"<b>Fronteira</b><br>qq={row.qq}, sigma={row.sigma:.2f}<br>Soma: {row.soma_taxas:.3f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in fronteira_restante.iterrows()],
    hoverinfo='text', name='Fronteira de Pareto'
))
fig.add_trace(go.Scatter(
    x=pontos_max_soma['taxa_first'], y=pontos_max_soma['taxa_last'], mode='markers',
    marker=dict(size=12, color='red', symbol='diamond', line=dict(width=1, color='black')),
    text=[f"<b>SOMA MÁXIMA</b><br>qq={row.qq}, sigma={row.sigma:.2f}<br>Soma: {row.soma_taxas:.3f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in pontos_max_soma.iterrows()],
    hoverinfo='text', name=f'Soma Máxima ({max_soma:.2f})'
))
fig.update_layout(
    title='<b>Fronteira de Pareto com Destaque para a Soma Máxima</b>',
    template='plotly_white', legend_title_text='<b>Tipo de Solução</b>',
    xaxis=dict(title='Taxa de Concordância - FIRST (maximizada)', range=range_x),
    yaxis=dict(title='Taxa de Concordância - LAST (maximizada)', range=range_y),
    width=1000, height=600,
    shapes=[dict(type='line', x0=0, y0=1, x1=1, y1=1, line=dict(color='Gray', width=1, dash='dash')), dict(type='line', x0=1, y0=0, x1=1, y1=1, line=dict(color='Gray', width=1, dash='dash'))]
)

# ==============================================================================
# SEÇÃO 6: SALVAMENTO DOS GRÁFICOS
# ==============================================================================
print("\n--- Seção 6: Salvando os gráficos em arquivos ---")

# --- NOVO: Criação do diretório de saída ---
output_dir = "graficos_fronteira_pareto"
os.makedirs(output_dir, exist_ok=True)
print(f"Os gráficos serão salvos no diretório: '{output_dir}'")

# --- NOVO: Lógica de salvamento ---
filename_base = "fronteira_pareto_soma_maxima"

try:
    # 1. Salvar em HTML (interativo)
    html_path = os.path.join(output_dir, f"{filename_base}.html")
    fig.write_html(html_path)
    print(f"  -> Gráfico salvo em: {os.path.basename(html_path)}")

    # 2. Salvar em PNG (alta resolução)
    png_path = os.path.join(output_dir, f"{filename_base}.png")
    fig.write_image(png_path, scale=3) # scale=3 para alta resolução
    print(f"  -> Gráfico salvo em: {os.path.basename(png_path)}")

    # 3. Salvar em PDF (vetorial)
    pdf_path = os.path.join(output_dir, f"{filename_base}.pdf")
    fig.write_image(pdf_path)
    print(f"  -> Gráfico salvo em: {os.path.basename(pdf_path)}")

except ValueError:
    print("\nAVISO: Para salvar em PNG/PDF, a biblioteca 'kaleido' é necessária.")
    print("Instale com: pip install kaleido")


# A linha abaixo foi comentada. Descomente se, além de salvar,
# você também quiser exibir o gráfico na tela.
# print("\n--- Exibindo o Gráfico ---")
# fig.show()

print("\nProcesso concluído!")

## Grafico HEATMAP da concordância nos quantis das extremidades

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import plotly.express as px
import os

# quantis_map_exemplo = [2, 4, 8, 16, 32, 64]
# sigmas_exemplo = np.round(np.arange(0.01, 0.31, 0.01), 2)


# ==============================================================================
# SEÇÃO 1: PARÂMETROS E CRIAÇÃO DO DIRETÓRIO
# ==============================================================================
quantis = [2, 4, 8, 16]
sigmas = np.arange(0.01, 0.31, 0.01)

df_rotulado_base = df_rotulos[df_rotulos['perturbacao'] == 0].copy()

# --- NOVO: Criação do diretório de saída ---
output_dir = "graficos_heatmap_concordancia"
os.makedirs(output_dir, exist_ok=True)
print(f"Os gráficos serão salvos no diretório: '{output_dir}'")

# ==============================================================================
# SEÇÃO 2: GERAÇÃO DO HEATMAP PARA O GRUPO FIRST
# ==============================================================================
print("\nGerando dados para o heatmap do grupo FIRST...")
heatmap_data_first = []

for i, q in enumerate(tqdm(quantis, desc="Quantis para FIRST")):
    row = []
    df_first = first_quantis[i]
    if df_first is None:
        row = [np.nan] * len(sigmas)
    else:
        ids_first = [int(id_val) for id_val in df_first['ID'].tolist()]
        colunas_rotulos_first = [f'rotulo_{id_val}' for id_val in ids_first]
        for sigma in sigmas:
            sigma = round(sigma, 2)
            linha_rotulos = df_rotulado_base[(df_rotulado_base['qq'] == q) & (df_rotulado_base['sigma'] == sigma)]
            if not linha_rotulos.empty:
                arr_first = linha_rotulos[colunas_rotulos_first].values.flatten()
                taxa_first = (len(arr_first) - np.sum(arr_first)) / len(arr_first)
                row.append(taxa_first)
            else:
                row.append(np.nan)
    heatmap_data_first.append(row)

# Plotagem do heatmap para FIRST
fig_first = px.imshow(
    heatmap_data_first,
    x=np.round(sigmas, 2),
    y=[str(q) for q in quantis],
    color_continuous_scale='Viridis',
    labels={'x': 'Sigma (σ)', 'y': 'Quantil (qq)', 'color': 'Taxa de Concordância'},
    title='<b>Heatmap da Taxa de Concordância - FIRST (Rótulo esperado = 0)</b>',
    width=1000, height=600 # Define um tamanho padrão
)

# ==============================================================================
# SEÇÃO 3: GERAÇÃO DO HEATMAP PARA O GRUPO LAST
# ==============================================================================
print("\nGerando dados para o heatmap do grupo LAST...")
heatmap_data_last = []

for i, q in enumerate(tqdm(quantis, desc="Quantis para LAST")):
    row = []
    df_last = last_quantis[i]
    if df_last is None:
        row = [np.nan] * len(sigmas)
    else:
        ids_last = [int(id_val) for id_val in df_last['ID'].tolist()]
        colunas_rotulos_last = [f'rotulo_{id_val}' for id_val in ids_last]
        for sigma in sigmas:
            sigma = round(sigma, 2)
            linha_rotulos = df_rotulado_base[(df_rotulado_base['qq'] == q) & (df_rotulado_base['sigma'] == sigma)]
            if not linha_rotulos.empty:
                arr_last = linha_rotulos[colunas_rotulos_last].values.flatten()
                taxa_last = np.sum(arr_last) / len(arr_last)
                row.append(taxa_last)
            else:
                row.append(np.nan)
    heatmap_data_last.append(row)

# Plotagem do heatmap para LAST
fig_last = px.imshow(
    heatmap_data_last,
    x=np.round(sigmas, 2),
    y=[str(q) for q in quantis],
    color_continuous_scale='Viridis',
    labels={'x': 'Sigma (σ)', 'y': 'Quantil (qq)', 'color': 'Taxa de Concordância'},
    title='<b>Heatmap da Taxa de Concordância - LAST (Rótulo esperado = 1)</b>',
    width=1000, height=600 # Define um tamanho padrão
)

# ==============================================================================
# SEÇÃO 4: SALVAMENTO DOS GRÁFICOS
# ==============================================================================
print("\n--- Seção 4: Salvando os gráficos em arquivos ---")

# Lista dos gráficos e seus nomes base para os arquivos
graficos_para_salvar = [
    (fig_first, "heatmap_concordancia_first"),
    (fig_last, "heatmap_concordancia_last")
]

try:
    for fig, filename_base in graficos_para_salvar:
        # 1. Salvar em HTML
        html_path = os.path.join(output_dir, f"{filename_base}.html")
        fig.write_html(html_path)
        print(f"  -> Gráfico salvo em: {os.path.basename(html_path)}")

        # 2. Salvar em PNG (alta resolução)
        png_path = os.path.join(output_dir, f"{filename_base}.png")
        fig.write_image(png_path, scale=3) # scale=3 para alta resolução
        print(f"  -> Gráfico salvo em: {os.path.basename(png_path)}")

        # 3. Salvar em PDF
        pdf_path = os.path.join(output_dir, f"{filename_base}.pdf")
        fig.write_image(pdf_path)
        print(f"  -> Gráfico salvo em: {os.path.basename(pdf_path)}")

except ValueError:
    print("\nAVISO: Para salvar em PNG/PDF, a biblioteca 'kaleido' é necessária.")
    print("Instale com: pip install kaleido")

# As chamadas .show() foram substituídas pela lógica de salvamento.
# Descomente as linhas abaixo se quiser exibir os gráficos na tela além de salvar.
# print("\nExibindo gráficos...")
# fig_first.show()
# fig_last.show()

print("\nProcesso concluído!")

## Graficos de linha ilustrando a taxa de concordancia para um valor especifico em % que representam o first e o last
Aqui da pra analisar a percentagem requerida, sem se informar em informacoes de quantis.

In [ ]:
df_rotulos.columns

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
from tqdm import tqdm

# --- PRÉ-REQUISITOS ---
# O código abaixo assume que a variável 'df_rotulos' já foi carregada.
# df_rotulos: DataFrame com colunas ['sigma', 'qq', 'perturbacao', 'rotulo_0', ...]

# ==============================================================================
# SEÇÃO 1: PARÂMETROS E CRIAÇÃO DO DIRETÓRIO
# ==============================================================================
quantis_map = [2, 4, 8, 16]
sigmas = np.round(np.arange(0.01, 0.31, 0.01), 2)
# Define qual a percentagem que formara o primeiro e o ultimo quantil a ser analisado.
percentagem_quantil = 0.10

acertos_first_25_percent = {q: [] for q in quantis_map}
acertos_last_25_percent = {q: [] for q in quantis_map}

df_rotulado_base = df_rotulos[df_rotulos['perturbacao'] == 0].copy()

# --- NOVO: Criação do diretório de saída ---
output_dir = "graficos_concordancia_percentual"
os.makedirs(output_dir, exist_ok=True)
print(f"Os gráficos serão salvos no diretório: '{output_dir}'")

# ==============================================================================
# SEÇÃO 2: CÁLCULO DAS TAXAS DE ACERTO (Inalterado)
# ==============================================================================
for q in tqdm(quantis_map, desc="Calculando Concordância por Bloco de Quantis"):
    try:
        file_path = os.path.join(f'{q}_quantis', f'dados_divididos_em_{q}_quantis.csv')
        df_dados = pd.read_csv(file_path, sep=';', decimal=',')
    except FileNotFoundError:
        print(f"Aviso: Arquivo '{file_path}' não encontrado. Pulando quantil {q}.")
        for _ in sigmas:
            acertos_first_25_percent[q].append(np.nan)
            acertos_last_25_percent[q].append(np.nan)
        continue

    num_quantis_bloco = max(1, int(q * percentagem_quantil))
    df_first_25 = df_dados[df_dados['quantil_idx'] < num_quantis_bloco]
    df_last_25 = df_dados[df_dados['quantil_idx'] >= q - num_quantis_bloco]

    ids_first = df_first_25['ID'].tolist()
    ids_last = df_last_25['ID'].tolist()

    colunas_rotulos_first = [f'rotulo_{int(i)}' for i in ids_first]
    colunas_rotulos_last = [f'rotulo_{int(i)}' for i in ids_last]

    for sigma in sigmas:
        linha_rotulos = df_rotulado_base[(df_rotulado_base['qq'] == q) & (df_rotulado_base['sigma'] == sigma)]
        if not linha_rotulos.empty and ids_first and ids_last:
            rotulos_first = linha_rotulos[colunas_rotulos_first].values.flatten()
            taxa_first = (len(rotulos_first) - np.sum(rotulos_first)) / len(rotulos_first)
            acertos_first_25_percent[q].append(taxa_first)

            rotulos_last = linha_rotulos[colunas_rotulos_last].values.flatten()
            taxa_last = np.sum(rotulos_last) / len(rotulos_last)
            acertos_last_25_percent[q].append(taxa_last)
        else:
            acertos_first_25_percent[q].append(np.nan)
            acertos_last_25_percent[q].append(np.nan)

# ==============================================================================
# SEÇÃO 3: PLOTAGEM DOS GRÁFICOS (Inalterado)
# ==============================================================================
print("\n--- Seção 3: Gerando os objetos dos gráficos ---")
# Gráfico para os X% iniciais
fig_first = go.Figure()
for q in quantis_map:
    fig_first.add_trace(go.Scatter(x=sigmas, y=acertos_first_25_percent[q], mode='lines+markers', name=f'{q} quantiles'))
fig_first.update_layout(
    # title=f'<b>Taxa de Concordância - {percentagem_quantil*100:.0f}% Iniciais (Rótulo esperado = 0)</b>',
    xaxis_title='Sigma', yaxis_title='Concordance Rate', template='plotly_white',
    # showlegend=False,
    width=1000, height=600 # Tamanho padrão para exportação
)

# Gráfico para os X% finais
fig_last = go.Figure()
for q in quantis_map:
    fig_last.add_trace(go.Scatter(x=sigmas, y=acertos_last_25_percent[q], mode='lines+markers', name=f'{q} quantiles'))
fig_last.update_layout(
    # title=f'<b>Taxa de Concordância - {percentagem_quantil*100:.0f}% Finais (Rótulo esperado = 1)</b>',
    xaxis_title='Sigma',
    yaxis_title='Concordance Rate',
    # showlegend=False,
    template='plotly_white',
    width=1000, height=600 # Tamanho padrão para exportação
)
print("Gráficos gerados com sucesso.")

# ==============================================================================
# SEÇÃO 4: SALVAMENTO DOS GRÁFICOS
# ==============================================================================
print("\n--- Seção 4: Salvando os gráficos em arquivos ---")

# Lista dos gráficos e seus nomes base para os arquivos
graficos_para_salvar = [
    (fig_first, f"taxa_concordancia_{percentagem_quantil*100:.0f}_iniciais"),
    (fig_last, f"taxa_concordancia_{percentagem_quantil*100:.0f}_finais")
]

try:
    for fig, filename_base in graficos_para_salvar:
        # 1. Salvar em HTML (interativo)
        html_path = os.path.join(output_dir, f"{filename_base}.html")
        fig.write_html(html_path)
        print(f"  -> Gráfico salvo em: {os.path.basename(html_path)}")

        # # 2. Salvar em PNG (alta resolução)
        # png_path = os.path.join(output_dir, f"{filename_base}.png")
        # fig.write_image(png_path, scale=3)
        # print(f"  -> Gráfico salvo em: {os.path.basename(png_path)}")

        # 3. Salvar em PDF (vetorial)
        pdf_path = os.path.join(output_dir, f"{filename_base}.pdf")
        fig.write_image(pdf_path)
        print(f"  -> Gráfico salvo em: {os.path.basename(pdf_path)}")

except ValueError:
    print("\nAVISO: Para salvar em PNG/PDF, a biblioteca 'kaleido' é necessária.")
    print("Instale com: pip install kaleido")

# As linhas abaixo foram substituídas pela lógica de salvamento.
# Descomente se quiser exibir os gráficos na tela além de salvá-los.
# print("\nExibindo gráficos...")
# fig_first.show()
# fig_last.show()

print("\nProcesso concluído!")

## Analise Multi-Objetivo das Taxas de Concordância para um valor de % requerido

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import plotly.graph_objects as go
from IPython.display import display
import os
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import plotly.express as px

# ==============================================================================
# NOVO: PALETA DE CORES POR QUANTIL
# Define 6 cores contrastantes para os quantis
QQ_COLORS = {
    2: px.colors.qualitative.Vivid[0],
    4: px.colors.qualitative.Vivid[1],
    8: px.colors.qualitative.Vivid[2],
    16: px.colors.qualitative.Vivid[3],
    32: px.colors.qualitative.Vivid[4],
    64: px.colors.qualitative.Vivid[5],
}
# ==============================================================================


# ==============================================================================
# SEÇÃO 1: CÁLCULO DAS TAXAS DE CONCORDÂNCIA (Assumido como funcional)
# ==============================================================================
print("--- Seção 1: Calculando taxas de concordância ---")

# Parâmetros
quantis_map = [2, 4, 8, 16, 32, 64]
percentagem_quantil = 0.10
sigma_limite = 0.30
passo = 0.01
sigmas = np.round(np.arange(passo, sigma_limite + passo, passo), 2)

# Dicionários para armazenar os resultados
acertos_first = {q: [] for q in quantis_map}
acertos_last = {q: [] for q in quantis_map}
# Nota: As variáveis df_rotulos, first_quantis, last_quantis são assumidas como carregadas.
df_rotulado_base = df_rotulos[df_rotulos['perturbacao'] == 0].copy()

for q in tqdm(quantis_map, desc=f"Calculando Concordância ({percentagem_quantil*100}%)"):
    try:
        file_path = os.path.join(f'{q}_quantis', f'dados_divididos_em_{q}_quantis.csv')
        df_dados = pd.read_csv(file_path, sep=';', decimal=',')
    except FileNotFoundError:
        acertos_first[q] = [np.nan] * len(sigmas)
        acertos_last[q] = [np.nan] * len(sigmas)
        continue

    total_amostras = len(df_dados)
    num_amostras_bloco = max(1, int(total_amostras * percentagem_quantil))

    df_first_group = df_dados.iloc[:num_amostras_bloco]
    df_last_group = df_dados.iloc[-num_amostras_bloco:]

    ids_first = df_first_group['ID'].tolist()
    ids_last = df_last_group['ID'].tolist()

    colunas_rotulos_first = [f'rotulo_{int(i)}' for i in ids_first]
    colunas_rotulos_last = [f'rotulo_{int(i)}' for i in ids_last]

    for sigma in sigmas:
        linha_rotulos = df_rotulado_base[(df_rotulado_base['qq'] == q) & (df_rotulado_base['sigma'] == sigma)]
        if not linha_rotulos.empty and ids_first and ids_last:
            rotulos_first = linha_rotulos[colunas_rotulos_first].values.flatten()
            taxa_first = (len(rotulos_first) - np.sum(rotulos_first)) / len(rotulos_first)
            acertos_first[q].append(taxa_first)

            rotulos_last = linha_rotulos[colunas_rotulos_last].values.flatten()
            taxa_last = np.sum(rotulos_last) / len(rotulos_last)
            acertos_last[q].append(taxa_last)
        else:
            acertos_first[q].append(np.nan)
            acertos_last[q].append(np.nan)

# ==============================================================================
# SEÇÃO 2: PREPARAÇÃO DOS DADOS
# ==============================================================================
print("\n--- Seção 2: Preparando dados ---")
dados_otimizacao = []
for i, q in enumerate(quantis_map):
    for j, s in enumerate(sigmas):
        dados_otimizacao.append({'qq': q, 'sigma': round(s, 2), 'taxa_first': acertos_first[q][j], 'taxa_last': acertos_last[q][j]})
df_lookup = pd.DataFrame(dados_otimizacao).dropna()

# ==============================================================================
# SEÇÃO 3: CÁLCULO DA FRONTEIRA DE PARETO
# ==============================================================================
print("\n--- Seção 3: Calculando a Fronteira de Pareto por Não-Dominância ---")
objective_values = -df_lookup[['taxa_first', 'taxa_last']].values
nds = NonDominatedSorting()
fronts = nds.do(objective_values)
pareto_indices = fronts[0]

# ==============================================================================
# SEÇÃO 4: ANÁLISE E SEPARAÇÃO DOS RESULTADOS
# ==============================================================================
print("\n--- Seção 4: Análise dos Resultados ---")
fronteira_de_pareto = df_lookup.iloc[pareto_indices].copy()
fronteira_de_pareto['soma_taxas'] = fronteira_de_pareto['taxa_first'] + fronteira_de_pareto['taxa_last']
fronteira_de_pareto = fronteira_de_pareto.sort_values(by=['qq', 'sigma'],ascending=[True, True]).reset_index(drop=True)

# Separação dos subgrupos para plotagem
max_soma = fronteira_de_pareto['soma_taxas'].max()
pontos_max_soma = fronteira_de_pareto[fronteira_de_pareto['soma_taxas'] == max_soma]
fronteira_restante = fronteira_de_pareto[fronteira_de_pareto['soma_taxas'] != max_soma]
dominados_indices = np.setdiff1d(df_lookup.index.values, pareto_indices)
pontos_dominados = df_lookup.iloc[dominados_indices]


# ==============================================================================
# SEÇÃO 5: PLOTAGEM DO GRÁFICO (Com Contornos e Cores por QQ)
# ==============================================================================
print("\n--- Seção 5: Gerando o gráfico comparativo com Contornos ---")

padding = 0.02
range_x = [df_lookup['taxa_first'].min() - padding, df_lookup['taxa_first'].max() + padding]
range_y = [df_lookup['taxa_last'].min() - padding, df_lookup['taxa_last'].max() + padding]

fig = go.Figure()

# 1. BASE: Plotagem das Soluções (Dominadas + Fronteira) coloridas por QQ.
# Concatena todos os pontos para plotar a cor base de cada um
pontos_base = pd.concat([pontos_dominados, fronteira_restante, pontos_max_soma]).sort_values(by=['qq', 'sigma']).reset_index(drop=True)

for q_val in quantis_map:
    df_qq = pontos_base[pontos_base['qq'] == q_val]
    if not df_qq.empty:
        fig.add_trace(go.Scatter(
            x=df_qq['taxa_first'], y=df_qq['taxa_last'], mode='markers',
            marker=dict(size=6, color=QQ_COLORS[q_val], opacity=0.7),
            text=[f"qq={row.qq}, sigma={row.sigma:.2f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in df_qq.iterrows()],
            hoverinfo='text',
            name=f'Pontos (qq={q_val})',
            legendgroup='qq_group'
        ))

# 2. CONTORNO: FRONTEIRA RESTANTE (Círculo vazio azul sobre a cor original)
fig.add_trace(go.Scatter(
    x=fronteira_restante['taxa_first'], y=fronteira_restante['taxa_last'], mode='markers',
    marker=dict(
        size=10,
        color='rgba(0,0,0,0)', # Preenchimento totalmente transparente
        symbol='circle',
        line=dict(width=2, color='blue') # Contorno sólido azul
    ),
    text=[f"<b>Fronteira</b><br>qq={row.qq}, sigma={row.sigma:.2f}<br>Soma: {row.soma_taxas:.3f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in fronteira_restante.iterrows()],
    hoverinfo='text', name='Fronteira (Círculo Azul)'
))

# 3. CONTORNO: PONTO DE SOMA MÁXIMA (Círculo vazio vermelho sobre a cor original)
fig.add_trace(go.Scatter(
    x=pontos_max_soma['taxa_first'], y=pontos_max_soma['taxa_last'], mode='markers',
    marker=dict(
        size=14,
        color='rgba(0,0,0,0)', # Preenchimento totalmente transparente
        symbol='circle', # ALTERADO: De 'x' (cruz) para 'circle' (círculo)
        line=dict(width=3, color='red') # Contorno sólido vermelho
    ),
    text=[f"<b>SOMA MÁXIMA</b><br>qq={row.qq}, sigma={row.sigma:.2f}<br>Soma: {row.soma_taxas:.3f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in pontos_max_soma.iterrows()],
    hoverinfo='text', name=f'Soma Máxima (Círculo Vermelho)'
))

fig.update_layout(
    title='<b>Fronteira de Pareto com Destaque por Contorno (QQ Visível)</b>',
    template='plotly_white', legend_title_text='<b>Tipo de Solução / Quantil (qq)</b>',
    xaxis=dict(title='Taxa de Concordância - FIRST (maximizada)', range=range_x),
    yaxis=dict(title='Taxa de Concordância - LAST (maximizada)', range=range_y),
    width=1000, height=700,
    shapes=[dict(type='line', x0=0, y0=1, x1=1, y1=1, line=dict(color='Gray', width=1, dash='dash')), dict(type='line', x0=1, y0=0, x1=1, y1=1, line=dict(color='Gray', width=1, dash='dash'))]
)

# ==============================================================================
# SEÇÃO 6: SALVAMENTO DOS GRÁFICOS (Mantido)
# ==============================================================================
print("\n--- Seção 6: Salvando o gráfico em arquivos ---")

output_dir = "graficos_fronteira_pareto"
os.makedirs(output_dir, exist_ok=True)
print(f"O gráfico será salvo no diretório: '{output_dir}'")

filename_base = f"fronteira_pareto_{percentagem_quantil*100:.0f}_percent"

try:
    # 1. Salvar em HTML (interativo)
    html_path = os.path.join(output_dir, f"{filename_base}.html")
    fig.write_html(html_path)
    print(f"  -> Gráfico salvo em: {os.path.basename(html_path)}")

    # 2. Salvar em PNG (alta resolução)
    png_path = os.path.join(output_dir, f"{filename_base}.png")
    fig.write_image(png_path, scale=3)
    print(f"  -> Gráfico salvo em: {os.path.basename(png_path)}")

    # 3. Salvar em PDF (vetorial)
    pdf_path = os.path.join(output_dir, f"{filename_base}.pdf")
    fig.write_image(pdf_path)
    print(f"  -> Gráfico salvo em: {os.path.basename(pdf_path)}")

except ValueError:
    print("\nAVISO: Para salvar em PNG/PDF, a biblioteca 'kaleido' é necessária.")

print("\nProcesso concluído!")

## Analise Multi-Objetivo das Taxas de Concordância para um valor de % requerido usando apenas uma faixa determinada de sigma

### SEM COR

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import plotly.graph_objects as go
from IPython.display import display
import os
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import plotly.express as px

# ==============================================================================
# SEÇÃO 1: CÁLCULO DAS TAXAS DE CONCORDÂNCIA
# ==============================================================================
print("--- Seção 1: Calculando taxas de concordância ---")

# Parâmetros
quantis_map = [2, 4, 8, 16] # Usando a lista reduzida do seu exemplo
percentagem_quantil = 0.10
sigma_limite = 0.30
passo = 0.01
sigmas = np.round(np.arange(passo, sigma_limite + passo, passo), 2)

# Dicionários para armazenar os resultados
acertos_first = {q: [] for q in quantis_map}
acertos_last = {q: [] for q in quantis_map}
# Nota: As variáveis df_rotulos, first_quantis, last_quantis são assumidas como carregadas.
df_rotulado_base = df_rotulos[df_rotulos['perturbacao'] == 0].copy()

for q in tqdm(quantis_map, desc=f"Calculando Concordância ({percentagem_quantil*100}%)"):
    try:
        file_path = os.path.join(f'{q}_quantis', f'dados_divididos_em_{q}_quantis.csv')
        df_dados = pd.read_csv(file_path, sep=';', decimal=',')
    except FileNotFoundError:
        acertos_first[q] = [np.nan] * len(sigmas)
        acertos_last[q] = [np.nan] * len(sigmas)
        continue

    total_amostras = len(df_dados)
    num_amostras_bloco = max(1, int(total_amostras * percentagem_quantil))

    df_first_group = df_dados.iloc[:num_amostras_bloco]
    df_last_group = df_dados.iloc[-num_amostras_bloco:]

    ids_first = df_first_group['ID'].tolist()
    ids_last = df_last_group['ID'].tolist()

    colunas_rotulos_first = [f'rotulo_{int(i)}' for i in ids_first]
    colunas_rotulos_last = [f'rotulo_{int(i)}' for i in ids_last]

    for sigma in sigmas:
        linha_rotulos = df_rotulado_base[(df_rotulado_base['qq'] == q) & (df_rotulado_base['sigma'] == sigma)]
        if not linha_rotulos.empty and ids_first and ids_last:
            rotulos_first = linha_rotulos[colunas_rotulos_first].values.flatten()
            taxa_first = (len(rotulos_first) - np.sum(rotulos_first)) / len(rotulos_first)
            acertos_first[q].append(taxa_first)

            rotulos_last = linha_rotulos[colunas_rotulos_last].values.flatten()
            taxa_last = np.sum(rotulos_last) / len(rotulos_last)
            acertos_last[q].append(taxa_last)
        else:
            acertos_first[q].append(np.nan)
            acertos_last[q].append(np.nan)

# ==============================================================================
# SEÇÃO 2: PREPARAÇÃO DOS DADOS (Com Filtro de Sigma)
# ==============================================================================
print("\n--- Seção 2: Preparando dados ---")
dados_otimizacao = []
for i, q in enumerate(quantis_map):
    for j, s in enumerate(sigmas):
        dados_otimizacao.append({'qq': q, 'sigma': round(s, 2), 'taxa_first': acertos_first[q][j], 'taxa_last': acertos_last[q][j]})
df_lookup = pd.DataFrame(dados_otimizacao).dropna()

sigma_min_aceito = 0.17
sigma_max_aceito = 0.20

# --- 1. FILTRO PARA MANTER APENAS SIGMA SELECIONADOS ---
df_lookup = df_lookup[
    (df_lookup['sigma'] >= sigma_min_aceito) &
    (df_lookup['sigma'] <= sigma_max_aceito)
].copy()

# --- 2. CORREÇÃO CRÍTICA: RESETAR O ÍNDICE ---
df_lookup.reset_index(drop=True, inplace=True)

print(f"Filtro aplicado: Mantidas apenas soluções com sigma entre {sigma_min_aceito:.2f} e {sigma_max_aceito:.2f}.")

if df_lookup.empty:
    print("ERRO: DataFrame vazio após filtro. Verifique os parâmetros.")
else:
    # ==============================================================================
    # SEÇÃO 3: CÁLCULO DA FRONTEIRA DE PARETO
    # ==============================================================================
    print("\n--- Seção 3: Calculando a Fronteira de Pareto ---")
    objective_values = -df_lookup[['taxa_first', 'taxa_last']].values
    nds = NonDominatedSorting()
    fronts = nds.do(objective_values)
    pareto_indices = fronts[0]

    # ==============================================================================
    # SEÇÃO 4: ANÁLISE E SEPARAÇÃO DOS RESULTADOS
    # ==============================================================================
    print("\n--- Seção 4: Análise dos Resultados ---")
    fronteira_de_pareto = df_lookup.iloc[pareto_indices].copy()
    fronteira_de_pareto['soma_taxas'] = fronteira_de_pareto['taxa_first'] + fronteira_de_pareto['taxa_last']
    fronteira_de_pareto = fronteira_de_pareto.sort_values(by=['qq', 'sigma'],ascending=[True, True]).reset_index(drop=True)

    max_soma = fronteira_de_pareto['soma_taxas'].max()
    pontos_max_soma = fronteira_de_pareto[fronteira_de_pareto['soma_taxas'] == max_soma]
    fronteira_restante = fronteira_de_pareto[fronteira_de_pareto['soma_taxas'] != max_soma]
    dominados_indices = np.setdiff1d(df_lookup.index.values, pareto_indices)
    pontos_dominados = df_lookup.iloc[dominados_indices]

    # ==============================================================================
    # SEÇÃO 5: PLOTAGEM DO GRÁFICO (PONTOS CINZA UNIFICADOS)
    # ==============================================================================
    print("\n--- Seção 5: Gerando o gráfico comparativo (Cinza Unificado) ---")

    padding = 0.02
    range_x = [df_lookup['taxa_first'].min() - padding, df_lookup['taxa_first'].max() + padding]
    range_y = [df_lookup['taxa_last'].min() - padding, df_lookup['taxa_last'].max() + padding]

    fig = go.Figure()

    # 1. BASE: Plota TODOS os pontos em CINZA
    # Concatenamos tudo para plotar de uma vez só
    pontos_base = pd.concat([pontos_dominados, fronteira_restante, pontos_max_soma]).sort_values(by=['qq', 'sigma']).reset_index(drop=True)

    fig.add_trace(go.Scatter(
        x=pontos_base['taxa_first'],
        y=pontos_base['taxa_last'],
        mode='markers',
        # ALTERADO: Cor fixa cinza para todos
        marker=dict(size=11, color='gray', opacity=0.6),
        # O texto do hover ainda mostra as informações de qq e sigma
        text=[f"qq={row.qq}, sigma={row.sigma:.2f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in pontos_base.iterrows()],
        hoverinfo='text',
        name='Demais Soluções'
    ))

    # 2. CONTORNO: FRONTEIRA RESTANTE (Círculo vazio azul)
    fig.add_trace(go.Scatter(
        x=fronteira_restante['taxa_first'], y=fronteira_restante['taxa_last'], mode='markers',
        marker=dict(
            size=11,
            color='rgba(0,0,0,0)', # Transparente
            symbol='circle',
            line=dict(width=2, color='blue') # Borda Azul
        ),
        text=[f"<b>Fronteira</b><br>qq={row.qq}, sigma={row.sigma:.2f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in fronteira_restante.iterrows()],
        hoverinfo='text', name='Fronteira Pareto'
    ))

    # 3. CONTORNO: PONTO DE SOMA MÁXIMA (Círculo vazio vermelho)
    fig.add_trace(go.Scatter(
        x=pontos_max_soma['taxa_first'], y=pontos_max_soma['taxa_last'], mode='markers',
        marker=dict(
            size=11,
            color='rgba(0,0,0,0)', # Transparente
            symbol='circle',
            line=dict(width=3, color='red') # Borda Vermelha
        ),
        text=[f"<b>SOMA MÁXIMA</b><br>qq={row.qq}, sigma={row.sigma:.2f}<br>Soma: {row.soma_taxas:.3f}" for _, row in pontos_max_soma.iterrows()],
        hoverinfo='text', name='Soma Máxima'
    ))

    fig.update_layout(
        # title=f'<b>Fronteira de Pareto (Cinza Unificado) - Sigmas {sigma_min_aceito:.2f}-{sigma_max_aceito:.2f}</b>',
        template='plotly_white',
        xaxis=dict(title=f'Concordance Rate - {int(percentagem_quantil*100)}% Iniciais', range=range_x),
        yaxis=dict(title=f'Concordance Rate- {int(percentagem_quantil*100)}% Finais', range=range_y),
        width=1000, height=700,
        legend_title_text='<b>Legenda</b>',
        shapes=[dict(type='line', x0=0, y0=1, x1=1, y1=1, line=dict(color='Gray', width=1, dash='dash')), dict(type='line', x0=1, y0=0, x1=1, y1=1, line=dict(color='Gray', width=1, dash='dash'))]
    )

    # ==============================================================================
    # SEÇÃO 6: SALVAMENTO
    # ==============================================================================
    print("\n--- Seção 6: Salvando o gráfico ---")
    output_dir = "graficos_fronteira_pareto"
    os.makedirs(output_dir, exist_ok=True)

    filename_base = f"fronteira_pareto_cinza_{sigma_min_aceito:.2f}_a_{sigma_max_aceito:.2f}_{round(100*percentagem_quantil)}_percent"

    try:
        html_path = os.path.join(output_dir, f"{filename_base}.html")
        fig.write_html(html_path)
        print(f"  -> HTML salvo: {os.path.basename(html_path)}")

        # png_path = os.path.join(output_dir, f"{filename_base}.png")
        # fig.write_image(png_path, scale=3)
        # print(f"  -> PNG salvo: {os.path.basename(png_path)}")

        pdf_path = os.path.join(output_dir, f"{filename_base}.pdf")
        fig.write_image(pdf_path, scale=5)
        print(f"  -> PDF salvo: {os.path.basename(pdf_path)}")

    except ValueError:
        print("\nAVISO: Para salvar em PNG/PDF, instale 'kaleido'.")

    print("\nProcesso concluído!")

## Analise Multi-Objetivo das Taxas de Concordância para um valor de % requerido usando apenas uma faixa determinada de sigma

### COM COR

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import plotly.graph_objects as go
from IPython.display import display
import os
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import plotly.express as px

# ==============================================================================
# NOVO: PALETA DE CORES POR QUANTIL
# Define 6 cores contrastantes para os quantis
QQ_COLORS = {
    2: px.colors.qualitative.Vivid[0],
    4: px.colors.qualitative.Vivid[1],
    8: px.colors.qualitative.Vivid[2],
    16: px.colors.qualitative.Vivid[3],
    32: px.colors.qualitative.Vivid[4],
    64: px.colors.qualitative.Vivid[5],
}
# ==============================================================================


# ==============================================================================
# SEÇÃO 1: CÁLCULO DAS TAXAS DE CONCORDÂNCIA (Inalterado)
# ==============================================================================
print("--- Seção 1: Calculando taxas de concordância ---")

# Parâmetros
quantis_map = [2, 4, 8, 16]
percentagem_quantil = 0.10
sigma_limite = 0.30
passo = 0.01
sigmas = np.round(np.arange(passo, sigma_limite + passo, passo), 2)

# Dicionários para armazenar os resultados
acertos_first = {q: [] for q in quantis_map}
acertos_last = {q: [] for q in quantis_map}
# Nota: As variáveis df_rotulos, first_quantis, last_quantis são assumidas como carregadas.
df_rotulado_base = df_rotulos[df_rotulos['perturbacao'] == 0].copy()

for q in tqdm(quantis_map, desc=f"Calculando Concordância ({percentagem_quantil*100}%)"):
    try:
        file_path = os.path.join(f'{q}_quantis', f'dados_divididos_em_{q}_quantis.csv')
        df_dados = pd.read_csv(file_path, sep=';', decimal=',')
    except FileNotFoundError:
        acertos_first[q] = [np.nan] * len(sigmas)
        acertos_last[q] = [np.nan] * len(sigmas)
        continue

    total_amostras = len(df_dados)
    num_amostras_bloco = max(1, int(total_amostras * percentagem_quantil))

    df_first_group = df_dados.iloc[:num_amostras_bloco]
    df_last_group = df_dados.iloc[-num_amostras_bloco:]

    ids_first = df_first_group['ID'].tolist()
    ids_last = df_last_group['ID'].tolist()

    colunas_rotulos_first = [f'rotulo_{int(i)}' for i in ids_first]
    colunas_rotulos_last = [f'rotulo_{int(i)}' for i in ids_last]

    for sigma in sigmas:
        linha_rotulos = df_rotulado_base[(df_rotulado_base['qq'] == q) & (df_rotulado_base['sigma'] == sigma)]
        if not linha_rotulos.empty and ids_first and ids_last:
            rotulos_first = linha_rotulos[colunas_rotulos_first].values.flatten()
            taxa_first = (len(rotulos_first) - np.sum(rotulos_first)) / len(rotulos_first)
            acertos_first[q].append(taxa_first)

            rotulos_last = linha_rotulos[colunas_rotulos_last].values.flatten()
            taxa_last = np.sum(rotulos_last) / len(rotulos_last)
            acertos_last[q].append(taxa_last)
        else:
            acertos_first[q].append(np.nan)
            acertos_last[q].append(np.nan)

# ==============================================================================
# SEÇÃO 2: PREPARAÇÃO DOS DADOS (Com Filtro de Sigma)
# ==============================================================================
print("\n--- Seção 2: Preparando dados ---")
dados_otimizacao = []
for i, q in enumerate(quantis_map):
    for j, s in enumerate(sigmas):
        dados_otimizacao.append({'qq': q, 'sigma': round(s, 2), 'taxa_first': acertos_first[q][j], 'taxa_last': acertos_last[q][j]})
df_lookup = pd.DataFrame(dados_otimizacao).dropna()

sigma_min_aceito = 0.17
sigma_max_aceito = 0.20

# --- 1. FILTRO PARA MANTER APENAS SIGMA ENTRE 0.05 E 0.09 ---
df_lookup = df_lookup[
    (df_lookup['sigma'] >= sigma_min_aceito) &
    (df_lookup['sigma'] <= sigma_max_aceito)
].copy()

# --- 2. CORREÇÃO CRÍTICA: RESETAR O ÍNDICE ---
# Isso garante que o índice posicional seja contínuo (0, 1, 2, ...)
# para que o pymoo (que usa .values) e o .iloc (que usa posições) funcionem corretamente.
df_lookup.reset_index(drop=True, inplace=True)

print(f"Filtro aplicado: Mantidas apenas soluções com sigma entre {sigma_min_aceito:.2f} e {sigma_max_aceito:.2f}.")

# ==============================================================================
# SEÇÃO 3: CÁLCULO DA FRONTEIRA DE PARETO (Ocorre sobre os dados filtrados)
# ==============================================================================
print("\n--- Seção 3: Calculando a Fronteira de Pareto por Não-Dominância ---")
objective_values = -df_lookup[['taxa_first', 'taxa_last']].values
nds = NonDominatedSorting()
fronts = nds.do(objective_values)
pareto_indices = fronts[0]

# ==============================================================================
# SEÇÃO 4: ANÁLISE E SEPARAÇÃO DOS RESULTADOS
# ==============================================================================
print("\n--- Seção 4: Análise dos Resultados ---")
fronteira_de_pareto = df_lookup.iloc[pareto_indices].copy()
fronteira_de_pareto['soma_taxas'] = fronteira_de_pareto['taxa_first'] + fronteira_de_pareto['taxa_last']
fronteira_de_pareto = fronteira_de_pareto.sort_values(by=['qq', 'sigma'],ascending=[True, True]).reset_index(drop=True)

# Separação dos subgrupos para plotagem
max_soma = fronteira_de_pareto['soma_taxas'].max()
pontos_max_soma = fronteira_de_pareto[fronteira_de_pareto['soma_taxas'] == max_soma]
fronteira_restante = fronteira_de_pareto[fronteira_de_pareto['soma_taxas'] != max_soma]
dominados_indices = np.setdiff1d(df_lookup.index.values, pareto_indices)
pontos_dominados = df_lookup.iloc[dominados_indices]


# ==============================================================================
# SEÇÃO 5: PLOTAGEM DO GRÁFICO (Com Contornos e Cores por QQ)
# ==============================================================================
print("\n--- Seção 5: Gerando o gráfico comparativo com Contornos ---")

padding = 0.02
range_x = [df_lookup['taxa_first'].min() - padding, df_lookup['taxa_first'].max() + padding]
range_y = [df_lookup['taxa_last'].min() - padding, df_lookup['taxa_last'].max() + padding]

fig = go.Figure()

# 1. BASE: Plota TODAS as Soluções (Dominadas + Fronteira) coloridas por QQ.
pontos_base = pd.concat([pontos_dominados, fronteira_restante, pontos_max_soma]).sort_values(by=['qq', 'sigma']).reset_index(drop=True)

for q_val in quantis_map:
    df_qq = pontos_base[pontos_base['qq'] == q_val]
    if not df_qq.empty:
        fig.add_trace(go.Scatter(
            x=df_qq['taxa_first'], y=df_qq['taxa_last'], mode='markers',
            marker=dict(size=6, color=QQ_COLORS[q_val], opacity=0.7),
            text=[f"qq={row.qq}, sigma={row.sigma:.2f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in df_qq.iterrows()],
            hoverinfo='text',
            name=f'{q_val} quantiles',
            legendgroup='qq_group'
        ))

# 2. CONTORNO: FRONTEIRA RESTANTE (Círculo vazio azul sobre a cor original)
fig.add_trace(go.Scatter(
    x=fronteira_restante['taxa_first'], y=fronteira_restante['taxa_last'], mode='markers',
    marker=dict(
        size=14,
        color='rgba(0,0,0,0)', # Preenchimento totalmente transparente
        symbol='circle',
        line=dict(width=2, color='blue') # Contorno sólido azul
    ),
    text=[f"<b>Fronteira</b><br>qq={row.qq}, sigma={row.sigma:.2f}<br>Soma: {row.soma_taxas:.3f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in fronteira_restante.iterrows()],
    hoverinfo='text', name='Pareto Frontier'
))

# 3. CONTORNO: PONTO DE SOMA MÁXIMA (Círculo vazio vermelho sobre a cor original)
fig.add_trace(go.Scatter(
    x=pontos_max_soma['taxa_first'], y=pontos_max_soma['taxa_last'], mode='markers',
    marker=dict(
        size=14,
        color='rgba(0,0,0,0)', # Preenchimento totalmente transparente
        symbol='circle',
        line=dict(width=3, color='red') # Contorno sólido vermelho
    ),
    text=[f"<b>SOMA MÁXIMA</b><br>qq={row.qq}, sigma={row.sigma:.2f}<br>Soma: {row.soma_taxas:.3f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in pontos_max_soma.iterrows()],
    hoverinfo='text', name=f'Máx Sum'
))

fig.update_layout(
    # title=f'<b>Fronteira de Pareto (Porção: {round(100*percentagem_quantil)}% - Sigmas {sigma_min_aceito:.2f}-{sigma_max_aceito:.2f})</b>',
    template='plotly_white',
    # legend_title_text='<b>Quantis (qq)</b>',
    xaxis=dict(title=f'Concordance Rate - Initial Samples ({int(percentagem_quantil*100)}%)', range=range_x),
    yaxis=dict(title=f'Concordance Rate - Final Samples ({int(percentagem_quantil*100)}%)', range=range_y),
    width=1000, height=700,
    shapes=[dict(type='line', x0=0, y0=1, x1=1, y1=1, line=dict(color='Gray', width=1, dash='dash')), dict(type='line', x0=1, y0=0, x1=1, y1=1, line=dict(color='Gray', width=1, dash='dash'))]
)

# ==============================================================================
# SEÇÃO 6: SALVAMENTO DOS GRÁFICOS
# ==============================================================================
print("\n--- Seção 6: Salvando o gráfico em arquivos ---")

output_dir = "graficos_fronteira_pareto"
os.makedirs(output_dir, exist_ok=True)
print(f"O gráfico será salvo no diretório: '{output_dir}'")

filename_base = f"fronteira_pareto_{sigma_min_aceito:.2f}_a_{sigma_max_aceito:.2f}_{round(100*percentagem_quantil)}_percent"

try:
    # 1. Salvar em HTML (interativo)
    html_path = os.path.join(output_dir, f"{filename_base}.html")
    fig.write_html(html_path)
    print(f"  -> Gráfico salvo em: {os.path.basename(html_path)}")

    # # 2. Salvar em PNG (alta resolução)
    # png_path = os.path.join(output_dir, f"{filename_base}.png")
    # fig.write_image(png_path, scale=5)
    # print(f"  -> Gráfico salvo em: {os.path.basename(png_path)}")

    # 3. Salvar em PDF (vetorial)
    pdf_path = os.path.join(output_dir, f"{filename_base}.pdf")
    fig.write_image(pdf_path, scale=5)
    print(f"  -> Gráfico salvo em: {os.path.basename(pdf_path)}")

    # # 4. Salvar em SVG (vetorial)
    # pdf_path = os.path.join(output_dir, f"{filename_base}.svg")
    # fig.write_image(pdf_path, scale=5)
    # print(f"  -> Gráfico salvo em: {os.path.basename(pdf_path)}")

except ValueError:
    print("\nAVISO: Para salvar em PNG/PDF, a biblioteca 'kaleido' é necessária.")

print("\nProcesso concluído!")

# MultiObjetivo com jitter

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import plotly.graph_objects as go
from IPython.display import display
import os
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import plotly.express as px

# ==============================================================================
# NOVO: PALETA DE CORES POR QUANTIL
# Define 6 cores contrastantes para os quantis
QQ_COLORS = {
    2: px.colors.qualitative.Vivid[0],
    4: px.colors.qualitative.Vivid[1],
    8: px.colors.qualitative.Vivid[2],
    16: px.colors.qualitative.Vivid[3],
    32: px.colors.qualitative.Vivid[4],
    64: px.colors.qualitative.Vivid[5],
}
# ==============================================================================


# ==============================================================================
# SEÇÃO 1: CÁLCULO DAS TAXAS DE CONCORDÂNCIA (Inalterado)
# ==============================================================================
print("--- Seção 1: Calculando taxas de concordância ---")

# Parâmetros
quantis_map = [2, 4, 8, 16]
percentagem_quantil = 0.10
sigma_limite = 0.30
passo = 0.01
sigmas = np.round(np.arange(passo, sigma_limite + passo, passo), 2)

# Dicionários para armazenar os resultados
acertos_first = {q: [] for q in quantis_map}
acertos_last = {q: [] for q in quantis_map}
# Nota: As variáveis df_rotulos, first_quantis, last_quantis são assumidas como carregadas.
df_rotulado_base = df_rotulos[df_rotulos['perturbacao'] == 0].copy()

for q in tqdm(quantis_map, desc=f"Calculando Concordância ({percentagem_quantil*100}%)"):
    try:
        file_path = os.path.join(f'{q}_quantis', f'dados_divididos_em_{q}_quantis.csv')
        df_dados = pd.read_csv(file_path, sep=';', decimal=',')
    except FileNotFoundError:
        acertos_first[q] = [np.nan] * len(sigmas)
        acertos_last[q] = [np.nan] * len(sigmas)
        continue

    total_amostras = len(df_dados)
    num_amostras_bloco = max(1, int(total_amostras * percentagem_quantil))

    df_first_group = df_dados.iloc[:num_amostras_bloco]
    df_last_group = df_dados.iloc[-num_amostras_bloco:]

    ids_first = df_first_group['ID'].tolist()
    ids_last = df_last_group['ID'].tolist()

    colunas_rotulos_first = [f'rotulo_{int(i)}' for i in ids_first]
    colunas_rotulos_last = [f'rotulo_{int(i)}' for i in ids_last]

    for sigma in sigmas:
        linha_rotulos = df_rotulado_base[(df_rotulado_base['qq'] == q) & (df_rotulado_base['sigma'] == sigma)]
        if not linha_rotulos.empty and ids_first and ids_last:
            rotulos_first = linha_rotulos[colunas_rotulos_first].values.flatten()
            taxa_first = (len(rotulos_first) - np.sum(rotulos_first)) / len(rotulos_first)
            acertos_first[q].append(taxa_first)

            rotulos_last = linha_rotulos[colunas_rotulos_last].values.flatten()
            taxa_last = np.sum(rotulos_last) / len(rotulos_last)
            acertos_last[q].append(taxa_last)
        else:
            acertos_first[q].append(np.nan)
            acertos_last[q].append(np.nan)

# ==============================================================================
# SEÇÃO 2: PREPARAÇÃO DOS DADOS (Com Filtro de Sigma)
# ==============================================================================
print("\n--- Seção 2: Preparando dados ---")
dados_otimizacao = []
for i, q in enumerate(quantis_map):
    for j, s in enumerate(sigmas):
        dados_otimizacao.append({'qq': q, 'sigma': round(s, 2), 'taxa_first': acertos_first[q][j], 'taxa_last': acertos_last[q][j]})
df_lookup = pd.DataFrame(dados_otimizacao).dropna()

sigma_min_aceito = 0.17
sigma_max_aceito = 0.20

# --- 1. FILTRO PARA MANTER APENAS SIGMA ENTRE 0.05 E 0.09 ---
df_lookup = df_lookup[
    (df_lookup['sigma'] >= sigma_min_aceito) &
    (df_lookup['sigma'] <= sigma_max_aceito)
].copy()

# --- 2. CORREÇÃO CRÍTICA: RESETAR O ÍNDICE ---
# Isso garante que o índice posicional seja contínuo (0, 1, 2, ...)
# para que o pymoo (que usa .values) e o .iloc (que usa posições) funcionem corretamente.
df_lookup.reset_index(drop=True, inplace=True)

print(f"Filtro aplicado: Mantidas apenas soluções com sigma entre {sigma_min_aceito:.2f} e {sigma_max_aceito:.2f}.")

# ==============================================================================
# SEÇÃO 3: CÁLCULO DA FRONTEIRA DE PARETO (Ocorre sobre os dados filtrados)
# ==============================================================================
print("\n--- Seção 3: Calculando a Fronteira de Pareto por Não-Dominância ---")
objective_values = -df_lookup[['taxa_first', 'taxa_last']].values
nds = NonDominatedSorting()
fronts = nds.do(objective_values)
pareto_indices = fronts[0]

# ==============================================================================
# SEÇÃO 4: ANÁLISE E SEPARAÇÃO DOS RESULTADOS
# ==============================================================================
print("\n--- Seção 4: Análise dos Resultados ---")
fronteira_de_pareto = df_lookup.iloc[pareto_indices].copy()
fronteira_de_pareto['soma_taxas'] = fronteira_de_pareto['taxa_first'] + fronteira_de_pareto['taxa_last']
fronteira_de_pareto = fronteira_de_pareto.sort_values(by=['qq', 'sigma'],ascending=[True, True]).reset_index(drop=True)

# Separação dos subgrupos para plotagem
max_soma = fronteira_de_pareto['soma_taxas'].max()
pontos_max_soma = fronteira_de_pareto[fronteira_de_pareto['soma_taxas'] == max_soma]
fronteira_restante = fronteira_de_pareto[fronteira_de_pareto['soma_taxas'] != max_soma]
dominados_indices = np.setdiff1d(df_lookup.index.values, pareto_indices)
pontos_dominados = df_lookup.iloc[dominados_indices]


# ==============================================================================
# SEÇÃO 5: PLOTAGEM DO GRÁFICO (COM JITTERING PARA EVITAR SOBREPOSIÇÃO)
# ==============================================================================
print("\n--- Seção 5: Gerando o gráfico comparativo com Jittering ---")

# FATOR DE TREMULAÇÃO (Ajuste conforme necessário)
# 0.002 é um valor pequeno o suficiente para separar visualmente sem distorcer a leitura
JITTER_AMOUNT = 0.008

def apply_jitter(arr, amount=0.0):
    """Adiciona ruído aleatório pequeno a um array para visualização."""
    if amount == 0: return arr
    noise = np.random.uniform(low=-amount, high=amount, size=len(arr))
    return arr + noise

# 1. BASE: Preparação dos dados para plotagem
# Concatena todos os pontos e garante que tenhamos as colunas originais
pontos_base = pd.concat([pontos_dominados, fronteira_restante, pontos_max_soma]).sort_values(by=['qq', 'sigma']).reset_index(drop=True)

# Aplica o jitter nas coordenadas DE PLOTAGEM (não nos dados originais)
# Criamos colunas temporárias '_jitter' para o gráfico
pontos_base['taxa_first_jitter'] = apply_jitter(pontos_base['taxa_first'], JITTER_AMOUNT)
pontos_base['taxa_last_jitter'] = apply_jitter(pontos_base['taxa_last'], JITTER_AMOUNT)

# Recalcula os limites do gráfico com base nos dados com jitter
padding = 0.02
range_x = [pontos_base['taxa_first_jitter'].min() - padding, pontos_base['taxa_first_jitter'].max() + padding]
range_y = [pontos_base['taxa_last_jitter'].min() - padding, pontos_base['taxa_last_jitter'].max() + padding]

fig = go.Figure()

# --- 1. PONTOS COLORIDOS (COM JITTER) ---
for q_val in quantis_map:
    df_qq = pontos_base[pontos_base['qq'] == q_val]
    if not df_qq.empty:
        fig.add_trace(go.Scatter(
            x=df_qq['taxa_first_jitter'], # Usa coordenada com jitter
            y=df_qq['taxa_last_jitter'],  # Usa coordenada com jitter
            mode='markers',
            marker=dict(size=8, color=QQ_COLORS[q_val], opacity=0.7), # Aumentei levemente o tamanho
            # O texto do hover continua mostrando os DADOS ORIGINAIS (sem jitter)
            text=[f"qq={row.qq}, sigma={row.sigma:.2f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in df_qq.iterrows()],
            hoverinfo='text',
            name=f'Pontos (qq={q_val})',
            legendgroup='qq_group'
        ))

# --- 2. DESTAQUES DE FRONTEIRA (Aplicar Jitter Correspondente) ---
# Precisamos encontrar as coordenadas com jitter correspondentes para os pontos de destaque
# A maneira mais fácil é filtrar do 'pontos_base' que já tem o jitter aplicado

# Filtra pontos da Fronteira Restante (baseado no índice ou valor original)
# Usamos uma máscara para encontrar as linhas que correspondem à fronteira
mask_fronteira = (pontos_base['soma_taxas'] != max_soma) & (pontos_base.index.isin(fronteira_restante.index if 'index' in fronteira_restante else []))
# Uma abordagem mais robusta se os índices foram resetados é merge ou filtro direto:
# Vamos filtrar pontos_base onde (qq, sigma) estão na fronteira_restante
for _, row_f in fronteira_restante.iterrows():
    # Encontra a linha correspondente em pontos_base para pegar a coordenada com jitter
    match = pontos_base[(pontos_base['qq'] == row_f['qq']) & (pontos_base['sigma'] == row_f['sigma'])]
    if not match.empty:
        fig.add_trace(go.Scatter(
            x=match['taxa_first_jitter'],
            y=match['taxa_last_jitter'],
            mode='markers',
            marker=dict(size=12, color='rgba(0,0,0,0)', symbol='circle', line=dict(width=2, color='blue')),
            hoverinfo='skip', # Hover já está no ponto colorido
            showlegend=False # Evita poluir a legenda (opcional)
        ))
# Adiciona um item de legenda falso apenas para identificar o círculo azul
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                         marker=dict(size=10, color='rgba(0,0,0,0)', symbol='circle', line=dict(width=2, color='blue')),
                         name='Fronteira (Círculo Azul)'))


# Filtra pontos de Soma Máxima
for _, row_m in pontos_max_soma.iterrows():
    match = pontos_base[(pontos_base['qq'] == row_m['qq']) & (pontos_base['sigma'] == row_m['sigma'])]
    if not match.empty:
        fig.add_trace(go.Scatter(
            x=match['taxa_first_jitter'],
            y=match['taxa_last_jitter'],
            mode='markers',
            marker=dict(size=16, color='rgba(0,0,0,0)', symbol='circle', line=dict(width=3, color='red')),
            hoverinfo='skip',
            showlegend=False
        ))
# Item de legenda falso para Soma Máxima
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                         marker=dict(size=14, color='rgba(0,0,0,0)', symbol='circle', line=dict(width=3, color='red')),
                         name='Soma Máxima (Círculo Vermelho)'))


fig.update_layout(
    title=f'<b>Fronteira de Pareto com Jittering (Visualização de Sobreposição)</b><br><span style="font-size: 12px;">Nota: Foi aplicado um pequeno ruído (jitter) para separar pontos sobrepostos.</span>',
    template='plotly_white', legend_title_text='<b>Tipo de Solução / Quantil</b>',
    xaxis=dict(title='Taxa de Concordância - FIRST', range=range_x),
    yaxis=dict(title='Taxa de Concordância - LAST', range=range_y),
    width=1000, height=700,
    shapes=[dict(type='line', x0=0, y0=1, x1=1, y1=1, line=dict(color='Gray', width=1, dash='dash')), dict(type='line', x0=1, y0=0, x1=1, y1=1, line=dict(color='Gray', width=1, dash='dash'))]
)

# ==============================================================================
# SEÇÃO 6: SALVAMENTO DOS GRÁFICOS
# ==============================================================================
print("\n--- Seção 6: Salvando o gráfico em arquivos ---")

output_dir = "graficos_fronteira_pareto"
os.makedirs(output_dir, exist_ok=True)
print(f"O gráfico será salvo no diretório: '{output_dir}'")

filename_base = f"fronteira_pareto_{sigma_min_aceito:.2f}_a_{sigma_max_aceito:.2f}_{round(100*percentagem_quantil)}_percent_Jitter"

try:
    # 1. Salvar em HTML (interativo)
    html_path = os.path.join(output_dir, f"{filename_base}.html")
    fig.write_html(html_path)
    print(f"  -> Gráfico salvo em: {os.path.basename(html_path)}")

    # 2. Salvar em PNG (alta resolução)
    png_path = os.path.join(output_dir, f"{filename_base}.png")
    fig.write_image(png_path, scale=3)
    print(f"  -> Gráfico salvo em: {os.path.basename(png_path)}")

    # 3. Salvar em PDF (vetorial)
    pdf_path = os.path.join(output_dir, f"{filename_base}.pdf")
    fig.write_image(pdf_path)
    print(f"  -> Gráfico salvo em: {os.path.basename(pdf_path)}")

except ValueError:
    print("\nAVISO: Para salvar em PNG/PDF, a biblioteca 'kaleido' é necessária.")

print("\nProcesso concluído!")

# VERIFICANDO TODOS OS QUANTIS
avaliar apenas os quantis extremos pode nao fornecer a quantidade de informacoes suficientes para uma correta avaliacao, pois ao aumentar o numero de quantis, estamos reduzindo a quantidade de amostras nestes, assim, em um caso extremo, usar um quantil que tenha apenas 1 amostras e esta for rotulada, teriamos uma taxa de concordância de 100%, independente do valor de sigma. Assim, avaliar os quantis intermediarios adjacentes ira nos fornecer dados que possam ratificar a qualidade da concordância nos quantis da extremidade, além de fornecer um panorama do comportamento da concordância ao longo dos quantis.


In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
from tqdm import tqdm

# --- PRÉ-REQUISITOS ---
# O código abaixo assume que as seguintes variáveis já existem:
# df_rotulos: DataFrame com os resultados dos rótulos.
# first_quantis: Lista de DataFrames para o grupo FIRST.
# last_quantis: Lista de DataFrames para o grupo LAST.
# df_dataset: DataFrame original para obter a lista completa de IDs.

# ==============================================================================
# SEÇÃO 1: PARÂMETROS E CRIAÇÃO DO DIRETÓRIO
# ==============================================================================
quantis_map = [2, 4, 8, 16]
quantis_plot = [2, 4, 8, 16]
sigma_to_plot = 0.07  # Defina aqui o sigma desejado

values_0 = []
values_1 = []

df_rotulos_filtrado = df_rotulos[
    (df_rotulos['sigma'] == sigma_to_plot) &
    (df_rotulos['perturbacao'] == 0)
].copy()

# --- NOVO: Criação do diretório de saída ---
output_dir = "graficos_contagem_rotulos_all"
os.makedirs(output_dir, exist_ok=True)
print(f"Os gráficos serão salvos no diretório: '{output_dir}'")

# ==============================================================================
# SEÇÃO 2: CÁLCULO DAS CONTAGENS (Inalterado)
# ==============================================================================
todos_os_ids = set(df_dataset.index)

for q in tqdm(quantis_plot, desc="Processando Quantis para Grupo 'ALL'"):
    try:
        idx = quantis_map.index(q)
    except ValueError:
        print(f"Aviso: Quantil {q} não encontrado no mapa. Pulando.")
        continue

    df_first = first_quantis[idx]
    df_last = last_quantis[idx]

    if df_first is None or df_last is None:
        values_0.append(0)
        values_1.append(0)
        continue

    ids_first = set(int(id_val) for id_val in df_first['ID'])
    ids_last = set(int(id_val) for id_val in df_last['ID'])
    ids_all = sorted(list(todos_os_ids - ids_first - ids_last))

    # Filtra apenas por colunas que realmente existem no df_rotulos
    colunas_existentes = [f'rotulo_{i}' for i in ids_all if f'rotulo_{i}' in df_rotulos_filtrado.columns]

    linha_rotulos = df_rotulos_filtrado[df_rotulos_filtrado['qq'] == q]

    if not linha_rotulos.empty and colunas_existentes:
        arr_all = linha_rotulos[colunas_existentes].values.flatten()
        total_0 = (arr_all == 0).sum()
        total_1 = (arr_all == 1).sum()
        values_0.append(total_0)
        values_1.append(total_1)
    else:
        values_0.append(0)
        values_1.append(0)

# ==============================================================================
# SEÇÃO 3: PLOTAGEM DO GRÁFICO (Inalterado)
# ==============================================================================
print("\n--- Seção 3: Gerando o gráfico de barras ---")
labels_x = [f'{q} quantis' for q in quantis_plot]

fig = go.Figure()
fig.add_trace(go.Bar(x=labels_x, y=values_0, name='Classe 0', marker_color='#636EFA'))
fig.add_trace(go.Bar(x=labels_x, y=values_1, name='Classe 1', marker_color='#EF553B'))

fig.update_layout(
    title=f'<b>Contagem de Rótulos no Grupo "ALL" (Sigma = {sigma_to_plot})</b>',
    xaxis_title='Quantis',
    yaxis_title='Quantidade de Amostras',
    barmode='group',
    template='plotly_white',
    legend_title_text='<b>Rótulo Previsto</b>',
    width=1200, # Define uma largura para consistência
    height=700  # Define uma altura para consistência
)

# ==============================================================================
# SEÇÃO 4: SALVAMENTO DO GRÁFICO
# ==============================================================================
print(f"\n--- Seção 4: Salvando o gráfico em arquivos ---")

# Nome de arquivo dinâmico para refletir o sigma utilizado
filename_base = f"contagem_rotulos_all_sigma_{sigma_to_plot:.2f}"

try:
    # 1. Salvar em HTML (interativo)
    html_path = os.path.join(output_dir, f"{filename_base}.html")
    fig.write_html(html_path)
    print(f"  -> Gráfico salvo em: {os.path.basename(html_path)}")

    # 2. Salvar em PNG (alta resolução)
    png_path = os.path.join(output_dir, f"{filename_base}.png")
    fig.write_image(png_path, scale=3) # scale=3 para alta resolução
    print(f"  -> Gráfico salvo em: {os.path.basename(png_path)}")

    # 3. Salvar em PDF (vetorial)
    pdf_path = os.path.join(output_dir, f"{filename_base}.pdf")
    fig.write_image(pdf_path)
    print(f"  -> Gráfico salvo em: {os.path.basename(pdf_path)}")

except ValueError:
    print("\nAVISO: Para salvar em PNG/PDF, a biblioteca 'kaleido' é necessária.")
    print("Instale com: pip install kaleido")

# A linha abaixo foi substituída pela lógica de salvamento.
# Descomente se, além de salvar, você também quiser exibir o gráfico na tela.
# fig.show()

print("\nProcesso concluído!")

In [ ]:
df_rotulos.head()

In [ ]:
df_dataset.head()

# Heatmap de dominância de rótulos (taxa de concordância)

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import plotly.graph_objects as go
import os

# --- PRÉ-REQUISITOS ---
# O código abaixo assume que as seguintes variáveis já existem:
# df_rotulos: DataFrame com os resultados dos rótulos.
# first_quantis: Lista de DataFrames para o grupo FIRST.
# last_quantis: Lista de DataFrames para o grupo LAST.
# df_dataset: DataFrame original para obter a lista completa de IDs.

# ==============================================================================
# SEÇÃO 1: CÁLCULO DAS CONTAGENS (Inalterado)
# ==============================================================================
quantis_map = [2, 4, 8, 16, 32, 64]
quantis_plot = [2, 4, 8, 16, 32, 64]
sigmas = np.arange(0.01, 0.31, 0.01)

counts_0 = {qq: [] for qq in quantis_plot}
counts_1 = {qq: [] for qq in quantis_plot}

df_rotulos_base = df_rotulos[df_rotulos['perturbacao'] == 0].copy()
todos_os_ids = set(df_dataset.index)

print("--- Seção 1: Calculando contagens para o grupo 'ALL' ---")
for i, q in enumerate(tqdm(quantis_plot, desc="Processando Quantis")):
    idx = quantis_map.index(q)
    df_first = first_quantis[idx]
    df_last = last_quantis[idx]

    if df_first is None or df_last is None:
        counts_0[q] = [0] * len(sigmas)
        counts_1[q] = [0] * len(sigmas)
        continue

    ids_first = set(int(id_val) for id_val in df_first['ID'])
    ids_last = set(int(id_val) for id_val in df_last['ID'])
    ids_all = sorted(list(todos_os_ids - ids_first - ids_last))

    colunas_rotulos_all = [f'rotulo_{id_val}' for id_val in ids_all if f'rotulo_{id_val}' in df_rotulos_base.columns]

    for sigma in sigmas:
        sigma = round(sigma, 2)
        linha_rotulos = df_rotulos_base[(df_rotulos_base['qq'] == q) & (df_rotulos_base['sigma'] == sigma)]

        if not linha_rotulos.empty and colunas_rotulos_all:
            arr_all = linha_rotulos[colunas_rotulos_all].values.flatten()
            total_0 = (arr_all == 0).sum()
            total_1 = (arr_all == 1).sum()
            counts_0[q].append(total_0)
            counts_1[q].append(total_1)
        else:
            counts_0[q].append(0)
            counts_1[q].append(0)

# ==============================================================================
# SEÇÃO 2: PREPARAÇÃO DOS DADOS E PLOTAGEM
# ==============================================================================
print("\n--- Seção 2: Preparando dados e gerando o heatmap ---")

quantis_str = [str(q) for q in quantis_plot]
heatmap_data = []
customdata = []

for i, qq in enumerate(quantis_plot):
    row, row_custom = [], []
    for j, sigma in enumerate(sigmas):
        val_0 = counts_0[qq][j]
        val_1 = counts_1[qq][j]
        diff = val_0 - val_1
        row.append(diff)
        row_custom.append([val_0, val_1])
    heatmap_data.append(row)
    customdata.append(row_custom)

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=[round(s, 2) for s in sigmas],
    y=quantis_str,
    customdata=customdata,
    colorscale=[[0, "red"], [0.5, "white"], [1, "blue"]],
    zmid=0,
    hovertemplate=
        '<b>Sigma</b>: %{x}<br>' +
        '<b>Quantil</b>: %{y}<br>' +
        '<b>Diferença (0-1)</b>: %{z}<br>' +
        '<hr>' +
        '<b>Contagem Classe 0</b>: %{customdata[0]}<br>' +
        '<b>Contagem Classe 1</b>: %{customdata[1]}<extra></extra>'
))

# --- ALTERAÇÃO FINAL AQUI ---
fig.update_layout(
    title='<b>Heatmap da Dominância de Rótulos no Grupo "ALL"</b>',
    template="plotly_white",
    width=1000,
    height=600,
    # Configuração de contorno completo para os eixos
    xaxis=dict(
        title='Sigma (σ)',
        showline=True,
        linewidth=2,
        linecolor='black',
        mirror=True  # Espelha a linha no topo
    ),
    yaxis=dict(
        title='Quantil (qq)',
        showline=True,
        linewidth=2,
        linecolor='black',
        mirror=True  # Espelha a linha à direita
    )
)

# ==============================================================================
# SEÇÃO 3: SALVAMENTO DO GRÁFICO (Inalterado)
# ==============================================================================
print("\n--- Seção 3: Salvando o gráfico em arquivos ---")

output_dir = "graficos_heatmap_dominancia"
os.makedirs(output_dir, exist_ok=True)
print(f"O gráfico será salvo no diretório: '{output_dir}'")
filename_base = "heatmap_dominancia_rotulos_all"

try:
    html_path = os.path.join(output_dir, f"{filename_base}.html")
    fig.write_html(html_path)
    print(f"  -> Gráfico salvo em: {os.path.basename(html_path)}")

    png_path = os.path.join(output_dir, f"{filename_base}.png")
    fig.write_image(png_path, scale=3)
    print(f"  -> Gráfico salvo em: {os.path.basename(png_path)}")

    pdf_path = os.path.join(output_dir, f"{filename_base}.pdf")
    fig.write_image(pdf_path)
    print(f"  -> Gráfico salvo em: {os.path.basename(pdf_path)}")

except ValueError:
    print("\nAVISO: Para salvar em PNG/PDF, a biblioteca 'kaleido' é necessária.")
    print("Instale com: pip install kaleido")

print("\nProcesso concluído!")

# Heatmaps de dominância quantil a quantil

In [ ]:
# ==============================================================================
# SEÇÃO 1: CARREGAMENTO DOS DADOS DE ENTRADA (Inalterado)
# ==============================================================================
print("--- Seção 1: Carregando arquivos de dados ---")

cwd = os.getcwd()
front_filepath = os.path.join(cwd, 'df_dataset.csv')
rotulos_filepath = os.path.join(cwd, 'rotulos', 'rotulos_unificados_todos_quantis_todos_sigmas.csv')

# ... (Seção 1: Carregamento - Mantido com o bloco try/except) ...

# 1. Carregamento do DataFrame principal (df_dataset/df_front)
try:
    df_front = pd.read_csv(front_filepath, sep=';', decimal=',')
    print(f"Arquivo '{os.path.basename(front_filepath)}' carregado com sucesso.")

    # CORREÇÃO AQUI: Garante que 'ID' exista no df_front (nosso DataFrame principal)
    if 'ID' not in df_front.columns:
        # Se 'ID' não existir, cria-o a partir do índice (que representa o ID original)
        df_front['ID'] = df_front.index.astype(int)
    else:
        df_front['ID'] = df_front['ID'].astype(int) # Apenas garante o tipo

except FileNotFoundError:
    print(f"ERRO: O arquivo '{front_filepath}' não foi encontrado. O script não pode continuar.")
    exit()
# ... (Carregamento df_rotulos) ...

# ==============================================================================
# SEÇÃO 2: PRÉ-CÁLCULO DOS SUBGRUPOS DE IDs (COM RANQUEAMENTO NSGA-II)
# ==============================================================================
print("\n--- Seção 2: Dividindo os IDs em subgrupos baseados no ranking ---")

# PRÉ-REQUISITO: df_all deve existir na memória e ter 'ID' e 'nsga2_rank'.
if 'df_all' not in locals():
    print("ERRO: df_all não foi encontrado na memória. Ranqueamento não pode ser aplicado.")
    exit()

# 1. Merge dos dados: Junta as coordenadas/ID com a coluna de ranqueamento (nsga2_rank)
# Assumimos que o df_front contém as colunas que o df_all não tem, e vice-versa.
df_ranqueado = pd.merge(
    df_front,
    df_all[['ID', 'nsga2_rank']], # Pega as chaves de ranqueamento
    on='ID',
    how='inner' # Garante apenas IDs que existem em ambos
)

# 2. Ordenação: Reordena o DataFrame principal pela coluna nsga2_rank
df_ranqueado.sort_values(by='nsga2_rank', inplace=True)

quantis_todos = [2, 4, 8, 16, 32, 64]
ids_quantis_dict = {}

# 3. Cálculo dos Subgrupos no DataFrame Ranqueado
for q in quantis_todos:
    # O array_split é feito no DataFrame JÁ ORDENADO PELO RANK
    subgrupos_df = np.array_split(df_ranqueado, q)

    # Extrai a coluna ID (que é o ID original da amostra) de cada subgrupo
    ids_para_q = [subgrupo['ID'].tolist() for subgrupo in subgrupos_df]
    ids_quantis_dict[q] = ids_para_q


# ==============================================================================
# SEÇÃO 3: CÁLCULO E PLOTAGEM DOS HEATMAPS
# ==============================================================================

# --- Parâmetros ---
quantis_plot = [2, 4, 8, 16, 32, 64]
sigmas = np.arange(0.01, 0.31, 0.01)
cell_height = 40
df_rotulos_base = df_rotulos[df_rotulos['perturbacao'] == 0].copy()

# --- NOVO: Criação do diretório de saída ---
output_dir = "graficos_heatmap_subgrupos"
os.makedirs(output_dir, exist_ok=True)
print(f"\nOs gráficos serão salvos no diretório: '{output_dir}'")

print("\n--- Seção 3:Gerando e salvando um heatmap para cada divisão por quantil ---")
for qq in tqdm(quantis_plot, desc="Gerando Gráficos por Quantil"):

    heatmap_data, customdata, y_labels = [], [], []

    for idx, ids_do_subgrupo in enumerate(ids_quantis_dict[qq]):
        row_data, row_custom = [], []
        colunas_rotulos_subgrupo = [f'rotulo_{i}' for i in ids_do_subgrupo if f'rotulo_{i}' in df_rotulos_base.columns]

        for sigma in sigmas:
            sigma = round(sigma, 2)
            linha_rotulos = df_rotulos_base[(df_rotulos_base['qq'] == qq) & (df_rotulos_base['sigma'] == sigma)]
            if not linha_rotulos.empty and colunas_rotulos_subgrupo:
                labels_subgrupo = linha_rotulos[colunas_rotulos_subgrupo].values.flatten()
                total_0 = (labels_subgrupo == 0).sum()
                total_1 = (labels_subgrupo == 1).sum()
                diff = total_0 - total_1
                row_data.append(diff)
                row_custom.append([total_0, total_1])
            else:
                row_data.append(np.nan)
                row_custom.append([np.nan, np.nan])
        heatmap_data.append(row_data)
        customdata.append(row_custom)
        y_labels.append(f'Quantil {idx+1}')

    # --- PLOTAGEM (Inalterado) ---
    if np.all(np.isnan(heatmap_data)):
        print(f"\nAviso: Não há dados para plotar para qq={qq}. Pulando.")
        continue

    max_val, min_val = np.nanmax(heatmap_data), np.nanmin(heatmap_data)
    fig_height = max(400, cell_height * len(y_labels))

    fig = go.Figure(data=go.Heatmap(
        z=heatmap_data, x=[round(s, 2) for s in sigmas], y=y_labels,
        customdata=customdata, colorscale=[[0, "red"], [0.5, "white"], [1, "blue"]],
        zmid=0, zmin=min_val, zmax=max_val,
        colorbar=dict(title='Diferença<br>(Classe 0 - Classe 1)'),
        hovertemplate= '<b>Sigma</b>: %{x}<br><b>Grupo</b>: %{y}<br><b>Diferença</b>: %{z}<br><hr><b>Classe 0</b>: %{customdata[0]}<br><b>Classe 1</b>: %{customdata[1]}<extra></extra>'
    ))

    fig.update_layout(
        title=f'<b>Heatmap da Dominância de Rótulos para {qq} Subgrupos</b>',
        template="plotly_white", # Retornando ao tema claro
        xaxis_title='Sigma (σ)', yaxis_title='Quantil Ranqueado',
        height=fig_height, width=1000,
        margin=dict(l=150), yaxis=dict(autorange='reversed')
    )

    # --- NOVA SEÇÃO DE SALVAMENTO (DENTRO DO LOOP) ---
    filename_base = f"heatmap_dominancia_subgrupos_{qq}_quantis"

    try:
        # 1. Salvar em HTML
        html_path = os.path.join(output_dir, f"{filename_base}.html")
        fig.write_html(html_path)
        print(f"  -> Gráfico para qq={qq} salvo em: {os.path.basename(html_path)}")

        # 2. Salvar em PNG (alta resolução)
        png_path = os.path.join(output_dir, f"{filename_base}.png")
        fig.write_image(png_path, scale=3)
        print(f"  -> Gráfico para qq={qq} salvo em: {os.path.basename(png_path)}")

        # 3. Salvar em PDF
        pdf_path = os.path.join(output_dir, f"{filename_base}.pdf")
        fig.write_image(pdf_path)
        print(f"  -> Gráfico para qq={qq} salvo em: {os.path.basename(pdf_path)}")

    except ValueError:
        print(f"\nAVISO: Não foi possível salvar em PNG/PDF. A biblioteca 'kaleido' é necessária.")
        print("Instale com: pip install kaleido")
        # Interrompe o loop para não repetir o aviso
        break

    # A chamada .show() foi substituída pela lógica de salvamento.
    # Descomente a linha abaixo se quiser exibir na tela além de salvar.
    # fig.show()

print("\n\nProcesso concluído.")

In [ ]:
# import pandas as pd
# import numpy as np
# import plotly.express as px
# import os
# import pickle
# from tqdm import tqdm

# --- 1. Parâmetros ---
QUANTIS_MAP = [2, 4, 8, 16, 32, 64]
# Assume que os arquivos .pkl estão no diretório atual ou em um subdiretório específico
# Se estiverem no diretório atual:
INPUT_DIR_BASE = os.getcwd()
# Se estiverem em outro lugar, ajuste o caminho:
# INPUT_DIR_BASE = "caminho/para/seus/arquivos_pkl"
print(f"Procurando arquivos .pkl em: '{INPUT_DIR_BASE}'")

# --- 2. Loop para processar cada qq ---
print("\n--- Gerando Histogramas de Tamanho de Comunidade por Quantil ---")
for qq in tqdm(QUANTIS_MAP, desc="Processando Quantis (qq)"):

    pickle_filename = f'dados_rotulados_{qq}_quantis.pkl'
    pickle_path = os.path.join(INPUT_DIR_BASE, pickle_filename)

    all_community_sizes = [] # Armazena os tamanhos das comunidades para este qq

    try:
        with open(pickle_path, 'rb') as f:
            dados_carregados = pickle.load(f)

        # --- Extração dos Grupos (Comunidades) ---
        # Assumindo que 'grupos' está na posição 4 (índice 4) da tupla salva
        # e que é um dicionário com chaves 'FIRST' e 'LAST'
        if len(dados_carregados) > 4 and isinstance(dados_carregados[4], dict):
            grupos_dict = dados_carregados[4]

            # Extrai tamanhos das comunidades FIRST
            if 'FIRST' in grupos_dict and isinstance(grupos_dict['FIRST'], list):
                # Calcula o tamanho (len) de cada item na lista 'FIRST', se for uma lista
                tamanhos_first = [len(comunidade) for comunidade in grupos_dict['FIRST'] if isinstance(comunidade, list)]
                all_community_sizes.extend(tamanhos_first)
            else:
                 print(f"  -> Aviso (qq={qq}): Chave 'FIRST' ou formato de lista inválido no dicionário de grupos.")


            # Extrai tamanhos das comunidades LAST
            if 'LAST' in grupos_dict and isinstance(grupos_dict['LAST'], list):
                # Calcula o tamanho (len) de cada item na lista 'LAST', se for uma lista
                tamanhos_last = [len(comunidade) for comunidade in grupos_dict['LAST'] if isinstance(comunidade, list)]
                all_community_sizes.extend(tamanhos_last)
            else:
                 print(f"  -> Aviso (qq={qq}): Chave 'LAST' ou formato de lista inválido no dicionário de grupos.")

        else:
            print(f"  -> Aviso (qq={qq}): Estrutura inesperada no arquivo pickle (índice 4 não é dict ou tupla curta). Não foi possível extrair grupos.")
            continue # Pula para o próximo qq se a estrutura estiver errada

        # --- Geração do Histograma ---
        if all_community_sizes:
            fig = px.histogram(
                x=all_community_sizes,
                # nbins=max(10, int(np.sqrt(len(all_community_sizes)))), # Heurística para número de bins
                nbins=20, # Ou use um número fixo se preferir
                title=f'<b>Distribuição do Tamanho das Comunidades - {qq} Quantis</b><br>(Combinando FIRST e LAST)',
                labels={'x': 'Tamanho da Comunidade (Nº de Amostras)', 'count':'Frequência (Nº de Comunidades)'}, # Corrigido label Y padrão do px.histogram
                template='plotly_white'
            )
            fig.update_layout(bargap=0.1) # Pequeno espaço entre as barras
            fig.show()
        else:
            print(f"  -> Aviso (qq={qq}): Nenhuma comunidade encontrada nos grupos FIRST ou LAST para gerar histograma.")

    except FileNotFoundError:
        print(f"\n  -> AVISO: Arquivo '{pickle_filename}' não encontrado. Pulando qq={qq}.")
    except (pickle.UnpicklingError, IndexError, TypeError, KeyError) as e:
        # Captura erros comuns de leitura ou estrutura incorreta
        print(f"\n  -> ERRO ao ler ou processar o arquivo '{pickle_filename}': {e}. Pulando qq={qq}.")

print("\n\nProcesso concluído!")

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import os
import pickle
from tqdm import tqdm

# --- 1. Parâmetros ---
QUANTIS_MAP = [2, 4, 8, 16, 32, 64]
INPUT_DIR_BASE = os.getcwd() # Ou ajuste o caminho se necessário
print(f"Procurando arquivos .pkl em: '{INPUT_DIR_BASE}'")

# Cores para os grupos
color_map = {'FIRST': 'green', 'LAST': 'red'}

# --- 2. Loop para processar cada qq ---
print("\n--- Gerando Histogramas de Tamanho de Comunidade por Quantil e Grupo ---")
for qq in tqdm(QUANTIS_MAP, desc="Processando Quantis (qq)"):

    pickle_filename = f'dados_rotulados_{qq}_quantis.pkl'
    pickle_path = os.path.join(INPUT_DIR_BASE, pickle_filename)

    # Listas para armazenar os dados para o DataFrame
    tamanhos_comunidade = []
    grupos_origem = []

    try:
        with open(pickle_path, 'rb') as f:
            dados_carregados = pickle.load(f)

        # --- Extração dos Grupos (Comunidades) ---
        if len(dados_carregados) > 4 and isinstance(dados_carregados[4], dict):
            grupos_dict = dados_carregados[4]

            # Processa comunidades FIRST
            if 'FIRST' in grupos_dict and isinstance(grupos_dict['FIRST'], list):
                tamanhos_first = [len(comunidade) for comunidade in grupos_dict['FIRST'] if isinstance(comunidade, list)]
                tamanhos_comunidade.extend(tamanhos_first)
                grupos_origem.extend(['FIRST'] * len(tamanhos_first))
            else:
                 print(f"  -> Aviso (qq={qq}): Chave 'FIRST' ou formato inválido.")

            # Processa comunidades LAST
            if 'LAST' in grupos_dict and isinstance(grupos_dict['LAST'], list):
                tamanhos_last = [len(comunidade) for comunidade in grupos_dict['LAST'] if isinstance(comunidade, list)]
                tamanhos_comunidade.extend(tamanhos_last)
                grupos_origem.extend(['LAST'] * len(tamanhos_last))
            else:
                 print(f"  -> Aviso (qq={qq}): Chave 'LAST' ou formato inválido.")

        else:
            print(f"  -> Aviso (qq={qq}): Estrutura inesperada no arquivo pickle. Não foi possível extrair grupos.")
            continue

        # --- Geração do Histograma Colorido ---
        if tamanhos_comunidade:
            # Cria DataFrame para Plotly Express
            df_plot = pd.DataFrame({
                'Tamanho': tamanhos_comunidade,
                'Grupo': grupos_origem
            })

            fig = px.histogram(
                df_plot,
                x='Tamanho',
                color='Grupo', # Coluna que define a cor das barras
                color_discrete_map=color_map, # Mapeia 'FIRST'->verde, 'LAST'->vermelho
                barmode='overlay', # Sobrepõe as barras para comparação
                # nbins=20, # Ajuste o número de bins se necessário
                opacity=0.7, # Adiciona transparência para ver sobreposições
                title=f'<b>Distribuição do Tamanho das Comunidades - {qq} Quantis</b>',
                labels={'Tamanho': 'Tamanho da Comunidade (Nº de Amostras)', 'count':'Frequência (Nº de Comunidades)'}
            )
            fig.update_layout(
                template='plotly_white',
                bargap=0.1,
                legend_title_text='Grupo Quantil'
                )
            fig.show()
        else:
            print(f"  -> Aviso (qq={qq}): Nenhuma comunidade encontrada para gerar histograma.")

    except FileNotFoundError:
        print(f"\n  -> AVISO: Arquivo '{pickle_filename}' não encontrado. Pulando qq={qq}.")
    except (pickle.UnpicklingError, IndexError, TypeError, KeyError) as e:
        print(f"\n  -> ERRO ao ler ou processar o arquivo '{pickle_filename}': {e}. Pulando qq={qq}.")

print("\n\nProcesso concluído!")


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import os
import pickle
from tqdm import tqdm

# --- 1. Parâmetros ---
QUANTIS_MAP = [2, 4, 8, 16, 32, 64]
INPUT_DIR_BASE = os.getcwd() # Ou ajuste o caminho se necessário
print(f"Procurando arquivos .pkl em: '{INPUT_DIR_BASE}'")

# Cores para os grupos
color_map = {'FIRST': 'green', 'LAST': 'red'}

# --- 2. Loop para processar cada qq ---
print("\n--- Gerando Histogramas de Tamanho de Comunidade por Quantil e Grupo ---")
for qq in tqdm(QUANTIS_MAP, desc="Processando Quantis (qq)"):

    pickle_filename = f'dados_rotulados_{qq}_quantis.pkl'
    pickle_path = os.path.join(INPUT_DIR_BASE, pickle_filename)

    tamanhos_comunidade = []
    grupos_origem = []

    try:
        with open(pickle_path, 'rb') as f:
            dados_carregados = pickle.load(f)

        if len(dados_carregados) > 4 and isinstance(dados_carregados[4], dict):
            grupos_dict = dados_carregados[4]

            if 'FIRST' in grupos_dict and isinstance(grupos_dict['FIRST'], list):
                tamanhos_first = [len(comunidade) for comunidade in grupos_dict['FIRST'] if isinstance(comunidade, list)]
                tamanhos_comunidade.extend(tamanhos_first)
                grupos_origem.extend(['FIRST'] * len(tamanhos_first))

            if 'LAST' in grupos_dict and isinstance(grupos_dict['LAST'], list):
                tamanhos_last = [len(comunidade) for comunidade in grupos_dict['LAST'] if isinstance(comunidade, list)]
                tamanhos_comunidade.extend(tamanhos_last)
                grupos_origem.extend(['LAST'] * len(tamanhos_last))

        else:
            print(f"  -> Aviso (qq={qq}): Estrutura inesperada no arquivo pickle.")
            continue

        if tamanhos_comunidade:
            df_plot = pd.DataFrame({
                'Tamanho': tamanhos_comunidade,
                'Grupo': grupos_origem
            })

            fig = px.histogram(
                df_plot,
                x='Tamanho',
                color='Grupo',
                color_discrete_map=color_map,
                # --- ALTERAÇÃO PRINCIPAL AQUI ---
                barmode='group', # Muda de 'overlay' para 'group'
                # opacity=0.7, # Removido ou definido como 1.0 (opcional)
                # ---------------------------------
                # nbins=20,
                title=f'<b>Distribuição do Tamanho das Comunidades - {qq} Quantis</b>',
                labels={'Tamanho': 'Tamanho da Comunidade (Nº de Amostras)', 'count':'Frequência (Nº de Comunidades)'}
            )
            fig.update_layout(
                template='plotly_white',
                bargap=0.1, # Espaço entre grupos de barras
                bargroupgap=0.05, # Espaço entre barras dentro de um grupo
                legend_title_text='Grupo Quantil'
                )
            fig.show()
        else:
            print(f"  -> Aviso (qq={qq}): Nenhuma comunidade encontrada.")

    except FileNotFoundError:
        print(f"\n  -> AVISO: Arquivo '{pickle_filename}' não encontrado. Pulando qq={qq}.")
    except (pickle.UnpicklingError, IndexError, TypeError, KeyError) as e:
        print(f"\n  -> ERRO ao ler ou processar '{pickle_filename}': {e}. Pulando qq={qq}.")

print("\n\nProcesso concluído!")

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
from tqdm import tqdm

# --- 1. Parâmetros ---
QUANTIS_MAP = [2, 4, 8, 16, 32, 64]
INPUT_DIR_BASE = os.getcwd() # Ou ajuste o caminho se necessário
print(f"Procurando arquivos .pkl em: '{INPUT_DIR_BASE}'")

# Dicionário para armazenar os resultados (opcional, mas bom para uso posterior)
resultados_contagem = {}

# --- 2. Loop para processar cada qq ---
print("\n--- Contando Comunidades por Quantil e Grupo ---")
for qq in tqdm(QUANTIS_MAP, desc="Processando Quantis (qq)"):

    pickle_filename = f'dados_rotulados_{qq}_quantis.pkl'
    pickle_path = os.path.join(INPUT_DIR_BASE, pickle_filename)

    count_first = 0 # Default count
    count_last = 0  # Default count
    status = "OK"   # Status inicial

    try:
        with open(pickle_path, 'rb') as f:
            dados_carregados = pickle.load(f)

        # --- Extração e Contagem ---
        # Verifica se a estrutura esperada (tupla com pelo menos 5 elementos, 5º é dict) existe
        if len(dados_carregados) > 4 and isinstance(dados_carregados[4], dict):
            grupos_dict = dados_carregados[4]

            # Conta comunidades FIRST
            if 'FIRST' in grupos_dict and isinstance(grupos_dict['FIRST'], list):
                # Conta quantos itens (comunidades) existem na lista
                count_first = len(grupos_dict['FIRST'])
            else:
                 # print(f"  -> Aviso (qq={qq}): Chave 'FIRST' ou formato inválido.") # Silenciado para tqdm
                 status = "Aviso: Estrutura FIRST inválida"


            # Conta comunidades LAST
            if 'LAST' in grupos_dict and isinstance(grupos_dict['LAST'], list):
                # Conta quantos itens (comunidades) existem na lista
                count_last = len(grupos_dict['LAST'])
            else:
                 # print(f"  -> Aviso (qq={qq}): Chave 'LAST' ou formato inválido.") # Silenciado para tqdm
                 status = "Aviso: Estrutura LAST inválida"

            resultados_contagem[qq] = {'FIRST': count_first, 'LAST': count_last}


        else:
            status = "Erro: Estrutura Pickle Inválida (índice 4)"
            # print(f"  -> Aviso (qq={qq}): Estrutura inesperada no arquivo pickle (índice 4 não é dict ou tupla curta).") # Silenciado
            resultados_contagem[qq] = {'FIRST': None, 'LAST': None} # Indica erro

    except FileNotFoundError:
        status = "Erro: Arquivo não encontrado"
        # print(f"\n  -> AVISO: Arquivo '{pickle_filename}' não encontrado. Pulando qq={qq}.") # Silenciado
        resultados_contagem[qq] = {'FIRST': None, 'LAST': None}
    except (pickle.UnpicklingError, IndexError, TypeError, KeyError) as e:
        status = f"Erro: {e}"
        # print(f"\n  -> ERRO ao ler ou processar o arquivo '{pickle_filename}': {e}. Pulando qq={qq}.") # Silenciado
        resultados_contagem[qq] = {'FIRST': None, 'LAST': None}

    # Imprime o resultado formatado após o loop interno ou erro
    if status == "OK":
        tqdm.write(f"  qq = {qq}: FIRST={count_first} comunidades, LAST={count_last} comunidades") # Usa tqdm.write para não quebrar a barra
    else:
        tqdm.write(f"  qq = {qq}: {status}")


print("\n\nProcesso concluído!")

# Opcional: Imprimir o dicionário final de forma mais estruturada
print("\nResumo das contagens:")
df_resumo = pd.DataFrame.from_dict(resultados_contagem, orient='index')
print(df_resumo)

In [ ]:
import numpy as np
import igraph as ig
from tqdm import tqdm
import time

# --- 1. PARÂMETROS GLOBAIS ---
N_NOS = 100            # Número de nós
N_REPETICOES = 30      # Repetições para cálculo da média

# NOVO: Lista de probabilidades P a serem variadas (ex: de muito esparso a moderadamente denso)
P_A_RODAR = np.round(np.arange(0.02, 0.30, 0.03), 3) # Ex: [0.02, 0.05, 0.08, 0.11, 0.14]

# Peso fixo (similaridade máxima) para todos os links existentes,
# garantindo que a variação na granularidade venha APENAS da probabilidade (P).
PESO_FIXO = 1.0


# --- 2. FUNÇÃO DE EXPERIMENTO (TESTANDO VARIAÇÃO DE P) ---
def detectar_comunidades_por_probabilidade(p_conexao):
    """
    Gera um grafo aleatório (Erdős–Rényi) com a probabilidade P_CONEXAO
    e peso uniforme.
    """
    contagem_comunidades = []

    nome_cenario = f"P={p_conexao:.3f}"
    print(f"\n--- Cenário: {nome_cenario} ---")

    for _ in tqdm(range(N_REPETICOES), desc=f"  Processando {nome_cenario}"):

        # 1. Geração do Grafo (P agora é variável)
        G = ig.Graph.Erdos_Renyi(n=N_NOS, p=p_conexao, directed=False)

        # 2. Atribuição de Peso Fixo (Similaridade = 1.0)
        # O peso é atribuído apenas aos links que foram criados (G.ecount())
        G.es["weight"] = PESO_FIXO

        # 3. Detecção de Comunidade (Fast Greedy usa o peso fixo)
        dendrogram = G.community_fastgreedy(weights='weight')
        clustering = dendrogram.as_clustering()

        num_comunidades = len(clustering)
        contagem_comunidades.append(num_comunidades)

    media_comunidades = np.mean(contagem_comunidades)

    return media_comunidades, nome_cenario


# --- 3. EXECUÇÃO DO EXPERIMENTO ---
print("\n=== ANÁLISE DA GRANULARIDADE PELA PROBABILIDADE DE CONEXÃO (P) ===")

resultados_finais = []
for p in P_A_RODAR:
    media, nome_cenario = detectar_comunidades_por_probabilidade(p)
    resultados_finais.append((nome_cenario, media))


# --- 4. EXIBIÇÃO DOS RESULTADOS ---
print("\n\n=== RESULTADO FINAL (Efeito da Probabilidade) ===")
print("Probabilidade (P) | Média de Comunidades")
print("-" * 40)

for nome, media in resultados_finais:
    print(f"{nome:17}| {media:.2f}")

print("=" * 40)

In [ ]:
# import os
#
# def apagar_arquivos_html_recursivamente(diretorio_raiz):
#     """
#     Percorre o diretório de trabalho fornecido e todas as suas subpastas,
#     apagando todos os arquivos que terminam com a extensão .html.
#
#     Args:
#         diretorio_raiz (str): O caminho para a pasta principal onde a busca começará.
#     """
#     print(f"--- Iniciando varredura em: '{diretorio_raiz}' ---")
#
#     # Contador para o resumo
#     arquivos_apagados = 0
#
#     # os.walk percorre a árvore de diretórios de forma top-down (de cima para baixo)
#     for root, dirs, files in os.walk(diretorio_raiz):
#
#         # O 'root' é o diretório atual que está sendo examinado
#
#         # 1. Filtra os arquivos no diretório atual que terminam em '.html'
#         arquivos_html = [f for f in files if f.lower().endswith('.html')]
#
#         if arquivos_html:
#             print(f"\nEncontrados {len(arquivos_html)} arquivos HTML em: {root}")
#
#         # 2. Itera sobre os arquivos HTML encontrados e os apaga
#         for filename in arquivos_html:
#             file_path = os.path.join(root, filename)
#
#             try:
#                 # 3. Executa a exclusão do arquivo
#                 os.remove(file_path)
#                 arquivos_apagados += 1
#                 # print(f"  -> Apagado: {filename}") # Descomente para log detalhado
#             except OSError as e:
#                 # Trata erros de permissão ou arquivo em uso
#                 print(f"  -> ERRO: Não foi possível apagar '{file_path}'. Motivo: {e}")
#
#     print("\n--- Processo de limpeza concluído ---")
#     print(f"Total de arquivos .html apagados: {arquivos_apagados}")
#
# # ==============================================================================
# # --- EXECUÇÃO ---
# # ==============================================================================
#
# # Defina a pasta principal que você deseja limpar.
# # ATENÇÃO: Use um caminho absoluto ou o nome de uma pasta que esteja no seu CWD.
# # Exemplo: pasta_destino = 'graficos_fronteira_pareto'
# # Exemplo: pasta_destino = os.getcwd() # Limpa tudo no diretório atual e subpastas!
#
# # Use 'r' antes das aspas
# pasta_destino = r'D:\DTLZ2\Var3\graficos_scatterplot_com_confianca - Copia'
#
# # Descomente a linha abaixo para executar:
# apagar_arquivos_html_recursivamente(pasta_destino)
#
# # --- AVISO DE SEGURANÇA ---
# print("\nAVISO DE SEGURANÇA: O código está comentado para prevenir exclusão acidental.")
# print("Para executar, substitua 'nome_da_sua_pasta_raiz_aqui' pelo caminho desejado e descomente a última linha.")

# lixo

In [ ]:
pwd

In [ ]:
import os
import pandas as pd

# 1. Carregar o arquivo (Atenção ao separador de colunas e decimal brasileiro)
output_dir_flip = os.path.join(os.getcwd(), 'flip_rates')
file_path = os.path.join(output_dir_flip, 'flip_rates_consolidados.csv')

# Fallback: Se o arquivo estiver solto na mesma pasta em vez da subpasta 'flip_rates'
if not os.path.exists(file_path):
    file_path = 'flip_rates_consolidados.csv'

df = pd.read_csv(file_path, sep=';', decimal=',')

# 2. Identificar todas as colunas de perturbação
pert_cols = [c for c in df.columns if 'flip_rate_pert_' in c]

# 3. Calcular a média do flip-rate de todas as perturbações para aquela configuração
df['flip_rate_mean'] = df[pert_cols].mean(axis=1)

# 4. Arredondar o sigma para evitar problemas de precisão
df['sigma_round'] = df['sigma'].round(2)

# ==========================================================
# 5. MUDANÇA AQUI: Filtrar pelo FLIP-RATE entre 0.03 (3%) e 0.07 (7%)
# ==========================================================
df_filtered = df[(df['flip_rate_mean'] >= 0.03) & (df['flip_rate_mean'] <= 0.07)]

# 6. Selecionar apenas as colunas relevantes, ordenar e converter a média para porcentagem
results = df_filtered[['qq', 'sigma_round', 'flip_rate_mean']].copy()
results.rename(columns={'sigma_round': 'sigma', 'qq': 'quantis'}, inplace=True)

# Transformar em porcentagem (%) para facilitar a leitura
results['flip_rate_mean_%'] = (results['flip_rate_mean'] * 100).round(2)

# Ordenar por Quantil e depois por Sigma
results = results.sort_values(by=['quantis', 'sigma']).reset_index(drop=True)

# Exibir os resultados
print("Configurações com Flip-Rate médio entre 3% e 7%:\n")
print(results[['quantis', 'sigma', 'flip_rate_mean_%']].to_string(index=False))

# Calculo da taxa de concordancia para valores de sigmas validos
sigmas validos sao aqueles que apresentam valores proximos de 5% de flip-rate

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import plotly.graph_objects as go
from IPython.display import display
import os
import plotly.express as px

# ==============================================================================
# PALETA DE CORES POR QUANTIL
# ==============================================================================
QQ_COLORS = {
    2: px.colors.qualitative.Vivid[0],
    4: px.colors.qualitative.Vivid[1],
    8: px.colors.qualitative.Vivid[2],
    16: px.colors.qualitative.Vivid[3],
    32: px.colors.qualitative.Vivid[4],
    64: px.colors.qualitative.Vivid[5],
}

# ==============================================================================
# SEÇÃO 1: CÁLCULO DAS TAXAS DE CONCORDÂNCIA
# ==============================================================================
print("--- Seção 1: Calculando taxas de concordância ---")

# Parâmetros
quantis_map = [2, 4, 8, 16, 32, 64]
percentagem_quantil = 0.10
sigma_limite = 0.30
passo = 0.01
sigmas = np.round(np.arange(passo, sigma_limite + passo, passo), 2)

acertos_first = {q: [] for q in quantis_map}
acertos_last = {q: [] for q in quantis_map}

# Nota: df_rotulos deve estar previamente carregado no ambiente
df_rotulado_base = df_rotulos[df_rotulos['perturbacao'] == 0].copy()

for q in tqdm(quantis_map, desc=f"Calculando Concordância ({percentagem_quantil*100}%)"):
    try:
        file_path = os.path.join(f'{q}_quantis', f'dados_divididos_em_{q}_quantis.csv')
        df_dados = pd.read_csv(file_path, sep=';', decimal=',')
    except FileNotFoundError:
        acertos_first[q] = [np.nan] * len(sigmas)
        acertos_last[q] = [np.nan] * len(sigmas)
        continue

    total_amostras = len(df_dados)
    num_amostras_bloco = max(1, int(total_amostras * percentagem_quantil))

    df_first_group = df_dados.iloc[:num_amostras_bloco]
    df_last_group = df_dados.iloc[-num_amostras_bloco:]

    ids_first = df_first_group['ID'].tolist()
    ids_last = df_last_group['ID'].tolist()

    colunas_rotulos_first = [f'rotulo_{int(i)}' for i in ids_first]
    colunas_rotulos_last = [f'rotulo_{int(i)}' for i in ids_last]

    for sigma in sigmas:
        linha_rotulos = df_rotulado_base[(df_rotulado_base['qq'] == q) & (df_rotulado_base['sigma'] == sigma)]
        if not linha_rotulos.empty and ids_first and ids_last:
            rotulos_first = linha_rotulos[colunas_rotulos_first].values.flatten()
            taxa_first = (len(rotulos_first) - np.sum(rotulos_first)) / len(rotulos_first)
            acertos_first[q].append(taxa_first)

            rotulos_last = linha_rotulos[colunas_rotulos_last].values.flatten()
            taxa_last = np.sum(rotulos_last) / len(rotulos_last)
            acertos_last[q].append(taxa_last)
        else:
            acertos_first[q].append(np.nan)
            acertos_last[q].append(np.nan)

# ==============================================================================
# SEÇÃO 2: PREPARAÇÃO DOS DADOS E FILTRO DOS SIGMAS VÁLIDOS
# ==============================================================================
print("\n--- Seção 2: Aplicando filtro de Sigmas válidos ---")

# Dicionário com os sigmas válidos para cada quantil (baseado no Flip-Rate)
# -> ATENÇÃO: Ajuste estes valores conforme a sua análise de Flip-Rate (~5%)
# Dicionário de Restrição para a base de INSEGURANÇA ALIMENTAR (FI)
# Sigmas que geram Flip-Rate entre 3% e 7%
sigmas_validos_por_quantil = {
    2: [0.15],
    4: [0.16],
    8: [0.16, 0.17],
    16: [0.16]
}

dados_otimizacao = []

for i, q in enumerate(quantis_map):
    sigmas_permitidos = sigmas_validos_por_quantil.get(q, [])
    for j, s in enumerate(sigmas):
        s_round = round(s, 2)

        # Filtra apenas as configurações válidas fornecidas no dicionário
        if s_round in sigmas_permitidos:
            dados_otimizacao.append({
                'qq': q,
                'sigma': s_round,
                'taxa_first': acertos_first[q][j],
                'taxa_last': acertos_last[q][j]
            })

df_resultados = pd.DataFrame(dados_otimizacao).dropna()

# ==============================================================================
# SEÇÃO 3: CÁLCULO DA SOMA E ORDENAÇÃO (SUBSTITUIU O NDS)
# ==============================================================================
print("\n--- Seção 3: Cálculo da Soma Máxima das Taxas ---")

# Calcula a soma das taxas para todas as configurações filtradas
df_resultados['soma_taxas'] = df_resultados['taxa_first'] + df_resultados['taxa_last']

max_soma = df_resultados['soma_taxas'].max()
df_resultados['classificacao'] = 'Válido'
df_resultados.loc[df_resultados['soma_taxas'] == max_soma, 'classificacao'] = 'Soma Máxima'

# Ordenação final para a tabela
df_resultados = df_resultados.sort_values(
    by=['soma_taxas', 'qq', 'sigma'],
    ascending=[False, True, True]
).reset_index(drop=True)

# EXIBIÇÃO DA TABELA
print("\nTABELA: CONFIGURAÇÕES E SOMA DAS TAXAS DE CONCORDÂNCIA")
cols_to_show = ['classificacao', 'qq', 'sigma', 'taxa_first', 'taxa_last', 'soma_taxas']
print(df_resultados[cols_to_show].to_string(index=False))

try:
    display(df_resultados[cols_to_show])
except NameError:
    pass

# ==============================================================================
# SEÇÃO 4: PLOTAGEM DO GRÁFICO (Configurações Válidas e Soma Máxima)
# ==============================================================================
print("\n--- Seção 4: Gerando o gráfico comparativo ---")

pontos_validos = df_resultados[df_resultados['classificacao'] == 'Válido']
pontos_max_soma = df_resultados[df_resultados['classificacao'] == 'Soma Máxima']

padding = 0.02
range_x = [df_resultados['taxa_first'].min() - padding, df_resultados['taxa_first'].max() + padding]
range_y = [df_resultados['taxa_last'].min() - padding, df_resultados['taxa_last'].max() + padding]

fig = go.Figure()

# 1. BASE: Plota TODAS as Soluções Válidas coloridas por QQ.
for q_val in quantis_map:
    df_qq = df_resultados[df_resultados['qq'] == q_val]
    if not df_qq.empty:
        fig.add_trace(go.Scatter(
            x=df_qq['taxa_first'], y=df_qq['taxa_last'], mode='markers',
            marker=dict(size=8, color=QQ_COLORS[q_val], opacity=0.7),
            text=[f"qq={row.qq}, sigma={row.sigma:.2f}<br>Soma: {row.soma_taxas:.3f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in df_qq.iterrows()],
            hoverinfo='text',
            name=f'{q_val} quantiles',
            legendgroup='qq_group'
        ))

# 2. CONTORNO: PONTO DE SOMA MÁXIMA (Círculo vazio vermelho)
fig.add_trace(go.Scatter(
    x=pontos_max_soma['taxa_first'], y=pontos_max_soma['taxa_last'], mode='markers',
    marker=dict(
        size=16,
        color='rgba(0,0,0,0)',
        symbol='circle',
        line=dict(width=3, color='red')
    ),
    text=[f"<b>SOMA MÁXIMA</b><br>qq={row.qq}, sigma={row.sigma:.2f}<br>Soma: {row.soma_taxas:.3f}<br>First: {row.taxa_first:.3f}<br>Last: {row.taxa_last:.3f}" for _, row in pontos_max_soma.iterrows()],
    hoverinfo='text', name=f'Max Sum Highlight'
))

fig.update_layout(
    template='plotly_white',
    title=f'<b>Taxas de Concordância - Seleção Restrita ({int(percentagem_quantil*100)}%)</b>',
    xaxis=dict(title='Concordance Rate - Initial Samples', range=range_x),
    yaxis=dict(title='Concordance Rate - Final Samples', range=range_y),
    width=1000, height=700,
    shapes=[
        dict(type='line', x0=0, y0=1, x1=1, y1=1, line=dict(color='Gray', width=1, dash='dash')),
        dict(type='line', x0=1, y0=0, x1=1, y1=1, line=dict(color='Gray', width=1, dash='dash'))
    ]
)

# ==============================================================================
# SEÇÃO 5: SALVAMENTO DOS GRÁFICOS
# ==============================================================================
print("\n--- Seção 5: Salvando o gráfico em arquivos ---")

output_dir = "graficos_concordancia"
os.makedirs(output_dir, exist_ok=True)
print(f"O gráfico será salvo no diretório: '{output_dir}'")

filename_base = f"concordancia_soma_{int(percentagem_quantil*100)}_percent"

try:
    # 1. Salvar em HTML (interativo)
    html_path = os.path.join(output_dir, f"{filename_base}.html")
    fig.write_html(html_path)
    print(f"  -> Gráfico salvo em: {os.path.basename(html_path)}")

    # 2. Salvar em PDF (vetorial)
    pdf_path = os.path.join(output_dir, f"{filename_base}.pdf")
    fig.write_image(pdf_path, scale=5)
    print(f"  -> Gráfico salvo em: {os.path.basename(pdf_path)}")

except ValueError:
    print("\nAVISO: Para salvar em PDF, a biblioteca 'kaleido' é necessária.")

print("\nProcesso concluído!")